<a href="https://colab.research.google.com/github/andrevt-cmd/customs/blob/main/KIOKO_GENERAL_CUSTOMS_FILTERS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LEMBRAR DE DESCOMENTAR NO FINAL PARA SALVAR NO CHECK TBM

Para rodar ou fazer alterações nesse código, acesse o link abaixo e leia como
ele funciona.

https://docs.google.com/document/d/1ebkdE31AYxsmAg2SmSGY562YBfuqv4j7M5RWZSw3ZCU/edit?tab=t.0#heading=h.rjixvg87kiow

# 0.0. IMPORTS

## 0.0.1 IMPORTS


In [1]:
# Instala a biblioteca necessária
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.7 MB/s eta 0:00:00


In [2]:
# Importa as bibliotecas necessárias
from rapidfuzz import process, fuzz
import pandas as pd
import regex as re
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Faz conexão com o Google Drive (para pegar os arquivos necessários)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## O.1. LOAD DATA


In [4]:
# Diretório BASE onde tiraremos as informações
pasta_drive = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/'

In [5]:
# Lista todos os diretórios dentro da pasta BASE
diretorios = os.listdir(pasta_drive)

# Loop para passar por todos os diretórios
for i, diretorio in enumerate(diretorios):
  # Se for um diretório:
  if os.path.isdir(os.path.join(pasta_drive, diretorio)):
    # Printa os diretórios
    print(f"{i+1}. {diretorio}")

1. LAR
2. CHINA
3. APA
4. EMEA
5. BRAZIL
6. RESULTS
7. REPORTS
8. EXPORT GENIUS
10. APA Compressors
12. concatenated_old
15. NAR
17. Old DBs


In [6]:
 # Recebe o input do usuário e escolhe o diretório correspondente ao número digitado
escolha = int(input("Digite o numero do diretório que você deseja abrir: ")) - 1
diretorio_escolhido = os.path.join(pasta_drive, diretorios[escolha])
print(f"Você escolheu o diretório: {diretorio_escolhido}")

Digite o numero do diretório que você deseja abrir: 1
Você escolheu o diretório: /content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/LAR


In [7]:
# Lista todos os arquivos que existem dentro da pasta escolhida
print("Arquivos possiveis\n")
arquivos = os.listdir(diretorio_escolhido)
# Se for um arquivo:
for i, arquivo in enumerate(arquivos):
  # Printe os arquivos
  print(f"{i+1}. {arquivo}")

Arquivos possiveis

1. OLD
2. ECUADOR_25.xlsx
3. ECUADOR_24.xlsx
4. COLOMBIA_24.xlsx
5. ARGENTINA_24.xlsx
6. ARGENTINA_25.xlsx
7. COLOMBIA_25_ate8.xlsx
8. BOLIVIA.xlsx
9. PARAGUAY.xlsx
10. PERU.xlsx
11. PANAMA.xlsx
12. ARGENTINA.xlsx
13. COLOMBIA.xlsx
14. ECUADOR.xlsx
15. CHILE.xlsx


In [8]:
 # Recebe o input do usuário e escolhe o arquivo correspondente ao número digitado
escolha = int(input("Digite o número do arquivo que você deseja tratar: ")) - 1
arquivo_escolhido = os.path.join(diretorio_escolhido, arquivos[escolha])
print(f"Você escolheu o arquivo: {arquivo_escolhido}")

# Depois de escolher o arquivo, o arquivo é lido.
df_raw = pd.read_excel(arquivo_escolhido)

Digite o número do arquivo que você deseja tratar: 15
Você escolheu o arquivo: /content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/LAR/CHILE.xlsx


## 0.2. FUNÇÕES DE AUXILIO

### 0.2.1. FUNÇÕES PARA CRIAR NOVAS COLUNAS


In [9]:
def criar_coluna_data(df):
  """Cria uma coluna 'Date' no DataFrame se ela não existir.

  Verifica se alguma das colunas 'Date', 'Data', 'Fecha' ou 'Declaration Date' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma coluna 'Date' combinando as colunas 'Year' e 'Month',
  convertendo-as para o tipo datetime.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Date' adicionada, se necessário.
  """
  # Verifica se a coluna 'Date' ou similares já existem no DataFrame
  if "Date" in df.columns:
    # Se sim, retorna o DataFrame sem alterações
    return df

  if "Data" in df.columns:
    df.rename(columns={"Data" : "Date"}, inplace = True)
    return df

  if "Fecha" in df.columns:
    df.rename(columns={"Fecha" : "Date"}, inplace = True)
    return df

  if "Declaration Date" in df.columns:
    df.rename(columns={"Declaration Date" : "Date"}, inplace = True)
    return df


  # Se não, continua a seguir:
  # Converte as colunas 'Year' e 'Month' para string e substitui valores de erro por strings vazias
  df['Year'] = df['Year'].astype(str).replace('#REF!-#REF!', '')  # Trata valores de erro na coluna 'Year'
  df['Month'] = df['Month'].astype(str).replace('#REF!-#REF!', '')  # Trata valores de erro na coluna 'Month'

  # Cria a coluna 'Date' combinando 'Year' e 'Month' e convertendo para datetime
  #df["Date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str), format="%Y-%m", errors='coerce')  # Cria a coluna 'Date'
  df['Date'] = '01'+'/'+df['Month'].astype(str) + '/' + df['Year'].astype(str)

  # Retorna o DataFrame com a coluna 'Date' adicionada
  return df

In [10]:
def criar_coluna_model(df):
  """Cria uma coluna 'Model' no DataFrame se ela não existir.

  Verifica se alguma das colunas 'Model', 'Modelo' ou 'Subitem: Modelo' já existe no DataFrame.
  Se 'Model' existir, retorna o DataFrame original sem alterações.
  Se 'Modelo' ou 'Subitem: Modelo' existir, renomeia para 'MODELO' e cria uma nova coluna 'Model' vazia.
  Se nenhuma das colunas existir, cria uma nova coluna 'Model' vazia.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Model' adicionada ou renomeada, se necessário.
  """
  # Verifica se a coluna 'Model' já existe no DataFrame
  if "Model" in df.columns:
    # Se sim, retorna o DataFrame sem alterações
    return df
  # Verifica se a coluna 'Modelo' existe no DataFrame
  elif "Modelo" in df.columns:
    # Se sim, renomeia a coluna 'Modelo' para 'MODELO'
    df = df.rename(columns={"Modelo": "MODELO"})
    # Cria uma nova coluna 'Model' vazia
    df["Model"] = ''
    # Retorna o DataFrame com a coluna 'Model' adicionada
    return df
  # Verifica se a coluna 'Subitem: Modelo' existe no DataFrame
  elif "Subitem: Modelo" in df.columns:
    # Se sim, renomeia a coluna 'Subitem: Modelo' para 'MODELO'
    df = df.rename(columns={"Subitem: Modelo": "MODELO"})
    # Cria uma nova coluna 'Model' vazia
    df["Model"] = ''
    # Retorna o DataFrame com a coluna 'Model' adicionada
    return df

  # Se nenhuma das colunas existir, cria uma nova coluna 'Model' vazia
  df["Model"] = ''

  # Retorna o DataFrame com a coluna 'Model' adicionada
  return df

In [11]:
def criar_coluna_range(df):
  """Cria uma coluna 'Range' no DataFrame se ela não existir.

  Verifica se a coluna 'Range' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'Range' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Range' adicionada, se necessário.
  """
  # Verifica se a coluna 'Range' já existe no DataFrame
  if "Range" in df.columns:
    # Se sim, retorna o DataFrame sem alterações
    return df

  # Se a coluna 'Range' não existir, cria uma nova coluna 'Range' com valores vazios
  df["Range"] = ''

  # Retorna o DataFrame com a coluna 'Range' adicionada
  return df

In [12]:
def criar_coluna_year_month(df):
  """Cria colunas 'Year' e 'Month' a partir da coluna de data.

  Identifica a coluna de data no DataFrame, que pode ter diferentes nomes
  ('Data', 'Data de Declaração', 'Date', 'Fecha', 'Declaration Date').
  Converte a coluna de data para o tipo datetime.
  Extrai o ano e o mês da coluna de data e cria as colunas 'Year' e 'Month'.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com as colunas 'Year' e 'Month' adicionadas.
  """
  # Verifica se a coluna de data existe no DataFrame, usando diferentes nomes possíveis
  if "Data" in df.columns:
    coluna_data = "Data"  # Define o nome da coluna de data se encontrada
  elif "Data de Declaração" in df.columns:
    coluna_data = "Data de Declaração"
  elif "Date" in df.columns:
    coluna_data = "Date"
  elif "Fecha" in df.columns:
    coluna_data = "Fecha"
  elif "Declaration Date" in df.columns:
    coluna_data = "Declaration Date"
  else:
    # Se a coluna de data não for encontrada,  retorna o df sem alteração
    return df

  # Converte a coluna de data para o tipo datetime, tratando erros de conversão
  df[coluna_data] = pd.to_datetime(df[coluna_data], errors='coerce')

  # Extrai o ano e o mês da coluna de data e cria as colunas 'Year' e 'Month'
  df["Year"] = df[coluna_data].dt.year  # Cria a coluna 'Year' com o ano
  df["Month"] = df[coluna_data].dt.month  # Cria a coluna 'Month' com o mês

  # Retorna o DataFrame com as colunas 'Year' e 'Month' adicionadas
  return df

In [13]:
def criar_coluna_quarter(df):
  """Cria uma coluna 'Quarter CY' no DataFrame, representando o trimestre do ano.

  Converte a coluna 'Month' para numérica,
  define condições para identificar cada trimestre com base no mês,
  e usa np.select para atribuir os valores 'Q1', 'Q2', 'Q3' ou 'Q4' à coluna 'Quarter CY'.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Quarter CY' adicionada.
  """
  # Converte a coluna 'Month' para numérica
  df['Month'] = pd.to_numeric(df['Month'], errors='coerce')

  # Define as condições para identificar cada trimestre com base no mês
  conditions = [
      df['Month'].isin([1, 2, 3]),  # Condição para o primeiro trimestre (Q1)
      df['Month'].isin([4, 5, 6]),  # Condição para o segundo trimestre (Q2)
      df['Month'].isin([7, 8, 9]),  # Condição para o terceiro trimestre (Q3)
      df['Month'].isin([10, 11, 12]),  # Condição para o quarto trimestre (Q4)
  ]

  # Define os valores correspondentes a cada trimestre
  quarters = ['Q1', 'Q2', 'Q3', 'Q4']

  # Usa np.select para atribuir os valores dos trimestres à coluna 'Quarter CY'
  df['Quarter CY'] = np.select(conditions, quarters, default='')

  # Retorna o DataFrame com a coluna 'Quarter CY' adicionada
  return df

In [14]:
def criar_coluna_samples(df):
  """Cria uma coluna 'Samples' no DataFrame com base no volume.

  Converte a coluna 'Volume' para numérica, tratando erros de conversão.
  Cria a coluna 'Samples', atribuindo "Samples" se o volume for menor ou igual a 500,
  e "Sales" caso contrário.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Samples' adicionada.
  """
  # Converte a coluna 'Volume' para numérica, tratando erros de conversão
  df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')

  # Cria a coluna 'Samples' usando uma função lambda para classificar com base no volume
  df["Samples"] = df['Volume'].apply(lambda x: "Samples" if x <= 500 else "Sales")  # Atribui "Samples" se volume <= 500, senão "Sales"

  # Retorna o DataFrame com a coluna 'Samples' adicionada
  return df

In [15]:
def criar_coluna_vertical(df):
  """Cria uma coluna 'Vertical' no DataFrame se ela não existir.

  Verifica se a coluna 'Vertical' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'Vertical' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Vertical' adicionada, se necessário.
  """
  # Verifica se a coluna 'Vertical' já existe no DataFrame
  if 'Vertical' not in df.columns:
    # Se a coluna 'Vertical' não existir, cria uma nova coluna 'Vertical' com valores vazios
    df['Vertical'] = ''
  # Retorna o DataFrame com a coluna 'Vertical' adicionada ou sem alterações, se já existir
  return df

In [16]:
def criar_coluna_manufacturer_category(df):
  """Cria uma coluna 'Compressor Manufacturer Category' no DataFrame se ela não existir.

  Verifica se a coluna 'Compressor Manufacturer Category' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'Compressor Manufacturer Category' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Compressor Manufacturer Category' adicionada, se necessário.
  """
  # Verifica se a coluna 'Compressor Manufacturer Category' já existe no DataFrame
  if 'Compressor Manufacturer Category' not in df.columns:
    # Se a coluna 'Compressor Manufacturer Category' não existir, cria uma nova coluna 'Compressor Manufacturer Category' com valores vazios
    df['Compressor Manufacturer Category'] = ''
  # Retorna o DataFrame com a coluna 'Compressor
  return df

In [17]:
def criar_coluna_segmento(df):
  """Cria uma coluna 'Segment' no DataFrame se ela não existir.

  Verifica se a coluna 'Segment' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'Segment' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' adicionada, se necessário.
  """
  # Verifica se a coluna 'Segment' já existe no DataFrame
  if 'Segment' not in df.columns:
    # Se a coluna 'Segment' não existir, cria uma nova coluna 'Segment' com valores vazios
    df['Segment'] = ''
  # Retorna o DataFrame com a coluna 'Segment' adicionada ou sem alterações, se já existir
  return df

In [18]:
def criar_coluna_cap_range(df):
  """Cria uma coluna 'CAP Range' no DataFrame se ela não existir.

  Verifica se a coluna 'CAP Range' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'CAP Range' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'CAP Range' adicionada, se necessário.
  """
  # Verifica se a coluna 'CAP Range' já existe no DataFrame
  if 'CAP Range' not in df.columns:
    # Se a coluna 'CAP Range' não existir, cria uma nova coluna 'CAP Range' com valores vazios
    df['CAP Range'] = ''
  # Retorna o DataFrame com a coluna 'CAP Range' adicionada ou sem alterações, se já existir
  return df

def criar_coluna_cop_range(df):
  """Cria uma coluna 'COP Range' no DataFrame se ela não existir.

  Verifica se a coluna 'COP Range' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'COP Range' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'COP Range' adicionada, se necessário.
  """
  # Verifica se a coluna 'COP Range' já existe no DataFrame
  if 'COP Range' not in df.columns:
    # Se a coluna 'COP Range' não existir, cria uma nova coluna 'COP Range' com valores vazios
    df['COP Range'] = ''
  # Retorna o DataFrame com a coluna 'COP Range' adicionada ou sem alterações, se já existir
  return df

In [19]:
def criar_coluna_market_aviability(df):
  """Cria uma coluna 'Market Aviability' no DataFrame se ela não existir.

  Verifica se a coluna 'Market Aviability' já existe no DataFrame.
  Se existir, retorna o DataFrame original sem alterações.
  Caso contrário, cria uma nova coluna 'Market Aviability' com valores vazios.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Market Aviability' adicionada, se necessário.
  """
  # Verifica se a coluna 'Market Aviability' já existe no DataFrame
  if 'Market Aviability' not in df.columns:
    # Se a coluna 'Market Aviability' não existir, cria uma nova coluna 'Market Aviability' com valores vazios
    df['Market Aviability'] = ''
  # Retorna o DataFrame com a coluna 'Market Aviability' adicionada ou sem alterações, se já existir
  return df

In [20]:
def criar_coluna_unit_price(df):
  """Cria ou calcula a coluna 'Unit Price (USD)' no DataFrame.

  Se a coluna 'Unit Price (USD)' já existir, calcula o preço unitário dividindo 'Value (USD)' por 'Volume'.
  Se a coluna 'Unit Price', 'Unit Value (USD)' ou 'Valor Unitário (USD)' existir, renomeia para 'Unit Price (USD)'.
  Se a coluna 'Unit Price (USD)' não existir após as verificações anteriores, calcula o preço unitário
  dividindo 'Value (USD)' por 'Volume' e cria a coluna.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Unit Price (USD)' criada ou calculada.
  """
  # Se a coluna 'Unit Price (USD)' já existir, calcula o preço unitário
  if 'Unit Price (USD)' in df.columns:
    df['Unit Price (USD)'] = df['Value (USD)'] / df['Volume']  # Calcula o preço unitário
    return df
  # Se a coluna 'Unit Price' existir, renomeia para 'Unit Price (USD)'
  if 'Unit Price' in df.columns:
    df = df.rename(columns={'Unit Price': 'Unit Price (USD)'})
  # Se a coluna 'Unit Value (USD)' existir, renomeia para 'Unit Price (USD)'
  if 'Unit Value (USD)' in df.columns:
    df = df.rename(columns={'Unit Value (USD)': 'Unit Price (USD)'})
  # Se a coluna 'Valor Unitário (USD)' existir, renomeia para 'Unit Price (USD)'
  elif 'Valor Unitário (USD)' in df.columns:
    df = df.rename(columns={'Valor Unitário (USD)': 'Unit Price (USD)'})
  # Se a coluna 'Unit Price (USD)' ainda não existir, calcula o preço unitário e cria a coluna
  if 'Unit Price (USD)' not in df.columns:
    df['Unit Price (USD)'] = df['Value (USD)'] / df['Volume']  # Calcula o preço unitário
  # Retorna o DataFrame com a coluna 'Unit Price (USD)' criada ou calculada
  return df

In [21]:
# Define um dicionario que mapeia o nome dos arquivos do drive com o nome do país de fato
# O nome do pais será o nome do arquivo de output no fim
country_mapping = {
  'SRI LANKA.xlsx' : 'Sri Lanka',
  'BANGLADESH.xlsx' : 'Bangladesh',
  'INDONESIA.xlsx' : 'Indonesia',
  'PHILIPPINES.xlsx' : 'Philippines',
  'MEXICO.xlsx' : 'Mexico',
  'INDIA.xlsx' : 'India',
  'INDIA 841480.xlsx' : 'India',
  'PAKISTAN.xlsx' : 'Pakistan',
  'AUSTRIA.xlsm' : 'Austria',
  'EGYPT.xlsx' : 'Egypt',
  'ETHIOPIA.xlsx' : 'Ethiopia',
  'FRANCE.xlsx' : 'France',
  'GERMANY.xlsx' : 'Germany',
  'IRAQ.xlsx' : 'Iraq',
  'ITALY.xlsm' : 'Italy',
  'KENYA.xlsx' : 'Kenya',
  'LESOTHO.xlsx' : 'Lesotho',
  'NIGERIA.xlsx' : 'Nigeria',
  'PORTUGAL.xlsx' :  'Portugal',
  'RUSSIA.xlsx' : 'Russia',
  'SAUDI ARABIA.xlsx' : 'Saudi',
  'TURKEY.xlsx' : 'Turkey',
  'UAE.xlsx' : 'UAE',
  'UGANDA.xlsx' : 'Uganda',
  'UK.xlsx' : 'UK',
  'UKRAINE.xlsx' : 'Ukraine',
  'UZBEKISTAN.xlsx' : 'Uzbekistan',
  'ZIMBABWE.xlsx' : 'Zimbabwe',
  'ARGENTINA.xlsx' : 'Argentina',
  'CHILE.xlsx' : 'Chile',
  'COLOMBIA.xlsx' : 'Colombia',
  'PERU.xlsx' : 'Peru',
  'ECUADOR.xlsx' : 'Ecuador',
  'PANAMA.xlsx' : 'Panama',
  'PARAGUAY.xlsx' : 'Paraguay',
  'SAUDI.xlsx' : 'Saudi',
  'VIETNAM.xlsx' : 'Vietnam',
  'USA.xlsx' : 'USA',
  'URUGUAY.xlsx' : 'Uruguay',
  'COSTARICA.xlsx' : 'Costa Rica',
  'BOLIVIA.xlsx' : 'Bolivia',
  'KAZAKHSTAN.xlsx' : 'Kazakhstan'
  }


In [22]:
# Obtém o nome do arquivo a partir do caminho completo do arquivo escolhido
file_name = os.path.basename(arquivo_escolhido)
# Obtém o nome do país correspondente ao nome do arquivo usando o dicionário country_mapping
country_name = country_mapping.get(file_name)

def criar_coluna_aduana(df):
  """Cria uma coluna 'ADUANA' no DataFrame com o nome do país.

  Utiliza a variável global 'country_name' para preencher a coluna 'ADUANA'
  com o nome do país correspondente ao arquivo de dados.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'ADUANA' adicionada.
  """
  # Adiciona a coluna 'ADUANA' ao DataFrame com o valor de 'country_name'
  df['ADUANA'] = country_name
  # Retorna o DataFrame com a coluna 'ADUANA' adicionada
  return df

In [23]:
def cria_dicionario_segmento():
    """Carrega o dicionário de segmentos de mercado do arquivo Excel.

    Lê o arquivo Excel 'SEGMENT DICTIONARY.xlsx' e retorna um DataFrame
    contendo os segmentos de mercado.

    Args:
        None

    Returns:
        pandas.DataFrame: DataFrame contendo os segmentos de mercado.
    """
    # Lê o arquivo Excel 'SEGMENT DICTIONARY.xlsx'
    segmentos = pd.read_excel('/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Utils/SEGMENT DICTIONARY.xlsx')
    # Retorna o DataFrame 'segmentos'
    return segmentos

### 0.2.2. FUNÇÕES PARA PADRONIZAR AS COLUNAS

In [24]:
def padronizar_coluna_hs_code(df):
  """Padroniza o nome da coluna de código HS.

  Verifica se a coluna 'HS Code' já existe no DataFrame.
  Se existir, retorna o DataFrame sem alterações.
  Caso contrário, procura por colunas com nomes alternativos
  ('Código HS', 'Código SH') e renomeia a primeira encontrada para 'HS Code'.
  Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'HS Code' vazia.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'HS Code' padronizada.
  """
  # Verifica se a coluna 'HS Code' já existe
  if "HS Code" in df.columns:
    return df

  # Procura por colunas com nomes alternativos e renomeia se encontrar
  for coluna in ["Código HS", "Código SH"]:
    if coluna in df.columns:
      df = df.rename(columns={coluna: "HS Code"})
      break  # Sai do loop após renomear a primeira coluna encontrada

  # Se a coluna 'HS Code' ainda não existir, cria uma nova coluna vazia
  if "HS Code" not in df.columns:
    df["HS Code"] = ""

  # Retorna o DataFrame com a coluna 'HS Code' padronizada
  return df

In [25]:
def padronizar_coluna_importer(df):
  """Padroniza o nome da coluna de importador.

  Verifica se a coluna 'Importer Raw' já existe no DataFrame.
  Se existir, retorna o DataFrame sem alterações.
  Caso contrário, procura por colunas com nomes alternativos
  e renomeia a primeira encontrada para 'Importer Raw'.
  Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Importer Raw' vazia.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Importer Raw' padronizada.
  """
  # Verifica se a coluna 'Importer Raw' já existe no DataFrame
  if "Importer Raw" in df.columns:
    return df

  if country_name == 'Kazakhstan':

    df = df.rename(columns={'Importer': 'Importer Raw'})

    df['Importer Raw'] = df['Importer Raw'].astype(str).replace('nan', '')
    df.loc[(df['Importer Raw'] == '') | (df['Importer Raw'].isna()), 'Importer Raw'] = df['Importer Declared']

    return df

  # Procura por colunas com nomes alternativos e renomeia se encontrar
  for coluna in ["Importer Declared", "Importador Declarado", "Posible Importador", "Importador", "Importer", "Possible Importer", "Consignee (Consolidated)"]:
    if coluna in df.columns:
      df = df.rename(columns={coluna: "Importer Raw"})
      break  # Sai do loop após renomear a primeira coluna encontrada

  # Se a coluna 'Importer Raw' ainda não existir, cria uma nova coluna vazia
  if "Importer Raw" not in df.columns:
    df["Importer Raw"] = ""

  # Retorna o DataFrame com a coluna 'Importer Raw' padronizada
  return df

In [26]:
def padronizar_coluna_country(df):
  """Padroniza o nome da coluna de país de origem.

  Verifica se a coluna 'Country Raw' já existe no DataFrame.
  Se existir, retorna o DataFrame sem alterações.
  Caso contrário, procura por colunas com nomes alternativos
  ('País de Origem', 'País de Origen', 'Country of Origin', 'Origin Country', 'Origin_Country')
  e renomeia a primeira encontrada para 'Country Raw'.
  Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Country Raw' vazia.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Country Raw' padronizada.
  """
  # Verifica se a coluna 'Country Raw' já existe no DataFrame
  if "Country Raw" in df.columns:
    return df

  # Procura por colunas com nomes alternativos e renomeia se encontrar
  for coluna in ["País de Origem", "País de Origen", "Country of Origin", "Origin Country", "Origin_Country"]:
    if coluna in df.columns:
      df = df.rename(columns={coluna: "Country Raw"})
      break  # Sai do loop após renomear a primeira coluna encontrada

  # Se a coluna 'Country Raw' ainda não existir, cria uma nova coluna vazia
  if "Country Raw" not in df.columns:
    df["Country Raw"] = ""

  # Retorna o DataFrame com a coluna 'Country Raw' padronizada
  return df

In [27]:
def padronizar_coluna_player_raw(df):
  """Padroniza o nome da coluna de player raw.

  Verifica se a coluna 'Player Raw' já existe no DataFrame.
  Se existir, retorna o DataFrame sem alterações.
  Caso contrário, se o nome do país for 'Ecuador', renomeia a coluna 'Supplier Declared' para 'Player Raw'.
  Se o nome do país não for 'Ecuador', procura por colunas com nomes alternativos
  e renomeia a primeira encontrada para 'Player Raw'.
  Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Player Raw' vazia.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Player Raw' padronizada.
  """
  # Verifica se a coluna 'Player Raw' já existe no DataFrame
  if "Player Raw" in df.columns:
    return df

  # Caso especial para o Equador
  if country_name == 'Ecuador':
    df = df.rename(columns={'Supplier Declared': 'Player Raw'})
    return df

  if country_name == 'Bolivia':
    df = df.rename(columns={'Supplier Declared': 'Player Raw'})
    return df

  if country_name == 'Uzbekistan':
    df = df.rename(columns={'Supplier Declared': 'Player Raw'})
    return df

  if country_name == 'Peru':
    df = df.rename(columns={'Brand': 'Player Raw'})
    return df

  if country_name == 'Paraguay':
    df = df.rename(columns={'Brand': 'Player Raw'})

    df['Player Raw'] = df['Player Raw'].astype(str).replace('nan', '')
    df.loc[(df['Player Raw'] == '') | (df['Player Raw'].isna()), 'Player Raw'] = df['Supplier']

    return df

  if country_name == 'Kazakhstan':
    df = df.rename(columns={'Supplier': 'Player Raw'})

    return df

  # Procura por colunas com nomes alternativos e renomeia se encontrar
  for coluna in ["Subitem: Brand", "Brand","Exporter", "Supplier", "Proveedor Declarado", "Proveedor", "Provedor Declarado", "Provider Declared",
                 "Provedor", "Fornecedor Declarado", "Fornecedor", "Subitem: Griffe/Marca", "Marca", "Shipper (Unified)"]:
    if coluna in df.columns:
      df = df.rename(columns={coluna: "Player Raw"})
      break  # Sai do loop após renomear a primeira coluna encontrada

  # Se a coluna 'Player Raw' ainda não existir, cria uma nova coluna vazia
  if "Player Raw" not in df.columns:
    df["Player Raw"] = ""

  # Retorna o DataFrame com a coluna 'Player Raw' padronizada
  return df

  # if country_name == 'Paraguay':
  #   # Fill empty 'Player Raw' values from 'Supplier' if 'Supplier' column exists
  #   if 'Supplier' in df.columns:
  #     # Ensure 'Player Raw' is string type for consistent empty string comparison
  #     df['Player Raw'] = df['Player Raw'].astype(str)
  #     # Fill empty or NaN values in 'Player Raw' with corresponding 'Supplier' values
  #     df.loc[(df['Player Raw'] == '') | (df['Player Raw'].isna()), 'Player Raw'] = df['Supplier']

  #   # Retorna o DataFrame com a coluna 'Player Raw' padronizada
  #   return df

In [28]:
def padronizar_coluna_description(df, country_name):
    """Padroniza o nome da coluna de descrição.

    Verifica se a coluna 'Description' já existe no DataFrame.
    Se existir, retorna o DataFrame sem alterações.
    Caso contrário, procura por colunas com nomes alternativos e renomeia a primeira
    encontrada para 'Description'.
    Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Description' vazia.

    Args:
        df: DataFrame de entrada.

    Returns:
        DataFrame com a coluna 'Description' padronizada.
    """

    # Verifica se a coluna 'Description' já existe no DataFrame

    if "Description" in df.columns:
        return df

    if country_name == 'Ecuador':
        df = df.rename(columns={"Commercial Description": "Description"})

    if country_name == 'Uruguay':
        df = df.rename(columns={"Merchandise Description": "Description"})

    else:
        for coluna in ["Merchandise Description", "Descrição Comercial", "Descrição da Mercadoria", "Descrição da mercadoria",
                      "Commercial Description", "Product Description", "Descrição", "Descripción", "Descripción Comercial", "Short Container Description", "Features", "Name", "Merchandise", "Merchandise Code"]:
            if coluna in df.columns:
                df = df.rename(columns={coluna: "Description"})
                break  # Sai do loop após renomear a primeira coluna encontrada

    # Se a coluna 'Description' ainda não existir, cria uma nova coluna vazia
    if "Description" not in df.columns:
        df["Description"] = ""

    # Retorna o DataFrame com a coluna 'Description' padronizada
    return df

In [29]:
def padronizar_coluna_value(df):
  """Padroniza o nome da coluna de valor e retorna o nome da coluna padronizada.

  Verifica se a coluna 'Value (USD)' já existe no DataFrame.
  Se existir, retorna o DataFrame e o nome da coluna sem alterações.
  Caso contrário, procura por colunas com nomes alternativos
  e renomeia a primeira encontrada para 'Value (USD)'.
  Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Value (USD)' com valores 0.

  Args:
    df: DataFrame de entrada.

  Returns:
    Tuple: DataFrame com a coluna 'Value (USD)' padronizada e o nome da coluna padronizada.
           - Se a coluna já existir ou for renomeada, retorna o nome da coluna 'Value (USD)'.
           - Se a coluna for criada, retorna o nome da última coluna verificada na lista de nomes alternativos.
  """
  # Verifica se a coluna 'Value (USD)' já existe no DataFrame
  if "Value (USD)" in df.columns:
    coluna = 'Value (USD)'  # Define o nome da coluna como 'Value (USD)'
    return df, coluna  # Retorna o DataFrame e o nome da coluna

  # Procura por colunas com nomes alternativos e renomeia se encontrar
  for coluna in ["Subitem: Calculated FOB Value (US$)", "FOB Value (USD)", "Calculated FOB Value (USD)", "Value", "Valor Total FOB US$",
                 "Valor FOB (USD)", "FOB Calculado  (USD)", "Valor (USD)", "Total Value USD (without $)",
                 "Total Value (USD)", "Total Value USD", "Preco da aduana UAH", "Preço Aduana (USD)", "CIF Value [USD]", "Valor CIF (USD)", "U$S CIF", "Customs Value (USD)", "CIF Value (USD)", "Invoice Value (USD)"]:
    if coluna in df.columns:
      df = df.rename(columns={coluna: "Value (USD)"})  # Renomeia a coluna para 'Value (USD)'
      break  # Sai do loop após renomear a primeira coluna encontrada

  # Se a coluna 'Value (USD)' ainda não existir, cria uma nova coluna com valores 0
  if "Value (USD)" not in df.columns:
    df["Value (USD)"] = 0  # Cria a coluna 'Value (USD)' com valores 0

  # Retorna o DataFrame e o nome da coluna padronizada ou da última coluna verificada
  return df, coluna

  # A IDEIA É RETORNAR A COLUNA VALUE TAMBÉM PARA QUE POSSAMOS USAR LÁ NA FRENTE PARA
  # SABER SE OS VALORES SÃO CIF OU FOB

In [30]:
def padronizar_coluna_volume(df):
    """Padroniza o nome da coluna de volume.

    Verifica se a coluna 'Volume' já existe no DataFrame.
    Se existir, retorna o DataFrame sem alterações.
    Caso contrário, procura por colunas com nomes alternativos e renomeia a primeira
    encontrada para 'Volume'.
    Se nenhuma coluna com esses nomes for encontrada, cria uma nova coluna 'Volume' com valor 0.

    Args:
        df: DataFrame de entrada.

    Returns:
        DataFrame com a coluna 'Volume' padronizada.
    """
    # Procura por colunas com nomes alternativos e renomeia se encontrar
    for coluna in ["Subitem: Quantity", "Quantity", "Amount of Unit", "Quantidade", "Cantidad",
                   "Cantidad Comercial", "Quantidade Comercial", "Commercial Quantity"]:
        if coluna in df.columns:
            df = df.rename(columns={coluna: "Volume"})
            break  # Sai do loop após renomear a primeira coluna encontrada

    # Se a coluna 'Volume' ainda não existir, cria uma nova coluna com valor 0
    if "Volume" not in df.columns:
        df["Volume"] = 0

    # Retorna o DataFrame com a coluna 'Volume' padronizada
    return df

### 0.2.3. FUNÇÕES PARA PREENCHER SEGMENT

In [31]:
def filter_importer_by_segment(df, keywords, segment):
  """Filtra importadores por segmento com base em palavras-chave.

  Converte a coluna 'Importer Final' para string.
  Cria um padrão de regex a partir das palavras-chave fornecidas.
  Filtra o DataFrame para linhas onde a coluna 'Importer Final' contém o padrão.
  Atribui o segmento fornecido à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.
    keywords: Lista de palavras-chave para filtrar a coluna 'Importer Final'.
    segment: Segmento a ser atribuído às linhas filtradas.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Converte a coluna 'Importer Final' para string
  df['Importer Final'] = df['Importer Final'].astype(str)
  # Cria um padrão de regex a partir das palavras-chave que a funcao recebe como parametro
  pattern = '|'.join(keywords)
  # Filtra o DataFrame e atribui o segmento à coluna 'Segment'
  df.loc[df['Importer Final'].str.contains(pattern, case=False, regex=True, na=False), 'Segment'] = segment
  # Retorna o DataFrame modificado
  return df

def importers_extra(df):
  """Filtra importadores do segmento 'EXTRA' com base em palavras-chave.

  Utiliza a função `filter_importer_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'EXTRA'.
  Atribui o segmento 'EXTRA' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Define a lista de palavras-chave relacionadas ao segmento 'EXTRA'
  keywords = [
      'nissan', 'kia motors', 'volkswagen', 'audi', 'toyota', 'hyundai', 'honda',
      'suzuki', 'mercedes', 'caterpillar', 'peugeot', 'schulz', 'chevrolet',
      'ford', 'volvo', 'bmw', 'mazda', 'renault', 'fiat',
      'camiones', 'sultana', 'tracto', 'marhen', 'vehiculos', 'automotive',
      'chrysler', 'general motors', 'moreno diesel', 'automobile',
      'eberspacher', 'john deere', 'boda',
      'icee de mexico', 'arrgo', 'valeo', 'scania','michelin', 'caloi', 'embraer',
      'wickbold', 'helicopteros', 'helicopter', 'jaguar', 'denso', 'truck', 'byd', 'bitzer'
  ]
  # Chama a função `filter_importer_by_segment` para filtrar o DataFrame e atribuir o segmento 'EXTRA'
  return filter_importer_by_segment(df, keywords, 'EXTRA')

def importers_ac(df):
  """Filtra importadores do segmento 'HVAC' com base em palavras-chave.

  Utiliza a função `filter_importer_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'HVAC'.
  Atribui o segmento 'HVAC' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Define a lista de palavras-chave relacionadas ao segmento 'HVAC'
  keywords = [
      'carrier', 'lennox', 'rheem', 'emerpowsys', 'air conditioning',
      'multiequipos climas y refacciones', 'aux imp', 'friedrich', 'rittal',
      'gree', 'carrier', 'emerson climate', 'air conditioner', 'daikin', 'hvac', 'hvacr', 'rechi','AIR CONDITIONING SPARES CENTRE'
  ]
  # Chama a função `filter_importer_by_segment` para filtrar o DataFrame e atribuir o segmento 'HVAC'
  return filter_importer_by_segment(df, keywords, 'HVAC')

def importers_ca(df):
  """Filtra importadores do segmento 'CA' com base em palavras-chave.

  Utiliza a função `filter_importer_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'CA'.
  Atribui o segmento 'CA' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Define a lista de palavras-chave relacionadas ao segmento 'CA'
  keywords = ['ojeda', 'metaplus', 'kitchen equipment','MANITOWOC']
  # Chama a função `filter_importer_by_segment` para filtrar o DataFrame e atribuir o segmento 'CA'
  return filter_importer_by_segment(df, keywords, 'CA')

In [32]:
def filter_player_by_segment(df, keywords, segment):
  """Filtra players por segmento com base em palavras-chave.

  Converte a coluna 'Player Final' para string.
  Cria um padrão de regex a partir das palavras-chave fornecidas.
  Filtra o DataFrame para linhas onde a coluna 'Player Final' contém o padrão.
  Atribui o segmento fornecido à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.
    keywords: Lista de palavras-chave para filtrar a coluna 'Player Final'.
    segment: Segmento a ser atribuído às linhas filtradas.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Converte a coluna 'Player Final' para string
  df['Player Final'] = df['Player Final'].astype(str)
  # Cria um padrão de regex a partir das palavras-chave
  pattern = '|'.join(keywords)
  # Filtra o DataFrame e atribui o segmento à coluna 'Segment'
  df.loc[df['Player Final'].str.contains(pattern, case=False, regex=True, na=False), 'Segment'] = segment
  # Retorna o DataFrame modificado
  return df

def players_extra(df):
  """Filtra players do segmento 'EXTRA' com base em palavras-chave.

  Utiliza a função `filter_player_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'EXTRA'.
  Atribui o segmento 'EXTRA' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  # Define a lista de palavras-chave relacionadas ao segmento 'EXTRA'
  keywords = [
      'nissan', 'kia motors', 'volkswagen', 'audi', 'toyota', 'hyundai', 'honda',
      'suzuki', 'mercedes', 'caterpillar', 'peugeot', 'schulz', 'chevrolet',
      'ford', 'volvo', 'bmw', 'mazda', 'renault', 'fiat',
      'camiones', 'sultana', 'tracto', 'marhen', 'vehiculos', 'automotive',
      'chrysler', 'general motors', 'moreno diesel', 'automobile',
      'eberspacher', 'john deere', 'boda',
      'icee de mexico', 'arrgo', 'makita', 'iveco', 'valeo', 'scania', 'michelin',
      'caloi', 'embraer', 'wickbold', 'helicoptero', 'jaguar', 'denso', 'byd','MERCEDE-BENZ', 'bitzer'
  ]
  # Chama a função `filter_player_by_segment` para filtrar o DataFrame e atribuir o segmento 'EXTRA'
  return filter_player_by_segment(df, keywords, 'EXTRA')

def players_ac(df):
  """Filtra players do segmento 'HVAC' com base em palavras-chave.

  Utiliza a função `filter_player_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'HVAC'.
  Atribui o segmento 'HVAC' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  keywords = [
      'lii united', 'rheem', 'emerpowsys', 'air conditioning',
      'multiequipos climas y refacciones', 'aux imp', 'friedrich', 'rittal', 'daikin', 'rechi', 'carrier', '\bgree\b', 'GREE (LANDA)'
  ]
  return filter_player_by_segment(df, keywords, 'HVAC')

def players_ca(df):
  """Filtra players do segmento 'CA' com base em palavras-chave.

  Utiliza a função `filter_player_by_segment` para filtrar o DataFrame `df`
  com base em uma lista de palavras-chave relacionadas ao segmento 'CA'.
  Atribui o segmento 'CA' à coluna 'Segment' para as linhas filtradas.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para as linhas filtradas.
  """
  keywords = ['ojeda', 'metaplus', 'kitchen equipment','MANITOWOC']
  return filter_player_by_segment(df, keywords, 'CA')

In [33]:
def ac_by_description(df):
  """Classifica registros como 'HVAC' com base na descrição.

  Verifica se a coluna 'Description' contém palavras-chave relacionadas a ar condicionado e HVAC.
  Se encontrar alguma correspondência, atribui o segmento 'HVAC' à coluna 'Segment'.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para 'HVAC' onde aplicável.
  """
  # Define a lista de palavras-chave relacionadas a ar condicionado e HVAC
  keywords = [
      'air conditioning', 'air-con','air conditioner','aire acondicionado','a/acondicionado',
      'a/conditioning','compresor para ac', 'compressor for a / c', 'ac compressor', 'acondicionado',
      'hvac', 'hvacr', 'air cond', 'airconditioner', 'airconditioning', 'air-conditioner', 'aircon'
  ]
  # Cria uma máscara booleana indicando as linhas onde a descrição contém as palavras-chave
  mask = df['Description'].str.contains('|'.join(keywords), regex=True, case=False, na=False)
  # Atribui o segmento 'HVAC' às linhas correspondentes à máscara
  df.loc[mask, 'Segment'] = 'HVAC'
  # Retorna o DataFrame modificado
  return df

In [34]:
def extra_by_description(df):
  """Classifica registros como 'EXTRA' com base na descrição.

  Verifica se a coluna 'Description' contém palavras-chave relacionadas a produtos do segmento EXTRA.
  Se encontrar alguma correspondência, atribui o segmento 'EXTRA' à coluna 'Segment'.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para 'EXTRA' onde aplicável.
  """
  keywords = [
      'automobile','extractor', 'air pump','ventilador','ventiladores',
      '\bfan\b', 'fans', 'automotive', 'aircraft', 'air compressor',
      'compresor de aire', 'air drain pump','sopladores','bomba de vacio',
      'bombas de vacio','automotriz', 'automotivo', 'radiador', 'veiculo',
      'vehicle', 'carro', 'freio', 'persiana', 'escavadora', 'engrenagem',
      'bicicleta', 'bicycle', 'helicopter', 'caminhao', 'aeronave', 'truck',
      'air comp', 'auto parts', 'inflador', 'compressores de aire', 'compresores de aire',
      'earthmoving', 'crane', 'VOLVO','NISSAN', 'KIA MOTORS', 'VOLKSWAGEN', 'AUDI', 'TOYOTA',
      'HYUNDAI', 'HONDA', 'SUZUKI', 'MERCEDES', 'CATERPILLAR', 'PEUGEOT', 'SCHULZ',
      'CHEVROLET', 'FORD', 'BMW', 'MAZDA', 'RENAULT', 'FIAT',
      'camiones', 'sultana', 'tracto', 'marhen', 'vehiculos', 'automotive',
      'chrysler', 'general motors', 'moreno diesel', 'eberspacher', 'johnbros',
      'caterpillar','VEHICULO','bitzer','COMPRESOR DE AUTO','EXCAVADORA','corsa','parabris','blood','tyre','TURBOCHARGE',
      'SWIMMING','MOBILE PHONE','TURBOCHARGER', 'BRAKE BLEEDING','BLOOD','CAR INFLATOR','TURBO CHARGER', 'AUTO A/C', 'CAR ACCESSORIES', 'INTERCOOLE',
      'HALLSCREW', 'HALLSREW'
  ]

  keywords2 = [
      'automobile'
  ]

  keywords3 = [
      'automobile','extractor','ventilador','ventiladores',
      '\bfan\b', 'fans', 'automotive', 'aircraft','air drain pump',
      'sopladores','bomba de vacio', 'bombas de vacio','automotriz', 'automotivo', 'radiador', 'veiculo',
      'vehicle', 'carro', 'freio', 'persiana', 'escavadora', 'engrenagem',
      'bicicleta', 'bicycle', 'helicopter', 'caminhao', 'aeronave', 'truck',
      'auto parts', 'inflador','earthmoving', 'crane', 'VOLVO',
      'NISSAN', 'KIA MOTORS', 'VOLKSWAGEN', 'AUDI', 'TOYOTA',
      'HYUNDAI', 'HONDA', 'SUZUKI', 'MERCEDES', 'CATERPILLAR', 'PEUGEOT', 'SCHULZ',
      'CHEVROLET', 'FORD', 'BMW', 'MAZDA', 'RENAULT', 'FIAT',
      'camiones', 'sultana', 'tracto', 'marhen', 'vehiculos', 'automotive',
      'chrysler', 'general motors', 'moreno diesel', 'eberspacher', 'johnbros',
      'caterpillar','VEHICULO','bitzer','COMPRESOR DE AUTO','EXCAVADORA','corsa','parabris','blood','tyre','TURBOCHARGE',
      'SWIMMING','MOBILE PHONE','TURBOCHARGER', 'BRAKE BLEEDING','BLOOD','CAR INFLATOR','TURBO CHARGER', 'AUTO A/C', 'CAR ACCESSORIES', 'INTERCOOLE'
  ]

  if country_name == 'Vietnam':

    mask = df['Description'].str.contains('|'.join(keywords3), regex = True, case=False, na=False)
    df.loc[mask, 'Segment'] = 'EXTRA'

  else:
    mask = df['Description'].str.contains('|'.join(keywords), regex = True, case=False, na=False)
    df.loc[mask, 'Segment'] = 'EXTRA'

  return df

def wet_by_description(df):
  """Classifica registros como 'WET' com base na descrição.

  Verifica se a coluna 'Description' contém palavras-chave relacionadas a produtos do segmento WET.
  Se encontrar alguma correspondência, atribui o segmento 'WET' à coluna 'Segment'.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada para 'WET' onde aplicável.
  """
  keywords = [ 'dryer', 'DRIER' ]

  mask = df['Description'].str.contains('|'.join(keywords), regex = True, case=False, na=False)

  df.loc[mask, 'Segment'] = 'WET'

  return df

In [35]:
def segment_by_hscode(df):
  """Classifica registros por segmento com base no código HS, específico para o México.

  Se o nome do país for 'Mexico', a função atribui segmentos 'HVAC' ou 'EXTRA'
  com base nos códigos HS especificados.
  Caso contrário, retorna o DataFrame original sem alterações.

  Args:
    df: DataFrame de entrada.

  Returns:
    DataFrame com a coluna 'Segment' atualizada, se aplicável.
  """
  if country_name == 'Mexico':
    # Atribui 'HVAC' para códigos HS específicos
    df['Segment'] = df.apply(lambda x: 'HVAC' if ((x['HS Code'] == 8414300800) | (x['HS Code'] == 84143008)) else x['Segment'], axis=1)
    # Atribui 'EXTRA' para códigos HS específicos
    df['Segment'] = df.apply(lambda x: 'EXTRA' if ((x['HS Code'] == 84143006) |
                                                   (x['HS Code'] == 84143003) |
                                                   (x['HS Code'] == 84141006) |
                                                   (x['HS Code'] == 84148099) |
                                                   (x['HS Code'] == 84146001)) else x['Segment'], axis=1)
    return df  # Retorna o DataFrame modificado
  else:
    return df  # Retorna o DataFrame original sem alterações

In [36]:
def intracompany_segment(df):
    """Classifica registros como 'INTRACOMPANY' se player e importador forem da mesma empresa.

    Verifica se o 'Player Final' e o 'Importer Final' são iguais a 'NIDEC', 'TECUMSEH' ou 'DANFOSS'.
    Se forem iguais, a função atribui o segmento 'INTRACOMPANY' à coluna 'Segment'.
    Caso contrário, retorna o valor original da coluna 'Segment'.

    Args:
        df: DataFrame de entrada.

    Returns:
        str: O segmento classificado como 'INTRACOMPANY' ou o valor original da coluna 'Segment'.
    """
    # Verifica se player e importador são 'NIDEC'
    if df['Player Final'] == 'NIDEC' and df['Importer Final'] == 'NIDEC':
        return 'INTRACOMPANY'
    # Verifica se player e importador são 'TECUMSEH'
    elif df['Player Final'] == 'TECUMSEH' and df['Importer Final'] == 'TECUMSEH':
        return 'INTRACOMPANY'
    # Verifica se o importador é 'DANFOSS'
    elif df['Player Final'] == 'DANFOSS' and df['Importer Final'] == 'DANFOSS':
        return 'INTRACOMPANY'

    elif df['Player Final'] == 'COPELAND' and df['Importer Final'] == 'COPELAND':
        return 'INTRACOMPANY'
    else:
        return df['Segment']

### 0.2.4. FUNÇÕES PARA PREENCHER RANGE

In [37]:
def preenche_range(df, mask):
    """Preenche a coluna 'Range' com 'Out of Range' para as linhas que correspondem à máscara.

    Cria uma cópia do DataFrame de entrada para evitar modificar o original.
    Aplica a máscara à coluna 'Range' do DataFrame copiado,
    definindo o valor como 'Out of Range' para as linhas onde a máscara é True.

    Args:
        df: O DataFrame de entrada.
        mask: Uma máscara booleana indicando as linhas a serem modificadas.

    Returns:
        Uma cópia do DataFrame com a coluna 'Range' atualizada.
    """
    # Cria uma cópia do DataFrame
    filtered_df = df.copy()
    # Aplica a máscara à coluna 'Range'
    filtered_df.loc[mask, 'Range'] = 'Out of Range'
    # Retorna a cópia do DataFrame modificado
    return filtered_df

# A IDEIA É PASSAR UMA MASCARA (FILTRO) PARA ESSA FUNÇÃO E TUDO QUE CAIR NO FILTRO
# VIRA OUT OF RANGE

In [38]:
def range_by_price(df4):
  """Classifica os registros como 'Out of Range' com base no preço unitário.

  Converte a coluna 'Unit Price (USD)' para numérica, tratando erros de conversão.
  Classifica os registros como 'Out of Range' se o preço unitário for maior que 300 ou menor que 10.
  Mantém a classificação original se o preço estiver dentro do intervalo.

  Args:
      df4: O DataFrame de entrada.

  Returns:
      O DataFrame com a coluna 'Range' atualizada.
  """
  # Converte a coluna 'Unit Price (USD)' para numérica
  df4['Unit Price (USD)'] = pd.to_numeric(df4['Unit Price (USD)'], errors='coerce')
  # Aplica a classificação 'Out of Range' com base no preço unitário
  df4['Range'] = df4.apply(lambda row: 'Out of Range' if row['Unit Price (USD)'] > 300 or row['Unit Price (USD)'] < 10 else row['Range'], axis=1)
  # Retorna o DataFrame com a coluna 'Range' atualizada
  return df4

In [39]:
def range_by_vertical(df):
  """Classifica os registros como 'Out of Range' com base na vertical.

  Verifica se a coluna 'Vertical' contém o valor 'Vertical'.
  Se o valor for 'Vertical', classifica o registro como 'Out of Range' na coluna 'Range'.
  Caso contrário, mantém a classificação original da coluna 'Range'.

  Args:
    df: O DataFrame de entrada.

  Returns:
    O DataFrame com a coluna 'Range' atualizada.
  """
  # Aplica a classificação 'Out of Range' se a vertical for 'Vertical'
  df['Range'] = df.apply(lambda row: 'Out of Range' if row['Vertical'] == 'Vertical' else row['Range'], axis=1)
  # Retorna o DataFrame com a coluna 'Range' atualizada
  return df

In [40]:
def range_by_description(df):
    """Classifica os registros como 'Out of Range' com base na descrição.

    Verifica se a coluna 'Description' contém padrões específicos que indicam
    que o registro está fora do intervalo desejado.
    Se encontrar alguma correspondência, atribui o segmento 'Out of Range' à coluna 'Range'.

    Args:
        df: O DataFrame de entrada.

    Returns:
        Uma cópia do DataFrame com a coluna 'Range' atualizada.
    """
    # Define a lista de padrões a serem pesquisados na descrição
    lista_patterns = [
        r'a lot of assembly material', r'material ensamble', r'bomba de vacio', r'bombas de vacio',
        r'sopladores', r'soplador', r'carcasa', r'COMPRESOR DE AIRE', r'CARCASA DE ACERO', r'PARTE DEL COMPRESOR', r'air compressors',
        r'air drain pump', r'air compressor', r'starting device', r'carcasa', r'canaleta', r'condensadora de aluminio', r'curing oven', r'condensadora'
    ]

    # Cria uma cópia do DataFrame para evitar modificar o original
    filtered_df = df.copy()

    # Itera sobre os padrões e aplica a classificação 'Out of Range'
    for pattern in lista_patterns:
        # Cria uma máscara booleana para as linhas que correspondem ao padrão
        mask = filtered_df['Description'].str.contains(pattern, case=False, regex=True, na=False)
        # Atribui 'Out of Range' à coluna 'Range' para as linhas correspondentes
        filtered_df.loc[mask, 'Range'] = 'Out of Range'

    # Retorna o DataFrame modificado
    return filtered_df

### 0.2.5. FUNÇÕES PARA PREENCHER AS OUTRAS COLUNAS

In [41]:
def categorize_technology(description):
  """Categoriza a tecnologia do compressor com base na descrição.

  Analisa a descrição do produto em busca de palavras-chave que indicam a tecnologia do compressor.
  Retorna a categoria da tecnologia correspondente ou None se não encontrar nenhuma correspondência.

  Args:
      description: A descrição do produto.

  Returns:
      Uma string representando a categoria da tecnologia do compressor, ou None.
  """
  # Converte a descrição para string para evitar erros
  description = str(description)
  # Verifica se a descrição contém palavras-chave relacionadas a "FIXED SPEED"

  if country_name == 'Bangladesh':

    if re.search(r"FIXED|MONOF[ÁA]SICO|SINGLE[- ]?PHASE|LINEAR|FIJA|FIJO", description, re.IGNORECASE):
      return "FIXED SPEED"

    elif re.search(r"VARIADA|VARIABLE|VCC|VCCREFRIGERATOR|THREE PHASE", description, re.IGNORECASE):
      return "VARIABLE SPEED"

  elif re.search(r"FIXED|MONOF[ÁA]SICO|SINGLE[- ]?PHASE|LINEAR|FIJA|FIJO|WITHOUT INVERTER|WITHOUT INV.|WITHOUT INVE.|NON-INVERTER|NON INVERTER", description, re.IGNORECASE):
    return "FIXED SPEED"
  # Verifica se a descrição contém palavras-chave relacionadas a "VARIABLE SPEED"
  elif re.search(r"INVERTER|INVERT|VARIADA|VARIABLE|VCC|VCCREFRIGERATOR|THREE PHASE|INVERSOR|INVERETER", description, re.IGNORECASE):
    return "VARIABLE SPEED"
  # Se nenhuma correspondência for encontrada, retorna None
  else:
    return None

In [42]:
def categorize_compressor_type(description):
  description = str(description)
  # Verifica se a descrição contém palavras-chave relacionadas a "FIXED SPEED"
  if re.search(r"ROTATIVO|ROTARY|ROTRY", description, re.IGNORECASE):
    return "ROTARY"
  # Verifica se a descrição contém palavras-chave relacionadas a "SCROLL"
  elif re.search(r"SCROLL|CARACOL", description, re.IGNORECASE):
    return "SCROLL"
  # Verifica se a descrição contém palavras-chave relacionadas a "SCREW"
  elif re.search(r'SCREW|TORNILLO', description, re.IGNORECASE):
    return 'SCREW'
  elif re.search(r'RECIPROCATORY|RECIPROCATING|Reciprocating|RICIPROCATING|RECOPROCATING|RECIPROCATION|RECIPR', description, re.IGNORECASE):
    return 'RECIPROCATING'
  # Se nenhuma correspondência for encontrada, retorna None
  else:
    return None

In [43]:
def player_in_description(row):
  """Identifica o player com base na descrição

  Verifica se a descrição do produto contém palavras-chave que indicam
  a presença de um player específico (JIAXIPERA, GMCC ou EMBRACO).
  Se encontrar uma correspondência, retorna o nome do player correspondente.
  Caso contrário, retorna o valor original da coluna 'Player Final'.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    O nome do player identificado na descrição ou o valor original da coluna 'Player Final'.
  """
  # Converte a descrição para string
  description = str(row['Description'])
  # Verifica se a descrição contém "jiaxipera"
  if re.search(r"jiaxipera", description, re.IGNORECASE):
    return "JIAXIPERA"
  # Verifica se a descrição contém "meizhi"
  elif re.search(r"meizhi", description, re.IGNORECASE):
    return "GMCC"
  # Verifica se a descrição contém "embraco"
  elif re.search(r"embraco", description, re.IGNORECASE):
    return "Nidec"
  elif re.search(r"nidec", description, re.IGNORECASE):
    return "Nidec"
  # Se nenhuma correspondência for encontrada, retorna o valor original da coluna 'Player Final'
  else:
    return row['Player Final']

In [44]:
def encontrar_vertical(row):
  """Identifica se o registro pertence a uma vertical ou ao mercado aberto.

  Verifica se o importador e o player correspondem a uma das verticais conhecidas
  (LG, PANASONIC, SAMSUNG, GODREJ, MABE).
  Se houver correspondência, retorna 'Vertical', indicando que o registro pertence a uma vertical.
  Caso contrário, retorna 'Merchant', indicando que o registro pertence ao mercado aberto.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    Uma string indicando se o registro pertence a uma vertical ('Vertical') ou ao mercado aberto ('Merchant').
  """
  # Verifica se o importador e o player são LG
  if row['Importer Final'] == 'LG' and row['Player Final'] == 'LG':
    return 'Vertical'
  # Verifica se o importador e o player são PANASONIC
  elif row['Importer Final'] == 'PANASONIC' and row['Player Final'] == 'PANASONIC':
    return 'Vertical'
  # Verifica se o importador e o player são SAMSUNG
  elif row['Importer Final'] == 'SAMSUNG' and row['Player Final'] == 'SAMSUNG':
    return 'Vertical'
  # Verifica se o importador e o player são GODREJ
  elif row['Importer Final'] == 'GODREJ' and row['Player Final'] == 'GODREJ':
    return 'Vertical'
  # Verifica se o importador e o player são MABE
  elif row['Importer Final'] == 'MABE' and row['Player Final'] == 'MABE':
    return 'Vertical'
  # Se nenhuma correspondência for encontrada, retorna 'Merchant'
  else:
    return 'Merchant'

In [45]:
def preencher_market_aviability(row):
  """Determina a disponibilidade no mercado com base na tecnologia e no COP.

  Verifica a tecnologia do compressor e o COP (Coeficiente de Performance)
  para determinar se o produto está disponível no mercado.
  A lógica de verificação varia de acordo com o país.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    Uma string indicando a disponibilidade no mercado:
      - 'Yes': Disponível no mercado.
      - 'No': Não disponível no mercado.
      - '': Não foi possível determinar a disponibilidade.
  """
  # Verifica se o país é a Índia
  if country_name == 'India':
    # Se a tecnologia for 'VARIABLE SPEED', está disponível no mercado
    if row['Technology'] == 'VARIABLE SPEED':
      return 'Yes'
    # Se o COP for menor que 1.75, não está disponível no mercado
    elif row['COP (W/W)'] < 1.75:
      return 'No'
    # Se a tecnologia for 'FIXED SPEED' e o COP for maior ou igual a 1.75, está disponível no mercado
    elif row['Technology'] == 'FIXED SPEED' and row['COP (W/W)'] >= 1.75:
      return 'Yes'
    # Caso contrário, não foi possível determinar a disponibilidade
    else:
      return ''
  # Se o país não for a Índia
  else:
    # Se a tecnologia for 'VARIABLE SPEED', está disponível no mercado
    if row['Technology'] == 'VARIABLE SPEED':
      return 'Yes'
    # Se o COP for menor que 1.55, não está disponível no mercado
    elif row['COP (W/W)'] < 1.55:
      return 'No'
    # Se a tecnologia for 'FIXED SPEED' e o COP for maior ou igual a 1.55, está disponível no mercado
    elif row['Technology'] == 'FIXED SPEED' and row['COP (W/W)'] >= 1.55:
      return 'Yes'
    # Caso contrário, não foi possível determinar a disponibilidade
    else:
      return ''

In [46]:
def preencher_competitor_category(row):
  """Categoriza o player como 'Nidec' ou 'Competitor'.

  Verifica se o nome do player na coluna 'Player Final' é igual a 'NIDEC'.
  Se for igual a 'NIDEC', retorna 'Nidec', indicando que o player é a própria Nidec.
  Caso contrário, retorna 'Competitor', indicando que o player é um competidor.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    Uma string indicando a categoria do player: 'Nidec' ou 'Competitor'.
  """
  # Verifica se o player é NIDEC
  if row['Player Final'] == 'NIDEC':
    return 'Nidec'  # Retorna 'Nidec' se o player for NIDEC
  # Se o player não for NIDEC
  else:
    return 'Competitor'  # Retorna 'Competitor' se o player for um competidor

In [47]:
def preencher_cap_range_india(row):
  if row['CAP (W)'] <= 100:
    return 'A. <= 100'
  elif row['CAP (W)'] > 100 and row['CAP (W)'] <= 110:
    return 'B. 100-110'
  elif row['CAP (W)'] > 110 and row['CAP (W)'] <= 120:
    return 'C. 110-120'
  elif row['CAP (W)'] > 120 and row['CAP (W)'] <= 130:
    return 'D. 120-130'
  elif row['CAP (W)'] > 130 and row['CAP (W)'] <= 180:
    return 'E. 130-180'
  elif row['CAP (W)'] > 180 and row['CAP (W)'] <= 234:
    return 'F. 180-234'
  elif row['CAP (W)'] > 234:
    return 'G. > 234'
  else:
    return ''

def preencher_cop_range_india(row):
  if row['COP (W/W)'] <= 1.45:
    return 'A. <= 1.45'
  elif row['COP (W/W)'] > 1.45 and row['COP (W/W)'] <=1.55:
    return 'B. 1.45-1.55'
  elif row['COP (W/W)'] > 1.55 and row['COP (W/W)'] <=1.65:
    return 'C. 1.55-1.65'
  elif row['COP (W/W)'] > 1.65 and row['COP (W/W)'] <=1.75:
    return 'D. 1.65-1.75'
  elif row['COP (W/W)'] > 1.75 and row['COP (W/W)'] <=1.85:
    return 'E. 1.75-1.85'
  elif row['COP (W/W)'] > 1.85 and row['COP (W/W)'] <=1.95:
    return 'F. 1.85-1.95'
  elif row['COP (W/W)'] > 1.95 and row['COP (W/W)'] <= 2.05:
    return 'G. 1.95-2.05'
  elif row['COP (W/W)'] > 2.05:
    return 'H. > 2.05'
  else:
    return ''

def preencher_cap_range(row):
  if row['CAP (W)'] <= 117:
    return 'A. <= 117'
  elif row['CAP (W)'] > 117 and row['CAP (W)'] <= 180:
    return 'B. 117-180'
  elif row['CAP (W)'] > 180 and row['CAP (W)'] <= 234:
    return 'C. 180-234'
  elif row['CAP (W)'] > 234:
    return 'D. > 234'
  else:
    return ''

def preencher_cop_range(row):
  if row['Technology'] == 'FIXED SPEED':  # Check for ON/OFF
    if row['COP (W/W)'] <= 1.60:
      return 'A. <= 1.60'
    elif row['COP (W/W)'] > 1.60 and row['COP (W/W)'] <=1.80:
      return 'B. 1.60-1.80'
    elif row['COP (W/W)'] > 1.80:
      return 'C. > 1.80'

  elif row['Technology'] == 'VARIABLE SPEED':  # Check for VCC
    if row['COP (W/W)'] <= 1.70:
      return 'D. ON/OFF replacement (<= 1.70)'
    elif row['COP (W/W)'] > 1.70 and row['COP (W/W)'] <=1.85:
      return 'E. Entry level (1.70 - 1.85)'
    elif row['COP (W/W)'] > 1.85 and row['COP (W/W)'] <=1.95:
      return 'F. Mid efficiency (1.85 - 1.95)'
    elif row['COP (W/W)'] > 1.95 and row['COP (W/W)'] <=2.05:
      return 'G. High efficiency (1.95 - 2.05)'
    elif row['COP (W/W)'] > 2.05 and row['COP (W/W)'] <=2.15:
      return 'H. Ultra efficiency (2.05 - 2.15)'
    elif row['COP (W/W)'] > 2.15:
      return 'I. > 2.15'

  else:
    return ''

In [48]:
def corrigir_models(df):
  """Corrige e padroniza os nomes dos modelos na coluna 'Model'.

  Utiliza um dicionário de mapeamento para corrigir e padronizar os nomes dos
  modelos na coluna 'Model'. Isso inclui substituir padrões incorretos ou
  incompletos pelos nomes corretos dos modelos.

  Args:
    df: O DataFrame de entrada.

  Returns:
    O DataFrame com a coluna 'Model' corrigida e padronizada.
  """
  # Define o dicionário de mapeamento de padrões e substituições
  pattern_mapping = {
      'VTD1070YWITH'  : 'VTD1070Y',
      'MODELVFJ050CY1': 'VFJ050CY1',
      'MODELVFLD050SY': 'VFLD050SY',
      'MODELA110WY1A' : 'A110WY1A',
      'MODELAZ110WY1A': 'AZ110WY1A',
      'MODELLR70WY'   : 'LR70WY' ,
      'EKZ60K0060707812AHAL' : 'EKZ60K',
      'VEKZ600060707357' : 'VEKZ60',
      'EY40L58W/1.1REFRIGERATOR' : 'EY40L58W',
      'EY40L58W 1 1REFRIGERATOR' : 'EY40L58W',
      'EY40L58W/1.1REFRIGERARTOR' : 'EY40L58W',
      'EY40L58W 1 1REFRIGERARTOR' : 'EY40L58W',
      'ARABCA105960' : 'PZ65E1',
      'ARABCA105940' : 'PZ65E1',
      '50VEMT11C280WCOP1' : 'VEMT11C',
      'MK 1090YW' : 'MK1090YW',
      'W11674235' : 'VFJ050CY1',
      'MODELLR70WY' : 'LR70WY',
      'MODELAZ110WY1A' : 'AZ110WY1A',
      'MODELA110WY1A' : 'A110WY1A',
      'MODELVFJ050CY1' : 'VFJ050CY1',
      'A99267801' : 'EM2U60HLP',
      '518000005ZU' : 'FMSY9C',
      '51320090599RE' : 'FFU 130HAX',
      '51370121497ZO'  : 'EGAS-80CLP',
      '51370137291QE' : 'EGAS-80HLR',
      'EZ85E1A-' : 'EZ85E1A',
      'D--5-1-' : 'UNKNOWN',
      'PRF115141103' : 'T1116YL',
      'PRF115141101' : 'T1113YL',
      '11101010000225' :  'SZ55C1J',
      'TB1112YWITH' : 'TB1112Y',
      'MODELNOA110WY1A' : 'A110WY1A',
      'MODELVFLD050SY' : 'VFLD050SY',
      'PARTGMCCPZ110H1A' : 'PZ110H1A',
      'VEMZ 9 C' : 'VEMZ9C',
      'MTI110Y' : 'MT1110Y',
      '-TC35351666' : 'TC35351666'
  }

  # Itera sobre os padrões e substituições
  for pattern, replacement in pattern_mapping.items():
    # Preenche valores ausentes na coluna 'Model' com strings vazias
    df['Model'] = df['Model'].fillna('')
    # Aplica a substituição se o padrão for encontrado na coluna 'Model'
    df.loc[df['Model'].str.contains(pattern, case=False), 'Model'] = replacement
  # Retorna o DataFrame modificado
  return df

def corrigir_models_com_players(df):
  """Corrige os nomes dos players com base nos modelos na coluna 'Model'.

  Utiliza um dicionário de mapeamento para corrigir os nomes dos players na coluna
  'Player Final' com base nos modelos na coluna 'Model'. Isso permite associar
  modelos específicos a players corretos, mesmo que o nome do player original
  esteja incorreto ou ausente.

  Args:
    df: O DataFrame de entrada.

  Returns:
    O DataFrame com a coluna 'Player Final' corrigida.
  """
  # Define o dicionário de mapeamento de padrões de modelos e players correspondentes
  pattern_mapping = {
      'EKZ60K'  : 'WANBAO',
      'ASD43KL': 'WANBAO',
      'PZ90H1Y 9': 'GMCC',
      'MSA170KS1G CE1' : 'SAMSUNG',
      'MSA141KS1A SIH': 'SAMSUNG',
      'MSA190HL2H ASH'   : 'SAMSUNG' ,
      'NC4AV80ALR TT3' : 'SAMSUNG',
      'MSA182HL2H ASH' : 'SAMSUNG',
      'GSSHRX0847' : 'TOSHIBA',
      'EFI100E13DGH C0ECS' : 'PANASONIC',
      'EFI100E13DJH' : 'PANASONIC',
      'MSA182HL2G ASH': 'SAMSUNG',
      '50VEMT11C280WCOP1' : 'NIDEC'
  }

  # Itera sobre os padrões de modelos e players
  for pattern, replacement in pattern_mapping.items():
    # Atribui o player correto à coluna 'Player Final' se o padrão do modelo for encontrado
    df.loc[df['Model'].str.contains(pattern, case=False), 'Player Final'] = replacement

  # Retorna o DataFrame modificado
  return df

### 0.2.6. FUNÇÕES PARA RENOMEAR OS DADOS

In [49]:
def check_pattern(x, pattern, ret):
  """Verifica se um valor corresponde a um padrão e retorna um valor específico se corresponder.

  Verifica se o valor de entrada 'x' é uma string e se corresponde ao padrão
  especificado em 'pattern' usando uma expressão regular com ignorecase.
  Se corresponder, retorna o valor especificado em 'ret'.
  Caso contrário, retorna o valor original de 'x'.

  Args:
    x: O valor de entrada a ser verificado.
    pattern: O padrão de expressão regular a ser usado na verificação.
    ret: O valor a ser retornado se houver correspondência com o padrão.

  Returns:
    O valor especificado em 'ret' se houver correspondência com o padrão,
    caso contrário, retorna o valor original de 'x'.
  """
  # Verifica se 'x' é uma string e se corresponde ao padrão
  if isinstance(x, str) and re.search(pattern, x, re.IGNORECASE):
    return ret  # Retorna 'ret' se houver correspondência
  # Se não houver correspondência
  else:
    return x  # Retorna o valor original de 'x'

In [50]:
def renomear_players(df1):
  """Ajusta os nomes dos players usando um arquivo JSON externo.

  Lê um arquivo JSON contendo padrões e substituições para ajustar os nomes
  dos players na coluna 'Player Final' do DataFrame. Aplica as substituições
  usando a função auxiliar 'check_pattern'.

  Args:
    df1: O DataFrame de entrada.

  Returns:
    O DataFrame com os nomes dos players ajustados.
  """
  # Abre o arquivo JSON com os padrões e substituições
  with open('/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Utils/PLAYERS DICTIONARY.json', 'r') as f:
    padroes_e_substituicoes = json.load(f)

  # Itera sobre os padrões e substituições
  for padrao, substituicao in padroes_e_substituicoes.items():
    # Aplica a função 'check_pattern' para ajustar os nomes dos players na coluna 'Player Final'
    df1['Player Final'] = df1['Player Final'].apply(
        lambda x: check_pattern(x, padrao, substituicao)
    )

  # Retorna o DataFrame modificado
  return df1

def renomear_importers(df1):
  """Ajusta os nomes dos importadores usando um arquivo JSON externo.

  Lê um arquivo JSON contendo padrões e substituições para ajustar os nomes
  dos importadores na coluna 'Importer Final' do DataFrame. Aplica as
  substituições usando a função auxiliar 'check_pattern'.

  Args:
    df1: O DataFrame de entrada.

  Returns:
    O DataFrame com os nomes dos importadores ajustados.
  """
  # Abre o arquivo JSON com os padrões e substituições
  with open('/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Utils/IMPORTERS DICTIONARY.json', 'r') as f:
    padroes_e_substituicoes = json.load(f)

  # Itera sobre os padrões e substituições
  for padrao, substituicao in padroes_e_substituicoes.items():
    # Aplica a função 'check_pattern' para ajustar os nomes dos importadores na coluna 'Importer Final'
    df1['Importer Final'] = df1['Importer Final'].apply(
        lambda x: check_pattern(x, padrao, substituicao)
    )

  # Retorna o DataFrame modificado
  return df1

In [51]:
def renomear_models(df):
  """Substitui valores específicos na coluna 'Model' por 'Unknown'.

  Aplica filtros para encontrar linhas com valores específicos na coluna 'Model'
  e substitui esses valores por 'Unknown'. Isso ajuda a padronizar os dados e
  lidar com valores inválidos ou irrelevantes.

  Args:
    df: O DataFrame de entrada.

  Returns:
    O DataFrame com os valores específicos na coluna 'Model' substituídos por 'Unknown'.
  """
  # Define os filtros para cada valor a ser substituído
  filtro = df['Model'] == 'Rotary'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Rotary' por 'Unknown'

  filtro = df['Model'] == 'Automotive'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Automotive' por 'Unknown'

  filtro = df['Model'] == 'Air Conditioning'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Air Conditioning' por 'Unknown'

  filtro = df['Model'] == 'INDUSTRIAL'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'INDUSTRIAL' por 'Unknown'

  filtro = df['Model'] == 'Out of Range'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Out of Range' por 'Unknown'

  filtro = df['Model'] == 'Out of range'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Out of range' por 'Unknown'

  filtro = df['Model'] == 'Not available'
  df.loc[filtro, 'Model'] = 'Unknown'  # Substitui 'Not available' por 'Unknown'

  # Retorna o DataFrame modificado
  return df

In [52]:
def unknown_importer(row):
  """Substitui valores específicos na coluna 'Importer Final' por 'UNKNOWN'.

  Verifica se a coluna 'Importer Final' contém valores específicos que
  indicam dados ausentes ou inválidos e, em caso afirmativo, substitui
  esses valores por 'UNKNOWN'.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    O valor 'UNKNOWN' se a coluna 'Importer Final' contiver um valor
    específico, caso contrário, retorna o valor original da coluna
    'Importer Final'.
  """
  # Verifica se 'Importer Final' é 'NOT AVAILABLE'
  if row['Importer Final'] == 'NOT AVAILABLE':
    return 'UNKNOWN'
  # Verifica se 'Importer Final' é '0'
  elif row['Importer Final'] == '0':
    return 'UNKNOWN'
  # Verifica se 'Importer Final' é 'SIN RAZON SOCIAL'
  elif row['Importer Final'] == 'SIN RAZON SOCIAL':
    return 'UNKNOWN'
  # Verifica se 'Importer Final' é 'OUT OF RANGE'
  elif row['Importer Final'] == 'OUT OF RANGE':
    return 'UNKNOWN'
  # Se nenhum dos valores específicos for encontrado
  else:
    return row['Importer Final']  # Retorna o valor original de 'Importer Final'


def unknown_player(row):
  """Substitui valores específicos na coluna 'Player Final' por 'UNKNOWN'.

  Verifica se a coluna 'Player Final' contém valores específicos que
  indicam dados ausentes ou inválidos e, em caso afirmativo, substitui
  esses valores por 'UNKNOWN'.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    O valor 'UNKNOWN' se a coluna 'Player Final' contiver um valor
    específico, caso contrário, retorna o valor original da coluna
    'Player Final'.
  """
  # Verifica se 'Player Final' é 'NOT AVAILABLE'
  if row['Player Final'] == 'NOT AVAILABLE':
    return 'UNKNOWN'
  # Verifica se 'Player Final' é '0'
  elif row['Player Final'] == '0':
    return 'UNKNOWN'
  # Verifica se 'Player Final' é 'SIN RAZON SOCIAL'
  elif row['Player Final'] == 'SIN RAZON SOCIAL':
    return 'UNKNOWN'
  # Verifica se 'Player Final' é 'OUT OF RANGE'
  elif row['Player Final'] == 'OUT OF RANGE':
    return 'UNKNOWN'
  # Verifica se 'Player Final' é 'NOT DECLARED'
  elif row['Player Final'] == 'NOT DECLARED':
    return 'UNKNOWN'
  # Verifica se 'Player Final' é 'NAN'
  elif row['Player Final'] == 'NAN':
    return 'UNKNOWN'
  # Se nenhum dos valores específicos for encontrado
  else:
    return row['Player Final']  # Retorna o valor original de 'Player Final'

In [53]:
def extrair_player_description(df):
  """Extrai e preenche o nome do player na coluna 'Player Final' com base na descrição.

  Utiliza um dicionário de mapeamento com padrões de expressões regulares para
  identificar nomes de players na coluna 'Description'. Se um padrão for
  encontrado na descrição, o nome do player correspondente é preenchido
  na coluna 'Player Final'.

  Args:
    df: O DataFrame de entrada.

  Returns:
    O DataFrame com a coluna 'Player Final' preenchida com base na descrição.
  """
  # Define o dicionário de mapeamento de padrões e players
  pattern_mapping = {
      r'wanbao': 'WANBAO',
      r'jiaxipera': 'JIAXIPERA',
  }

  # Itera sobre os padrões e players no dicionário
  for pattern, replacement in pattern_mapping.items():
    # Preenche valores ausentes na coluna 'Description' com strings vazias
    df['Description'] = df['Description'].fillna('')
    # Cria uma máscara booleana para as linhas onde o padrão é encontrado na descrição
    mask = df['Description'].str.contains(pattern, case=False, regex=True, na=False)
    # Atribui o nome do player à coluna 'Player Final' nas linhas correspondentes à máscara
    df.loc[mask, 'Player Final'] = replacement

  # Retorna o DataFrame modificado
  return df

### 0.2.7. FUNÇÕES DE MERGE COM CATALOGO

In [54]:
def find_models_on_description(description):
  """Encontra o modelo mais semelhante na descrição usando fuzzy matching.

  Compara a descrição do produto com uma lista de modelos em um catálogo
  usando a técnica de fuzzy matching. A função utiliza a biblioteca 'rapidfuzz'
  para calcular a similaridade entre a descrição e os modelos. Se encontrar
  um modelo com uma similaridade acima do limite definido, retorna o nome
  do modelo. Caso contrário, retorna None.

  Args:
    description: A descrição do produto.

  Returns:
    O nome do modelo mais semelhante encontrado na descrição, ou None se
    nenhum modelo com similaridade suficiente for encontrado.
  """
  # Realiza a correspondência difusa (fuzzy matching) entre a descrição e os modelos no catálogo
  match = process.extractOne(description, catalogo['Model'], scorer=fuzz.token_set_ratio, score_cutoff=100)
  # Retorna o nome do modelo se houver uma correspondência com pontuação acima do limite
  return match[0] if match else None  # Retorna None se não houver correspondência

In [55]:
# A IDEIA AQUI É PARA RESOLVER OS PROBLEMAS QUE SURGEM COM O MERGE
def tech_right(row):
  """Seleciona a tecnologia correta entre duas colunas.

  Prioriza o valor da coluna 'Technology_y' se não for nulo.
  Caso contrário, retorna o valor da coluna 'Technology_x'.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    O valor da tecnologia correta.
  """
  # Verifica se 'Technology_y' não é nulo
  if pd.notna(row['Technology_y']):
    return row['Technology_y']  # Retorna 'Technology_y' se não for nulo
  # Se 'Technology_y' for nulo
  else:
    return row['Technology_x']  # Retorna 'Technology_x'


def player_right(row):
  """Seleciona o player correto entre duas colunas.

  Prioriza o valor da coluna 'Player' se não for nulo.
  Caso contrário, retorna o valor da coluna 'Player Final'.

  Args:
    row: Uma linha do DataFrame.

  Returns:
    O nome do player correto.
  """
  # Verifica se 'Player' não é nulo
  if pd.notna(row['Player']):
    return row['Player']  # Retorna 'Player' se não for nulo
  # Se 'Player' for nulo
  else:
    return row['Player Final']  # Retorna 'Player Final'

def type_right(row):
  # Verifica se 'Technology_y' não é nulo
  if pd.notna(row['COMPRESSOR TYPE_y']):
    return row['COMPRESSOR TYPE_y']  # Retorna 'Technology_y' se não for nulo
  # Se 'Technology_y' for nulo
  else:
    return row['COMPRESSOR TYPE_x']  # Retorna 'Technology_x'


def arrumar_coluna_tech(df3):
  """Ajusta as colunas de tecnologia e player no DataFrame.

  Aplica as funções 'tech_right' e 'player_right' para selecionar os
  valores corretos de tecnologia e player, respectivamente. Remove as
  colunas 'Technology_x', 'Technology_y' e 'Player' após a correção.

  Args:
    df3: O DataFrame de entrada.

  Returns:
    O DataFrame com as colunas de tecnologia e player ajustadas.
  """
  # Aplica 'tech_right' para criar a coluna 'Technology'
  df3['Technology'] = df3.apply(lambda row: tech_right(row), axis=1)
  # Aplica 'player_right' para ajustar a coluna 'Player Final'
  df3['Player Final'] = df3.apply(lambda row: player_right(row), axis=1)
  #df3['COMPRESSOR TYPE'] = df3.apply(lambda row: type_right(row), axis=1)

  if country_name != 'Colombia':
    df3['COMPRESSOR TYPE'] = df3.apply(lambda row: type_right(row), axis=1)
    df3.drop(columns=['Technology_x', 'Technology_y', 'Player', 'COMPRESSOR TYPE_x', 'COMPRESSOR TYPE_y'], inplace=True)
  # Remove as colunas desnecessárias
  #df3.drop(columns=['Technology_x', 'Technology_y', 'Player', 'COMPRESSOR TYPE_x', 'COMPRESSOR TYPE_y'], inplace=True)
  if country_name == 'Colombia':
    df3.drop(columns=['Technology_x', 'Technology_y', 'Player'], inplace=True)
  # Retorna o DataFrame modificado
  return df3

In [56]:
def remover_modelos_curtos(df, coluna='Model'):
  """Remove linhas com modelos curtos em uma coluna específica.

  Filtra o DataFrame para remover linhas onde o valor na coluna especificada
  (por padrão, 'Model') possui menos de 3 caracteres. Isso ajuda a remover
  modelos inválidos ou incompletos.

  Args:
    df: O DataFrame de entrada.
    coluna: O nome da coluna a ser verificada (padrão: 'Model').

  Returns:
    O DataFrame com as linhas contendo modelos curtos removidas.
  """
  # Filtra o DataFrame para manter apenas linhas com valores na coluna especificada com mais de 2 caracteres
  df = df[df[coluna].str.len() >= 2]
  # Retorna o DataFrame filtrado
  return df

### 0.2.8. FUNÇÕES PARA COLOMBIA

In [57]:
def extrair_quantidades(texto):

  quantidades_0 = re.findall(r"QTY\.\s*(\d+)", texto)
  quantidades_1 = re.findall(r"CANTIDAD \s*(\d+)", texto)
  quantidades_2 = re.findall(r"CANT \s*(\d+)", texto)
  quantidades_3 = re.findall(r"CANTIDAD: \s*(\d+)", texto)
  quantidades_4 = re.findall(r"CANTIDAD \s*\((\d+)\)", texto)
  quantidades_5 = re.findall(r"CANTIDAD:\s*\((\d+)\s*UNIDADES?\)", texto)
  quantidades_6 = re.findall(r"CANTIDAD:\s*\((\d+)\s*UNIDAD?\)", texto)
  quantidades_7 = re.findall(r"CANTIDAD:\s*(\d+)\s*PIEZA?", texto)
  quantidades_8 = re.findall(r"CANTIDAD\s*\(UND\):\s*(\d+)", texto)
  quantidades_9 = re.findall(r"\((\d+)\s*UNDS?\)", texto)
  quantidades_10 = re.findall(r"CANTIDAD(\d+)", texto)
  quantidades_11 = re.findall(r"CANT\s*\((\d+)\)", texto) # Added regex for CANT (5) format
  quantidades_12 = re.findall(r"CANTIDAD\s*\(\s*(\d+)\s*\)", texto) # Added regex for CANTIDAD ( 3) format

  for quantidade in [quantidades_0, quantidades_1, quantidades_2, quantidades_3, quantidades_4, quantidades_5, quantidades_6, quantidades_7, quantidades_8, quantidades_9, quantidades_10, quantidades_11, quantidades_12]:

      if quantidade:
            return quantidade

  return []

In [58]:
def extrair_modelos(texto):

  # modelos_0 = re.findall(r"MODELO SEGUN ETIQUETA EN EQUIPO:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_1 = re.findall(r"MODELO SEGUN PROVEEDOR:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_2 = re.findall(r"MODELO.\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_3 = re.findall(r"MODEL:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_4 = re.findall(r"REFERENCIA:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_5 = re.findall(r"MODELO:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)
  # modelos_6 = re.findall(r"MODELO EN PIEZA:\s*([a-zA-Z0-9-.]+)(?![.,])", texto)

  # modelos_0 = re.findall(r"MODELO SEGUN ETIQUETA EN EQUIPO:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_1 = re.findall(r"MODELO SEGUN PROVEEDOR:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_2 = re.findall(r"MODELO.\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_3 = re.findall(r"MODEL:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_4 = re.findall(r"REFERENCIA:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_5 = re.findall(r"MODELO:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)
  # modelos_6 = re.findall(r"MODELO EN PIEZA:\s*([a-zA-Z0-9-.,]+)(?![.,])", texto)

  modelos_0 = re.findall(r"MODELO SEGUN ETIQUETA EN EQUIPO:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_1 = re.findall(r"MODELO SEGUN PROVEEDOR:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_2 = re.findall(r"MODELO.\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_3 = re.findall(r"MODEL:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_4 = re.findall(r"REFERENCIA:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_5 = re.findall(r"MODELO:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_6 = re.findall(r"MODELO EN PIEZA:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_7 = re.findall(r"GMCCREFERENCIA:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  modelos_8 = re.findall(r"MDBAZGMABE240108D:\s*([a-zA-Z0-9-.\s]+)(?![.])", texto)
  for modelo in [modelos_0, modelos_1, modelos_6, modelos_2, modelos_3, modelos_4, modelos_5, modelos_7, modelos_8]:

      if modelo:
          if modelo[0] in ['E', 'NO', 'EZ.', 'SIN', 'P', 'REFERENCIA', 'N', 'NO TIENE', 'SIN MODELO', 'PZ','EZ','FZ']:
            return modelos_4  # Aplica modelos_4 se a condição for verdadeira
          else:
            return modelo

  return []

In [59]:
def split_models(row):
  """Divide a coluna 'Modelos' em múltiplas linhas."""
  models = row['Modelos']
  quantities = row['Quantidades']

  if isinstance(models, list) and len(models) > 0:

    # Cria um dicionário com os dados da linha original
    base_row = row.to_dict()

    # Cria novas linhas, uma para cada modelo
    new_rows = []

    total_volume = 0 # Initialize total volume for the row

    for i, model in enumerate(models):
      new_row = base_row.copy()
      # Assume a mesma quantidade se só houver 1 valor em 'Quantidades', ou divide entre os modelos se houver mais
      #quantity = quantities[0] if isinstance(quantities, list) and len(quantities) == 1 else int(quantities[i]) if isinstance(quantities, list) and len(quantities) > 1 and i < len(quantities) else row['Quantity']  # Valor padrão se não houver correspondência = quantity

      # If there are fewer quantities than models and quantities is not empty, use '' (empty string) for the missing quantities
      if i < len(quantities) and quantities and isinstance(quantities, list):
          quantity = int(quantities[i]) if isinstance(quantities[i], str) and quantities[i].isdigit() else quantities[i]

      elif i < len(models) and quantities:  # If quantities is not empty but there's no quantity for this model
          remaining_quantity = pd.to_numeric(row['Volume'], errors='coerce') - total_volume  # Assign empty string
          quantity = remaining_quantity / (len(models) - i) if (len(models) - i) > 0 else 0

      elif len(models) == 1:  # Check if there's only one model
          quantity = row['Volume']

      else:
          quantity = 0  # Default to 0 for other cases

      new_row['Model'] = model
      new_row['Volume'] = quantity
      new_rows.append(new_row)

      total_volume = pd.to_numeric(total_volume, errors='coerce') + pd.to_numeric(quantity, errors='coerce')

    return new_rows  # Retorna a lista de novas linhas

  else: # Se 'Modelos' estiver vazio:

    new_row = row.to_dict() # Retorna a linha original se não precisar de divisão

    if models == [] and quantities == []:
        new_row['Model'] = 'UNKNOWN'
        new_row['Volume'] = row['Volume']  # Use 'Quantity' for 'Volume'

    else:
        new_row['Model'] = 'UNKNOWN'  # Define 'Model' como 'Unknown' if only models is empty

    new_row['Model'] = 'UNKNOWN'  # Define 'Model' como 'Unknown'

    return [new_row]

In [60]:
def modificar_unit_price(df):
    """Modifica o valor da coluna 'UNIT PRICE' com base na coluna 'Modelos'.

    Esta função itera pelas linhas do DataFrame e, se a coluna 'Modelos'
    contiver uma lista com mais de um item, define o valor da coluna
    'UNIT PRICE' como None (nulo) para essa linha.

    Args:
        df (pd.DataFrame): O DataFrame a ser modificado.

    Returns:
        pd.DataFrame: O DataFrame modificado.
    """
    for index, row in df.iterrows():  # Itera pelas linhas do DataFrame
        # Verifica se a coluna 'Modelos' é uma string representando uma lista com mais de um item
        if isinstance(row['Modelos'], str) and row['Modelos'].startswith('[') and row['Modelos'].endswith(']'):
            modelos = eval(row['Modelos'])  # Converte a string em uma lista usando eval
            if len(modelos) > 1:  # Verifica se a lista tem mais de um item
                df.loc[index, 'UNIT PRICE'] = None  # Define o valor de 'UNIT PRICE' como None
    return df  # Retorna o DataFrame modificado

# 1.0. PADRONIZANDO O NOME DAS COLUNAS

In [61]:
# Faz uma cópia do arquivo original, para não sobreescrevê-lo
df = df_raw.copy()

In [62]:
# Aplica todas as funções de padronização de colunas.
df = padronizar_coluna_hs_code(df)
df = padronizar_coluna_importer(df)
df = padronizar_coluna_country(df)
df = padronizar_coluna_player_raw(df)
df = padronizar_coluna_description(df, country_name)
df = padronizar_coluna_volume(df)
df, valor = padronizar_coluna_value(df)

# 2.0. LIMPEZA DOS DADOS

In [63]:
# Faz uma cópia do df, para não sobreescrevê-lo, assim se garante consistência.
df1 = df.copy()

In [64]:
df1.columns

Index(['Date', 'Importer Raw', 'Importer Tax ID', 'Package Type', 'Unit',
       'Carrier Flag', 'Unit Value CIF (USD)', 'Carrier ID', 'Carrier',
       'Free Zone', 'Payment Method', 'Incoterm', 'Transport Document',
       'Transport Document Date', 'Agreement', 'Customs Declaration ID',
       'Central Bank', 'Commercial Bank', 'Shipment Load Type',
       'Bill of Lading', 'Operation Type', 'Regime', 'Item', 'Agent',
       'Agent ID', 'Observations/Code', 'Observations/Description', 'HS Code',
       'HS Description', 'Merchandise Code', 'Merchandise',
       'Full HS Description', 'Player Raw', 'Description',
       'Country of Purchase', 'Country Raw', 'Customs', 'Transport Method',
       'Shipping Port', 'Delivery Port', 'Value (USD)',
       'Calculated Freight (USD)', 'Calculated Insurance (USD)',
       'Gross Weight (Clearance Total)', 'Quantity of Packages', 'Volume',
       'Unit Value FOB (USD)', 'CIF Value (USD)', 'Tax %',
       'Declaration Quantity'],
      dtype='o

## 2.1. LIMPAR DADOS NULOS

# 3.0. CRIANDO NOVAS COLUNAS

In [65]:
# Lista todas as colunas
colunas_para_modificar = df1.columns.tolist()
# Remove a coluna 'Volume' pois precisamos dela e ela não pode ser excluída de jeito algum
colunas_para_modificar.remove('Volume')

# Percorre cada uma das colunas
for coluna in colunas_para_modificar:
  # Se a coluna for inteiramente nula:
  if df1[coluna].isnull().all():
    # Exclui ela da nossa database
    df1.drop(coluna, axis=1, inplace=True)

# A IDEIA AQUI É TIRAR COLUNAS QUE DEIXARÃO NOSSO DATASET MAIS PESADO MAS QUE NÃO CARREGAM NENHUMA INFORMAÇÃO.

In [66]:
# Faz uma cópia do df1, para não sobreescrevê-lo, assim se garante consistência.
df2 = df1.copy()

In [67]:
# Lista dos possíveis nomes das colunas em que o valor é CIF
cif_values = ["CIF Value [USD]", "Valor CIF (USD)", "U$S CIF"]
# Lista dos possíveis nomes das colunas em que o valor é FOB
fob_values = ["Valor Total FOB US$", "Valor FOB (USD)", "FOB Calculado  (USD)", "Subitem: Calculated FOB Value (US$)", "FOB Value (USD)"]

# Se nossa coluna 'Value' for uma entre as colunas CIF:
if valor in cif_values:
  # Atribua na coluna "CIF/FOB" o valor 'CIF'
  df2['CIF/FOB'] = 'CIF'
# Se nossa coluna 'Value' for uma entre as colunas CIF:
elif valor in fob_values:
  # Atribua na coluna "CIF/FOB" o valor 'FOB'
  df2['CIF/FOB'] = 'FOB'
else:
  # Caso nossa coluna Value não tenha sido derivada de nenhuma daquelas nas listas presentes então mantenha como UNKNOWN
  df2['CIF/FOB'] = 'UNKNOWN'

# A IDEIA AQUI É SABER SE OS VALORES QUE ESTAREMOS VENDO SERÃO FOB, CIF OU NÃO TEREMOS ESSA INFORMAÇÃO

In [68]:
df2.columns

Index(['Date', 'Importer Raw', 'Importer Tax ID', 'Package Type', 'Unit',
       'Carrier Flag', 'Unit Value CIF (USD)', 'Carrier ID', 'Carrier',
       'Payment Method', 'Incoterm', 'Transport Document',
       'Transport Document Date', 'Agreement', 'Customs Declaration ID',
       'Shipment Load Type', 'Bill of Lading', 'Operation Type', 'Regime',
       'Item', 'Agent', 'Agent ID', 'Observations/Code',
       'Observations/Description', 'HS Code', 'HS Description',
       'Merchandise Code', 'Merchandise', 'Full HS Description', 'Player Raw',
       'Description', 'Country of Purchase', 'Country Raw', 'Customs',
       'Transport Method', 'Shipping Port', 'Delivery Port', 'Value (USD)',
       'Calculated Freight (USD)', 'Calculated Insurance (USD)',
       'Gross Weight (Clearance Total)', 'Quantity of Packages', 'Volume',
       'Unit Value FOB (USD)', 'CIF Value (USD)', 'Declaration Quantity',
       'CIF/FOB'],
      dtype='object')

In [69]:
# Aplica todas as funções para criar novas colunas
df2 = criar_coluna_year_month(df2)
df2 = criar_coluna_data(df2)
df2 = criar_coluna_quarter(df2)
df2 = criar_coluna_samples(df2)
df2 = criar_coluna_range(df2)
df2 = criar_coluna_model(df2)
df2 = criar_coluna_segmento(df2)
df2 = criar_coluna_unit_price(df2)
df2 = criar_coluna_aduana(df2)
df2 = criar_coluna_vertical(df2)
df2 = criar_coluna_manufacturer_category(df2)
df2 = criar_coluna_market_aviability(df2)
df2 = criar_coluna_cap_range(df2)
df2 = criar_coluna_cop_range(df2)

In [70]:
# Caso você tenha interesse em um ano específico, basta descomentar o código abaixo alterar para os anos que você quer.

# df2['Year'] = pd.to_numeric(df2['Year'], errors='coerce')
# # Pega somente os dados cujos anos são 2023, 2024 ou 2022. Se quiser algum diferente, basta seguir a mesma estrutura.
# condition = (df2['Year'] == 2023) | (df2['Year'] == 2024) | (df2['Year'] == 2022)
# # Recorta os dados para pegar somente os dados selecionados
# df2 = df2[condition]

# 4.0. PREENCHER OS DADOS

In [71]:
# Faz uma cópia do df2, para não sobreescrevê-lo, assim se garante consistência.
df3 = df2.copy()

## 4.1. COLUNA IMPORTER

In [72]:
# Importer Final se torna uma cópia de Importer Raw.
# A IDEIA É PRIMEIRO FAZER UMA CÓPIA PARA DEPOIS TRATARMOS ESSES DADOS E CHEGAR NO RESULTADO ESPERADO
df3['Importer Final'] = df3['Importer Raw']

## 4.2. COLUNA PLAYER

In [73]:
# Player Final se torna uma cópia de Player Raw.
# A IDEIA É PRIMEIRO FAZER UMA CÓPIA PARA DEPOIS TRATARMOS ESSES DADOS E CHEGAR NO RESULTADO ESPERADO
df3.columns
df3['Player Final'] = df3['Player Raw']

## 4.4. COLUNA TECHNOLOGY

In [74]:
# Aplica a função 'categorize_technology' em cada uma das linhas do meu dataset
df3['Technology'] = df3['Description'].apply(categorize_technology)

## 4.5. COLUNAS ROTATIVO, RECIPROCO E SPLIT

In [75]:
#df3['COMPRESSOR TYPE'] = df3['Description'].apply(categorize_compressor_type)

if country_name != 'Colombia':

    df3['COMPRESSOR TYPE'] = df3['Description'].apply(categorize_compressor_type)

# 5.0. TRATAMENTO DOS DADOS


In [76]:
# Faz uma cópia do df3, para não sobreescrevê-lo, assim se garante consistência.
df4 = df3.copy()

## 5.1. COLUNA IMPORTER

In [77]:
# Converter a coluna para string (formato de texto)
df4['Importer Final'] = df4['Importer Final'].astype(str)

# Aplica a função 'renomear_importers' no dataset
df4 = renomear_importers(df4)

# Deixa tudo da coluna 'Importer Final' em maisculo
df4['Importer Final'] = df4['Importer Final'].str.upper()

# Aplica a função 'unknown_importer' em cada linha
df4['Importer Final'] = df4.apply(unknown_importer, axis=1)

## 5.2. COLUNA PLAYER

In [78]:
# Converter a coluna para string (formato de texto)
df4['Player Final'] = df4['Player Final'].astype(str)

# Aplica a função 'renomear_players' no dataset
df4 = renomear_players(df4)

# Deixa tudo da coluna 'Player Final' em maisculo
df4['Player Final'] = df4['Player Final'].str.upper()

# Aplica a função 'player_in_description' em cada linha
df4['Player Final'] = df4.apply(player_in_description, axis=1)

# Aplica a função 'extrair_player_description' no dataset
df4 = extrair_player_description(df4)

In [79]:
# Se o Player Final for 'EMBRACO', troca para 'NIDEC'
df4.loc[df4['Player Final'] == 'EMBRACO', 'Player Final'] = 'NIDEC'

## 5.3. COLUNA MODEL

In [80]:
# Aplica a função renomear_models no dataset
df4 = renomear_models(df4)

## 5.4. Colombia

In [81]:
if country_name == 'Colombia':
  # Aplica a função 'extrair_quantidades' à coluna 'Description'
  # e armazena os resultados na nova coluna 'Quantidades'.
  df4['Quantidades'] = df4['Description'].apply(extrair_quantidades)

  # Aplica a função 'extrair_modelos' à coluna 'Description'
  # e armazena os resultados na nova coluna 'Modelos'.
  df4['Modelos'] = df4['Description'].apply(extrair_modelos)

  # Aplica a função processar_dataframe
  df4 = df4.apply(split_models, axis=1).explode()
  df4 = pd.DataFrame(list(df4))  # Converte de volta para DataFrame

In [82]:
df4.columns

Index(['Date', 'Importer Raw', 'Importer Tax ID', 'Package Type', 'Unit',
       'Carrier Flag', 'Unit Value CIF (USD)', 'Carrier ID', 'Carrier',
       'Payment Method', 'Incoterm', 'Transport Document',
       'Transport Document Date', 'Agreement', 'Customs Declaration ID',
       'Shipment Load Type', 'Bill of Lading', 'Operation Type', 'Regime',
       'Item', 'Agent', 'Agent ID', 'Observations/Code',
       'Observations/Description', 'HS Code', 'HS Description',
       'Merchandise Code', 'Merchandise', 'Full HS Description', 'Player Raw',
       'Description', 'Country of Purchase', 'Country Raw', 'Customs',
       'Transport Method', 'Shipping Port', 'Delivery Port', 'Value (USD)',
       'Calculated Freight (USD)', 'Calculated Insurance (USD)',
       'Gross Weight (Clearance Total)', 'Quantity of Packages', 'Volume',
       'Unit Value FOB (USD)', 'CIF Value (USD)', 'Declaration Quantity',
       'CIF/FOB', 'Year', 'Month', 'Quarter CY', 'Samples', 'Range', 'Model',
       '

# 6.0. MERGE COM CATALOGO

In [83]:
# Faz uma cópia do df4, para não sobreescrevê-lo, assim se garante consistência.
df5 = df4.copy()

In [84]:
# Lê o catalogo que está no drive
catalogo  = pd.read_excel('/content/drive//Shared drives/Trade_Marketing_HA/CUSTOMS/Catalogos/Catalogo 2024.xlsx')

In [85]:
catalogo.columns

Index(['Unnamed: 0', 'Player', 'Model', 'Displacement (cm³)', 'Voltage',
       'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology',
       'COMPRESSOR TYPE', 'Model_Aux', 'Modelo Elux',
       'VERMELHO É MODELO ERRADO PRO CODIGO PEGAR', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'LARANJA É SCROLL/ROTATIVO'],
      dtype='object')

In [86]:
df5.columns

Index(['Date', 'Importer Raw', 'Importer Tax ID', 'Package Type', 'Unit',
       'Carrier Flag', 'Unit Value CIF (USD)', 'Carrier ID', 'Carrier',
       'Payment Method', 'Incoterm', 'Transport Document',
       'Transport Document Date', 'Agreement', 'Customs Declaration ID',
       'Shipment Load Type', 'Bill of Lading', 'Operation Type', 'Regime',
       'Item', 'Agent', 'Agent ID', 'Observations/Code',
       'Observations/Description', 'HS Code', 'HS Description',
       'Merchandise Code', 'Merchandise', 'Full HS Description', 'Player Raw',
       'Description', 'Country of Purchase', 'Country Raw', 'Customs',
       'Transport Method', 'Shipping Port', 'Delivery Port', 'Value (USD)',
       'Calculated Freight (USD)', 'Calculated Insurance (USD)',
       'Gross Weight (Clearance Total)', 'Quantity of Packages', 'Volume',
       'Unit Value FOB (USD)', 'CIF Value (USD)', 'Declaration Quantity',
       'CIF/FOB', 'Year', 'Month', 'Quarter CY', 'Samples', 'Range', 'Model',
       '

In [87]:
# Se a aduana for da Argentina:
if country_name == 'Argentina':
  # Exclui a coluna Model
  df5.drop(columns = 'Model', axis = 1, inplace = True)
  # Troca o nome de 'Subitem: Model' para 'Model'
  #df5.rename(columns = {'Subitem: Model': 'Model'}, inplace = True)

  df5['Description_Aux'] = df5['Subitem: Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  # Exclui linhas do catálogo cujos modelos são repetidos
  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

# Se a aduana for do Ecuador:
elif country_name == 'Ecuador':
  # # Exclui linhas do catálogo cujos modelos são repetidos
  # catalogo.drop_duplicates(subset = 'Model', inplace = True)

  # # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
  # df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player']], on='Model', how='left')

  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  df5['Description_Aux'] = df5['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna() | (df5['Description_Aux'].astype(str).str.len() <= 2) , 'Description_Aux_2'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna() | (df5['Description_Aux'].astype(str).str.len() <= 2), 'Model'] = df5['Description_Aux_2'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna() | (df5['Description_Aux'].astype(str).str.len() <= 2), 'Description_Aux_3'] = df5['Commercial Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna() | (df5['Description_Aux'].astype(str).str.len() <= 2), 'Model'] = df5['Description_Aux_3'].apply(find_models_on_description)

  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

elif country_name == 'Colombia':

  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  df5['Description_Aux'] = df5['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

  #df5.drop(columns = ['Description_Aux'], inplace = True)

elif country_name == 'Peru':


  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  df5['Description_Aux'] = df5['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_2'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_2'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_3'] = df5['Features'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_3'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_4'] = df5['Name'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_4'].apply(find_models_on_description)

  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')


elif country_name == 'Chile':


  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  df5['Description_Aux'] = df5['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_2'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_2'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_3'] = df5['Merchandise'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_3'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_4'] = df5['Merchandise Code'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_4'].apply(find_models_on_description)

  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

elif country_name == 'Paraguay':

  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  df5['Description_Aux'] = df5['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_2'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_2'].apply(find_models_on_description)

  df5.loc[df5['Model'].isna(), 'Description_Aux_3'] = df5['Commercial Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  df5.loc[df5['Model'].isna(), 'Model'] = df5['Description_Aux_3'].apply(find_models_on_description)

  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

# Se a aduana for qualquer outra:
else:
  # Tira todos os simbolos que não sejam letras ou números dos modelos do catálogo
  catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
  # Tira todos os simbolos que não sejam letras ou números da Description e atribua isso a uma nova coluna chamada 'Description_Aux'
  df5['Description_Aux'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

  # Aplica a função 'find_models_on_description' na coluna 'Description_Aux'
  df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

  # Exclui linhas do catálogo cujos modelos são repetidos
  catalogo.drop_duplicates(subset = 'Model', inplace = True)

  # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
  df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player', 'COMPRESSOR TYPE']], on='Model', how='left')

  # Exclui a coluna auxiliar de description
  df5.drop(columns = ['Description_Aux'], inplace = True)

In [88]:
# Preenche as linhas cujo Model esteja vazio com 'Unknown'
df5['Model'] = df5['Model'].fillna('Unknown')

# Deixa tudo da coluna Model em maisculo
df5['Model'] = df5['Model'].str.upper()

In [89]:
# Aplica a funcao 'arrumar_coluna_tech' no dataset
df5 = arrumar_coluna_tech(df5)

# 7.0. CORREÇÕES FINAIS PÓS MERGE

In [90]:
# Faz uma cópia do df5, para não sobreescrevê-lo, assim se garante consistência.
df6 = df5.copy()

## 7.1 COLUNA MARKET AVIABILITY

In [91]:
# Converte a coluna COP para numerico
df6['COP (W/W)'] = pd.to_numeric(df6['COP (W/W)'], errors='coerce')

# Aplica a função 'preencher_market_aviability' no dataset
df6['Market Aviability'] = df6.apply(preencher_market_aviability, axis=1)

In [92]:
# Aplica a função 'encontrar_vertical' no dataset
df6['Vertical'] = df6.apply(encontrar_vertical, axis=1)

In [93]:
# Aplica a função 'preencher_competitor_category' no dataset
df6['Compressor Manufacturer Category'] = df6.apply(preencher_competitor_category, axis=1)

## 7.2. COLUNA CAP E COP RANGE

In [94]:
# Converte a coluna CAP para numerico
df6['CAP (W)'] = pd.to_numeric(df6['CAP (W)'], errors='coerce')
# Converte a coluna COP para numerico
df6['COP (W/W)'] = pd.to_numeric(df6['COP (W/W)'], errors='coerce')

# Verifica se o país é Índia e aplica as funções correspondentes
if 'India' in df6['ADUANA'].unique():
    india_mask = df6['ADUANA'] == 'India'
    df6.loc[india_mask, 'CAP Range'] = df6.loc[india_mask].apply(preencher_cap_range_india, axis=1)
    df6.loc[india_mask, 'COP Range'] = df6.loc[india_mask].apply(preencher_cop_range_india, axis=1)

    # Aplica as funções para outros países
    other_mask = ~india_mask
    df6.loc[other_mask, 'CAP Range'] = df6.loc[other_mask].apply(preencher_cap_range, axis=1)
    df6.loc[other_mask, 'COP Range'] = df6.loc[other_mask].apply(preencher_cop_range, axis=1)
else:
    df6['CAP Range'] = df6.apply(preencher_cap_range, axis=1)
    df6['COP Range'] = df6.apply(preencher_cop_range, axis=1)

## 7.3. COLUNA SEGMENTO

In [95]:
# Cria um dicionario de segmentos
dic = cria_dicionario_segmento()

# Faz um dictionary em que cada Importer está associado a um Segment
seg_dic = dict(zip(dic['Importer'], dic['Segment']))
# Usa o dicionario para preencher a coluna Segment de acordo com o Importer Final
df6['Segment'] = df6['Importer Final'].map(seg_dic).fillna(df6['Segment'])

In [96]:
# Aplica a função 'segment_by_hscode' no dataset
df6 = segment_by_hscode(df6)

In [97]:
# Aplica todas as funções para definir segmento a partir dos importers
df6 = importers_extra(df6)
df6 = importers_ac(df6)
df6 = importers_ca(df6)

In [98]:
# Aplica todas as funções para definir segmento a partir dos players
df6 = players_extra(df6)
df6 = players_ac(df6)
df6 = players_ca(df6)

In [99]:
# Aplica todas as funções para definir segmento a partir do que está escrito na descrição
df6 = ac_by_description(df6)
df6 = extra_by_description(df6)

In [100]:
# Aplica a função 'intracompany_segment' no dataset
df6['Segment'] = df6.apply(intracompany_segment, axis=1)

In [101]:
# Reaplicar as funções para definir EXTRA como segmento para garantir que não erros
df6 = importers_extra(df6)
df6 = players_extra(df6)
df6 = extra_by_description(df6)
df6 = wet_by_description(df6)
df6 = segment_by_hscode(df6)

In [102]:
# # Se o player for 'JABIL CIRCUIT (GUANGZHOU) LIMITED', o segmento é EXTRA
# df6.loc[df6['Player Raw'] == 'JABIL CIRCUIT (GUANGZHOU) LIMITED', 'Segment'] = 'EXTRA'
# df6.loc[df6['Importer Raw'] == 'TATA HITACHI CONSTRUCTION MACHINERY COMPANY PRIVAT', 'Segment'] = 'EXTRA'

# # Define um filtro para ser alterado. Se o Importer for Midea e o Player for Toshiba.
# filtro = (df6['Importer Final'] == 'MIDEA' ) & (df6['Player Final'] == 'TOSHIBA')
# # Pega os valores do filtro acima e define segmento como HVAC. (Tudo que tem midea de importer e toshiba de player é HVAC)
# df6.loc[filtro, 'Segment'] = 'HVAC'

# # Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
# filtro2 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'SCROLL')
# # Pega os valores do filtro acima e define segmento como HVAC.
# df6.loc[filtro2, 'Segment'] = 'HVAC'

# # Define um filtro. Tudo que é HA e o modelo tem Technology Screw
# filtro3 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'SCREW')
# # Pega os valores do filtro acima e define segmento como EXTRA.
# df6.loc[filtro3, 'Segment'] = 'EXTRA'

# filtro4 = (df6['Importer Raw'] == 'SERVICIOS LOGISTICOS PALCO S DE RL DE CV' ) & (df6['Player Final'] == 'WHIRLPOOL')
# df6.loc[filtro4, 'Segment'] = 'HA'

# filtro4 = (df6['Importer Raw'] == 'VOLTAS LIMITED Total' ) & (df6['Player Raw'] == 'MIDEA ELECTRIC TRADING SINGAPORE  C')
# df6.loc[filtro4, 'Segment'] = 'HVAC'

# # Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
# filtro5 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'ROTARY')
# # Pega os valores do filtro acima e define segmento como HVAC.
# df6.loc[filtro5, 'Segment'] = 'HVAC'

# # Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
# filtro6 = (df6['Segment'] == 'HA') & (df6['Player Final'] == 'TCL')
# # Pega os valores do filtro acima e define segmento como HVAC.
# df6.loc[filtro6, 'Segment'] = 'HVAC'


## 7.4. COLUNA MODEL

In [103]:
# Aplica as funções para corrigir os modelos
df6 = corrigir_models(df6)
df6 = corrigir_models_com_players(df6)

In [104]:
# Aqui, cada uma dessas linhas faz o seguinte:
# Se o modelo for o especificado, altera o segmento para o definido ao fim.
# Exemplo: Se o modelo for 'L126WY1', então segmento vai ser CA.
df6.loc[df6['Model'] == 'ANA120XL ',  'Segment'] = 'CA'
df6.loc[df6['Model'] == 'DST156MAA',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'SNB140FCAMC',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'MNB36FABMC',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'C 6RZ132H3AAF',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSN108D34UEZ3',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSK89D29UEZD',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PH290M2C 4FT6',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSF180S1VFPA',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZB45KQE TFD 523',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZB29KQE TFD 524',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == '5RD132XBB21',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == '5RD132XHA21',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PBL-CB417BLDU',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'C SWS225H01C',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ANB52FKFMT',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'PZ65E1',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'PUS18KEB125U',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PUS12KEB121U',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'GSD113RKQF6JV6B',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSF160S2VFPC3',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSN140D21UFZ',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSN108D64UFZ3',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KTM240D43UKUA2',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ATF310D43UMT',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ATF420D50UMV',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZPV0662E 4X9',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZP122KCE TFD 522',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'COMPRESSORATQ580D66UNT',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'LNB42FSAMC',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'TNB306FPNMC',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSK75D15UEZ3',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KTF310D43UMT',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'SYB220FFEMC L2',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'LNB53FDKMC L',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'TNB306FPPMC L',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'MNB42FFDMC L',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'DST102MAA',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZR61KCE-TFD-522',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZR72KC-TFD-52E',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ZBHJ 21 0008', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXVW045', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN132', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN045', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN004D', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN004.D', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN004', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXUN003', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXTT139', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXTT019', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXTK247', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXTK213', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXTK101', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXRN051', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXRN027', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXPG009', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXPG008', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXPG006', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXOP006', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXOP003', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXOP001', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXFT015', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXFD081', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXFD008', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV059', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV052', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV051', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV047', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV044', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCV011', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXCC014', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'WXBS067', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'VS16 5GRV', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST976285', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST972863GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST972529GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST970404GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST96A148', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST965198', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST963418U', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST962207GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST961415GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST960102GW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST792535', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST780116', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST770344', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST770306', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST699291', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST697855', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST697775', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST697237', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST696121', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST695815', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST570101C', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST561517', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST550101C', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST541301C', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST530210', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST500101C', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST351111', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST281127', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST250707U', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST210606XW', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST210201', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST172839', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST172435', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST171314', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST155859', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST154072', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST152529', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST131240', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST120406', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST111432', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST100406', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'ST070101', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'SE7H15_V', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'SD7H15HD', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'SD7H15', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'SD6V12', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'SD5L11', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'S4BCF5.2Y25D', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QR50', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QR390', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QPS65_2770', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 2653', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 2531', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 2085', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 2081', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 2058G', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 1994', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 1837', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP16XD 1249', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP15XD 2027', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP15XD 1766', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QP15XD 1624', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'QL362', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'PXE16 8886', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'PXC16 8890', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'PXC16', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'OSNA8571_K', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'FLX7 4435', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'FKX40655K', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'FKX40655', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'FKX30275TK', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'DZ95189154010', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20719', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20717', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20707', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20705', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20704', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20701', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20698', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20569', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20522', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20484', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20419', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20239', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20227', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20069', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'CS20054', 'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'AR05JRFLAWKXSE',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'AR09JRFLAWKXSE',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'AR30MRFUPGMXHC',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'VSH1111Y',  'Segment'] = 'HA'
df6.loc[df6['Model'] == '3DS5 1500 EWK',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'KCX 6',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'KCX 4',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'NC4EVA3ALM',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'MSV172AL2J SM3',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'NC4AV80ALR TT3',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'NC4EVA5ALN',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'NC4AV80ALR TT3',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'MSV4A1AL1R',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'SE40E1K 9',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'PE75H1H',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'PINVO 18K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'POWERCON X',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'SPRINTER X',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'CS CU UE18XKF 9',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PINVO 12K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ECONO X',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ATMOS COOL',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PINVO 24K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'INSPIRE PRO',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PFS 24K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PINVO',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PCONVO 18K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PCONVO 12K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PFSAC 48K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'MSE166CL1H ASH',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'NI34T9102ADTS7',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'NF54M7151ANASH',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'MSV4A1AL1RTSJ',  'Segment'] = 'HA'
df6.loc[df6['Model'] == 'PSAC 18K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PSAC 12K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'PSAC 24K',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'ATMOS INSPIRE',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'HGX56E 995 4',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'HGX56E 1155 4 S',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'HGX44E 565 4 S',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'C-L75M31',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'C-L105M83',  'Segment'] = 'EXTRA'
df6.loc[df6['Model'] == 'UG5T450FXAJX',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSN103D42UEZS31',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'KSN89D21UEZS31',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'DSN200D27UEZ31',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'DXT103MAB',  'Segment'] = 'HVAC'
df6.loc[df6['Model'] == 'JQC068MBA',  'Segment'] = 'HVAC'
# df6.loc[df6['Model'] == '',  'Segment'] = ''

In [105]:
# Aqui, cada uma dessas linhas faz o seguinte:
# Se o modelo for o especificado, altera o Player para o definido ao fim.
# Exemplo: Se o modelo for 'EY40L58W', então Player vai ser HUAGUANG.
df6.loc[df6['Model'] == 'EY40L58W', 'Player Final'] = 'HUAGUANG'
df6.loc[df6['Model'] == 'PZ65E1C 9', 'Player Final'] = 'GMCC'
df6.loc[df6['Model'] == 'MSV172AL2J SM3', 'Player Final'] = 'SAMSUNG'
df6.loc[df6['Model'] == 'MKV190CL2B   ASH', 'Player Final'] = 'SAMSUNG'

In [106]:
#df6['Model'] = df6['Model'].map(str)

df6.loc[df6['Model'].str.contains('TT1114HY-ID'), 'Model']='TT1114HY'
df6.loc[df6['Model'].str.contains('TT1114HY-ID-L'), 'Model']='TT1114HY'
df6.loc[df6['Model'].str.contains('COMPRESSORATQ580D66UNT'), 'Model']='ATQ580D66UNT'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1086427EM2U 3115U'), 'Model']='EM2U3115U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1089191EM2U 3115U'), 'Model']='EM2U3115U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1095556FMFT 411U'), 'Model']='FMFT411U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1100896EM2X 3125U'), 'Model']='EM2X3125U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1101990VEMT404U'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACO1105748FMFT 408U'), 'Model']='FMFT408U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOEM2U3115U'), 'Model']='EM2U3115U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT 408U'), 'Model']='FMFT408U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT 411U'), 'Model']='FMFT411U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT 413U'), 'Model']='FMFT413U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT 415U'), 'Model']='FMFT415U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT408U'), 'Model']='FMFT408U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT411U'), 'Model']='FMFT411U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOPOTENCIA 271WEM2X 3121U'), 'Model']='EM2X3121U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOPOTENCIA 568WEM2X 3121U'), 'Model']='EM2X3121U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404U'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('271WEM2X 3121U'), 'Model']='EM2X3121U'
df6.loc[df6['Model'].str.contains('EM2X 3121U115'), 'Model']='EM2X-3121U'
df6.loc[df6['Model'].str.contains('HERMETICOHUAYI FNX21TB'), 'Model']='FNX21TB'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACONEK-2170U'), 'Model']='NEK2170U'
df6.loc[df6['Model'].str.contains('COMPRESORHISPANIAZP120KCE'), 'Model']='ZP120KCE-TFD'
df6.loc[df6['Model'].str.contains('COMPRESORHISPANIATAG2516ZPARA'), 'Model']='TAG2516Z'
df6.loc[df6['Model'].str.contains('COMPRESORVERTIV FCOMPZPDT27MCE380 460'), 'Model']='ZPDT27MCE'
df6.loc[df6['Model'].str.contains('AIREDANFOSS F120H1561DE'), 'Model']='F120H'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404U550W'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404U108'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOEM2U3115UVELOCIDAD'), 'Model']='EM2U3115U'
df6.loc[df6['Model'].str.contains('COMPRESORESEMBRACOFMFT 408UDE'), 'Model']='FMFT408U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404U230V'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('COMPRESORESEMBRACOEM2X 31250U220'), 'Model']='EM2X31250U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404U5'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('COMPRESORESEMBRACOEM2X3121U115'), 'Model']='EM2X3121U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT408UDE230V'), 'Model']='FMFT408U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOFMFT411UDE230V'), 'Model']='FMFT411U'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACOVEMT404UDE230V'), 'Model']='VEMT404U'
df6.loc[df6['Model'].str.contains('FFMSA9C220V'), 'Model']='FMSA9C'
df6.loc[df6['Model'].str.contains('FMXY9CPARTE'), 'Model']='FMXY9C'
df6.loc[df6['Model'].str.contains('H300 CSPOTENCIA'), 'Model']='H300-CS'
df6.loc[df6['Model'].str.contains('COMPRESOREMBRACONEK 2170U'), 'Model']='NEK2170U'
df6.loc[df6['Model'].str.contains('LZ68CULOS'), 'Model']='LZ68CU'
df6.loc[df6['Model'].str.contains('COMPRESSORPZ99H1E9'), 'Model']='PZ99H1E-9'
df6.loc[df6['Model'].str.contains('COMPRESSORPZ99H1E-9'), 'Model']='PZ99H1E-9'
df6.loc[df6['Model'].str.contains('GMCCDZ150A1W'), 'Model']='DZ150A1W'
df6.loc[df6['Model'].str.contains('(MODEL:DG43DY1)'), 'Model']='DG43DY1'
df6.loc[df6['Model'].str.contains('GMCCSZ90E1J-N'), 'Model']='SZ90E1J-N'
df6.loc[df6['Model'].str.contains('44R312A G-AJSC'), 'Model']='44R312A'
df6.loc[df6['Model'].str.contains('GMCCSZ90E1J-N'), 'Model']='SZ90E1J-N'
df6.loc[df6['Model'].str.contains('EF1120E13DGH-AS0EUS'), 'Model']='EFI120E13DGH'

df6.loc[df6['Model'].str.contains('ACCESSORIES 6DHNR35ME FSD C00  TERMS'), 'Model']='6DHNR35ME-FSD-C00'
df6.loc[df6['Model'].str.contains('VKZ130CUSPLET'), 'Model']='VKZ130CU'
df6.loc[df6['Model'].str.contains('QXASH F295N450  TERMS'), 'Model']='QXASH-F295N450'
df6.loc[df6['Model'].str.contains(' MODELHYS69YCA'), 'Model']='HYS69YCA'
df6.loc[df6['Model'].str.contains('GSD113RKQF6JY6B '), 'Model']='GSD113RKQF-6JY6B'
df6.loc[df6['Model'].str.contains('ACCESSORIESKLF5 6CNT'), 'Model']='KLF5.6CNT'
df6.loc[df6['Model'].str.contains(' MODELHYS60YCA'), 'Model']='HYS60YCA'
df6.loc[df6['Model'].str.contains(' MODELHYS60YCA'), 'Model']='HYS60YCA'
df6.loc[df6['Model'].str.contains(' MODELHYS45YH81A'), 'Model']='HYS45YH81A'
df6.loc[df6['Model'].str.contains('GSD11 3RKQF6JV6B '), 'Model']='GSD113RKQ-F6JV6B'
df6.loc[df6['Model'].str.contains(' NBC90NK'), 'Model']='NBC90NK'
df6.loc[df6['Model'].str.contains(' NBC80NK'), 'Model']='NBC80NK'
df6.loc[df6['Model'].str.contains(' NBC70NK'), 'Model']='NBC70NK'
df6.loc[df6['Model'].str.contains(' NBC60NK'), 'Model']='NBC60NK'
df6.loc[df6['Model'].str.contains(' NBC45NK'), 'Model']='NBC45NK'
df6.loc[df6['Model'].str.contains('QX D253RF050A  TERMS'), 'Model']='QX-D253RF050A'
df6.loc[df6['Model'].str.contains('AZ A9411 '), 'Model']='AZ A9411'
df6.loc[df6['Model'].str.contains('GAS SZ65FIM'), 'Model']='SZ65FIM'
df6.loc[df6['Model'].str.contains('CHARGE  MXH230NA'), 'Model']='MXH230NA'
df6.loc[df6['Model'].str.contains('COMPPZ99H1E 9 '), 'Model']='PZ99H1E-9'
df6.loc[df6['Model'].str.contains('AE4440YGH-'), 'Model']='AE4440YGH'
df6.loc[df6['Model'].str.contains('SERIESCSB035NKDG'), 'Model']='CSB035NKEG'
df6.loc[df6['Model'].str.contains('  GSD108CKQA6JV8B'), 'Model']='GSD108CKQA6JV8B'
df6.loc[df6['Model'].str.contains('NO GSD108CKQA6JV8B'), 'Model']='GSD108CKQA6JV8B'
df6.loc[df6['Model'].str.contains('NO GSX102CKTA6JT8 '), 'Model']='GSX102CKTA6JT8'
df6.loc[df6['Model'].str.contains(' KSN108D31UFZ31 '), 'Model']='KSN108D31UFZ31'
df6.loc[df6['Model'].str.contains(' KSN98D24UEZ31 '), 'Model']='KSN98D24UEZ31'
df6.loc[df6['Model'].str.contains(' KSN108D43UFZA '), 'Model']='KSN108D43UFZA'
df6.loc[df6['Model'].str.contains(' KTN130D42UFZ '), 'Model']='KTN130D42UFZ'
df6.loc[df6['Model'].str.contains(' MODEL  GTD150RKQA7JV6B'), 'Model']='GTD150RKQA7JV6B'
df6.loc[df6['Model'].str.contains(' KSN120D43UFZ31 '), 'Model']='KSN120D43UFZ31'
df6.loc[df6['Model'].str.contains('KSN108D31UEE31 '), 'Model']='KSN108D31UEE31'
df6.loc[df6['Model'].str.contains(' KSN108D34UFZ3  ACA20CMP10901 '), 'Model']='KSN108D34UFZ3'
df6.loc[df6['Model'].str.contains('GSD088RKQA6JV6BCOMPRESSOR'), 'Model']='GSD088RKQA6JV6B'
df6.loc[df6['Model'].str.contains('ACCESSORIES KSN89D21UEZ31'), 'Model']='KSN89D21UEZ31'
df6.loc[df6['Model'].str.contains('GSD108CKQA6JV8BA  WITH'), 'Model']='GSD108CKQA6JV8BA'
df6.loc[df6['Model'].str.contains('P N GSD102SKQA6JV6B'), 'Model']='GSD102SKQA6JV6B'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN108D31UEZ31'), 'Model']='KSN108D31UEZ31'
df6.loc[df6['Model'].str.contains(' KSN120D53UFZ3 '), 'Model']='KSN120D53UFZ3'
df6.loc[df6['Model'].str.contains('NO14D7151ALTT9'), 'Model']='NO14D7151ALTT9'
df6.loc[df6['Model'].str.contains(' KSN108D34UFZ3 '), 'Model']='KSN108D34UFZ3'
df6.loc[df6['Model'].str.contains('GSX073CKTA6JT8ACOMPRESSOR'), 'Model']='GSX073CKTA6JT8A'
df6.loc[df6['Model'].str.contains('KSK103D33SERC3 WITHOUT'), 'Model']='KSK103D33SERC3'
df6.loc[df6['Model'].str.contains('GSD102SKQA6JV6BCOMPRESSOR'), 'Model']='GSD102SKQA6JV6B'
df6.loc[df6['Model'].str.contains(' MODEL 1YC20HXD A6 '), 'Model']='1YC20HXDA6'
df6.loc[df6['Model'].str.contains('GSG108TV A6DU3COMPRESSOR'), 'Model']='GSG108TVA6DU3'
df6.loc[df6['Model'].str.contains('ACCESSORIESFORAIRCONDITIONERS GSD120RKQA6JV6B'), 'Model']='GSD120RKQA6JV6B'
df6.loc[df6['Model'].str.contains(' MODEL JT16KCVDYR TF '), 'Model']='JT16KCVDYRTF'
df6.loc[df6['Model'].str.contains(' KSN98D34UFZ '), 'Model']='KSN98D34UFZ'
df6.loc[df6['Model'].str.contains(' KSF155S2VFPC3 '), 'Model']='KSF155S2VFPC3'
df6.loc[df6['Model'].str.contains(' KSN98D34UFZ  ACA20CMP06001 '), 'Model']='KSN98D34UFZ'
df6.loc[df6['Model'].str.contains(' KSG220S1VMP ACW5CMP01201 '), 'Model']='KSG220S1VMP'
df6.loc[df6['Model'].str.contains(' KSF190S2VFPC'), 'Model']='KSF190S2VFPC'
df6.loc[df6['Model'].str.contains(' KSN108D43UFZ'), 'Model']='KSN108D43UFZ'
df6.loc[df6['Model'].str.contains(' DSG089MDA 9389556009'), 'Model']='DSG089MDA'
df6.loc[df6['Model'].str.contains('ACCESSORIES KSG175S1VKP  ACA20CMP02701  W O'), 'Model']='KSG175S1VKP'
df6.loc[df6['Model'].str.contains('STANDARD KSM140V1VDZ'), 'Model']='KSM140V1VDZ'
df6.loc[df6['Model'].str.contains(' KSN66N11VBZB1  ACT09CMP00301  W O'), 'Model']='KSN66N11VBZB1'
df6.loc[df6['Model'].str.contains(' KSM140V1VDZ  ACW4CMP02001 '), 'Model']='KSM140V1VDZ'
df6.loc[df6['Model'].str.contains(' KSG220S1VKT '), 'Model']='KSG220S1VKT'
df6.loc[df6['Model'].str.contains(' KSN98D66UFZ3 '), 'Model']='KSN98D66UFZ3'
df6.loc[df6['Model'].str.contains(' KSF170S1VKPA '), 'Model']='KSF170S1VKPA'
df6.loc[df6['Model'].str.contains('ACCESSORIES KTQ375V1UMU  CAA32CMP00501  W O'), 'Model']='KTQ375V1UMU'
df6.loc[df6['Model'].str.contains(' KTM240D43UKP  CAA32CMP00601  W O'), 'Model']='KTM240D43UKP'
df6.loc[df6['Model'].str.contains(' KTQ480Y1UMT '), 'Model']='KTQ480Y1UMT'
df6.loc[df6['Model'].str.contains(' KTQ375V1UMU '), 'Model']='KTQ375V1UMU'
df6.loc[df6['Model'].str.contains(' KSN84E12VBDB1  ACE10CMP00501  W O'), 'Model']='KSN84E12VBDB1'
df6.loc[df6['Model'].str.contains(' KSG175S1VKP ACA20CMP02701 '), 'Model']='KSG175S1VKP'
df6.loc[df6['Model'].str.contains(' KSN120D53UFZ3  ACA20CMP12101 '), 'Model']='KSN120D53UFZ3'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSK103D33UEZ3'), 'Model']='KSK103D33UEZ3'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSM125S1VFT'), 'Model']='KSM125S1VFT'
df6.loc[df6['Model'].str.contains('ASSY KTN150D53UKRC3PARTS'), 'Model']='KTN150D53UKRC3'
df6.loc[df6['Model'].str.contains('WHP13300PSDPC8FQ   R290'), 'Model']='WHP13300PSDPC8FQ'
df6.loc[df6['Model'].str.contains(' ZF06KQE TFD 551'), 'Model']='ZF06KQE-TFD-551'
df6.loc[df6['Model'].str.contains('NE56YDNMTCOMPRESSOR'), 'Model']='NE56YDNMT'
df6.loc[df6['Model'].str.contains('NE41YDNMTCOMPRESSOR'), 'Model']='NE41YDNMT'
df6.loc[df6['Model'].str.contains('WHP12900GSKPC8LT8C'), 'Model']='WHP12900GSKPC8LT8C'
df6.loc[df6['Model'].str.contains(' DA150A1T 20FD '), 'Model']='DA150A1T-20FD'
df6.loc[df6['Model'].str.contains(' AG66YCAMTS '), 'Model']='AG66YCAMTS'
df6.loc[df6['Model'].str.contains(' DA330A2T 20MD '), 'Model']='DA330A2T-20MD'
df6.loc[df6['Model'].str.contains(' C 1RZ130H2AAF '), 'Model']='C-1RZ130H2AAF'
df6.loc[df6['Model'].str.contains(' ZP165KFE TFD 522 '), 'Model']='ZP165KFE-TFD-522'
df6.loc[df6['Model'].str.contains('A ANB66FVKMT SERVICE '), 'Model']='ANB66FVKMT'
df6.loc[df6['Model'].str.contains(' ZF11KQE TFD 551'), 'Model']='ZF11KQE-TFD-551'
df6.loc[df6['Model'].str.contains(' FVR L 100 350'), 'Model']='FVRL100350'

df6.loc[df6['Model'].str.contains('KSN108D22UFZCCPGMRF006'), 'Model']='KSN108D22UFZ'
df6.loc[df6['Model'].str.contains(' C 1RVN85H0A '), 'Model']='C-1RVN85H0A'
df6.loc[df6['Model'].str.contains(' KSM120S2VEZL '), 'Model']='KSM120S2VEZL'
df6.loc[df6['Model'].str.contains(' GSL165UY C7EU1 '), 'Model']='GSL165UY-C7EU1'
df6.loc[df6['Model'].str.contains('KTN130D42UFZCCPGMRF010'), 'Model']='KTN130D42UFZ'
df6.loc[df6['Model'].str.contains('NO GSD102SKQA6JT6B'), 'Model']='GSD102SKQA6JT6B'
df6.loc[df6['Model'].str.contains('MODEL GSD108CKQA6JV8B'), 'Model']='GSD108CKQA6JV8B'
df6.loc[df6['Model'].str.contains('KSN108D34UFZ3ACA20CMP10901'), 'Model']='KSN108D34UFZ'
df6.loc[df6['Model'].str.contains(' KSF190S2VFPC3 '), 'Model']='KSF190S2VFPC3'
df6.loc[df6['Model'].str.contains(' KSN98D34UFZACA20CMP06001 '), 'Model']='KSN98D34UFZ'
df6.loc[df6['Model'].str.contains('MKV190CL2B   ASH'), 'Model']='MKV190CL2-ASH'
df6.loc[df6['Model'].str.contains(' KSF170S1VFP3ACA20CMP05101 '), 'Model']='KSF170S1VFP3'
df6.loc[df6['Model'].str.contains(' KSG240S1VKTACA28CMP01701 '), 'Model']='KSG240S1VKT'
df6.loc[df6['Model'].str.contains(' KSM120S2VEZLACW3CMP01401 '), 'Model']='KSM120S2VEZ'
df6.loc[df6['Model'].str.contains(' KS M106S2VFEL '), 'Model']='KSM106S2VFEL'
df6.loc[df6['Model'].str.contains(' KSG210S1VMPACA22CMP02301 '), 'Model']='KSG210S1VMP'
df6.loc[df6['Model'].str.contains('EMY 6170Z220 240V 50HZ'), 'Model']='EMY6170Z'
df6.loc[df6['Model'].str.contains(' KSM106S2VFELACA20CMP05801 '), 'Model']='KSM106S2VFEL'
df6.loc[df6['Model'].str.contains(' KTQ480Y1UMT  CAA34CMP00701 '), 'Model']='KTQ480Y1UMT'
df6.loc[df6['Model'].str.contains(' KTN150D42UFZACA22CMP01501 '), 'Model']='KTN150D42UFZ'
df6.loc[df6['Model'].str.contains('ACCESSORIES KSF170S1VKPA  ACA20CMP03801  W O'), 'Model']='KSF170S1VKPA'
df6.loc[df6['Model'].str.contains(' KTN130D42UFZACA20CMP02201 '), 'Model']='KTN130D42UFZ'
df6.loc[df6['Model'].str.contains(' KSG175S1VKPACA20CMP02701 '), 'Model']='KSG175S1VKP'
df6.loc[df6['Model'].str.contains('ACCESSORIESMODEL ANB66FKMMT'), 'Model']='ANB66FKMMT'
df6.loc[df6['Model'].str.contains(' KTQ375V1UMU  CAA32CMP00501 '), 'Model']='KTQ375V1UMU'
df6.loc[df6['Model'].str.contains(' KTF310D43UMTACA22CMP02801 '), 'Model']='KTF310D43UMT'
df6.loc[df6['Model'].str.contains(' KSF170S1VKPAACA20CMP03801 '), 'Model']='KSF170S1VKPA'
df6.loc[df6['Model'].str.contains('NO BSD122DT P6AU '), 'Model']='BSD122DT-P6AU'
df6.loc[df6['Model'].str.contains('NO BSA645DT R1EN '), 'Model']='BSA645DT-R1EN'
df6.loc[df6['Model'].str.contains(' MODEL  C SB373H8F '), 'Model']='C-SB373H8F'
df6.loc[df6['Model'].str.contains(' KTQ480Y1UMTCAA34CMP00701 '), 'Model']='KTQ480Y1UMT'
df6.loc[df6['Model'].str.contains('E537 PA250G2CS 4KTM2'), 'Model']='PA250G2CS-4KTM2'
df6.loc[df6['Model'].str.contains('CODE 60024512ZR61KE TFM 52E'), 'Model']='ZR61KETFM52E'
df6.loc[df6['Model'].str.contains('NO TE638RC3Q9RK '), 'Model']='TE638RC3Q9RK'
df6.loc[df6['Model'].str.contains(' FVR L 70 230'), 'Model']='FVRL70230'
df6.loc[df6['Model'].str.contains(' KSF155N2VETB3 '), 'Model']='KSF155N2VETB3'
df6.loc[df6['Model'].str.contains('ASSY KTN150D53UKRC3'), 'Model']='KTN150D53UKRC3'
df6.loc[df6['Model'].str.contains('BF1Z2584423 013000 2Z25 84 42Y'), 'Model']='2Z25-84-42Y'
df6.loc[df6['Model'].str.contains('ACCESSORIES ANB87FGBMT '), 'Model']='ANB87FGBMT'
df6.loc[df6['Model'].str.contains(' KTN130D53UFE3 '), 'Model']='KTN130D53UFE3'
df6.loc[df6['Model'].str.contains('CODE 60024423ZP61KUE TFM 52E '), 'Model']='ZP61KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('CODE 60024193ZR72KC TFD 52E'), 'Model']='ZR72KC-TFD-52E'
df6.loc[df6['Model'].str.contains('CODE 60024193ZP36KUE TFM 52E'), 'Model']='ZP36KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('ACCESSORIES AN66YQNMT '), 'Model']='AN66YQNMT'
df6.loc[df6['Model'].str.contains('CSV35103V 3Y V35 103Y'), 'Model']='CSV35103V-3Y-V35-103Y'
df6.loc[df6['Model'].str.contains('CODE 60024423ZR108KF TFD 522'), 'Model']='ZR108KF-TFD-522'
df6.loc[df6['Model'].str.contains('CODE 60024193ZR108KF TFD 522'), 'Model']='ZR108KF-TFD-522'
df6.loc[df6['Model'].str.contains('NO YM350E1G 100 '), 'Model']='YM350E1G-100'
df6.loc[df6['Model'].str.contains('M NO ZP42KUE TFM 52E'), 'Model']='ZP42KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('M NO ZP36KUE TFM 52E'), 'Model']='ZP36KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('F3BL VVR 150 380 60 VFDE'), 'Model']='F3BL-VVR-150-380-60-VFDE'
df6.loc[df6['Model'].str.contains('FR3BL 2 0VR VFD'), 'Model']='FR3BL2--0VR-VFD'
df6.loc[df6['Model'].str.contains(' ANB66FKMT1'), 'Model']='ANB66FKMT1'
df6.loc[df6['Model'].str.contains('NO ZE49KME TFM 52ESCROLL'), 'Model']='ZE49KME-TFM-52E'
df6.loc[df6['Model'].str.contains('NO ZE49KME TFM 52E'), 'Model']='ZE49KME-TFM-52E'
df6.loc[df6['Model'].str.contains('NO YF45E1G Q100 '), 'Model']='YF45E1G-Q100'
df6.loc[df6['Model'].str.contains('NN34H9112APTT5 '), 'Model']='NN34H9112APTT5'
df6.loc[df6['Model'].str.contains('M3U0 ANB78FTBMT'), 'Model']='ANB78FTBMT'
df6.loc[df6['Model'].str.contains('VZH088AGANA'), 'Model']='VZH088AGANA'
df6.loc[df6['Model'].str.contains('SAMPLES KSN120D43UF4C31'), 'Model']='KSN120D43UF4C31'
df6.loc[df6['Model'].str.contains('NUE6214U '), 'Model']='NUE6214U'
df6.loc[df6['Model'].str.contains('NUE6210U '), 'Model']='NUE6210U'
df6.loc[df6['Model'].str.contains('DKT208MAB B11EILB'), 'Model']='DKT208MAB-B11EILB'
df6.loc[df6['Model'].str.contains('BF1V1042293 013000 2V10 42 29Y'), 'Model']='2V10-42-29Y'
df6.loc[df6['Model'].str.contains('5RD132XBB21'), 'Model']='5RD132XBB21'
df6.loc[df6['Model'].str.contains('ZP143KCE TFD 423 FOC '), 'Model']='ZP143KCE-TFD-423'
df6.loc[df6['Model'].str.contains('M3T8 ANB66FTBMT'), 'Model']='ANB66FTBMT'
df6.loc[df6['Model'].str.contains('KSN89D21UEZ31 COMPRESSOR'), 'Model']='KSN89D21UEZ31'
df6.loc[df6['Model'].str.contains('H648 DKT208MAH'), 'Model']='DKT208MAH'
df6.loc[df6['Model'].str.contains('EUW86E87RAW'), 'Model']='EUW86E87RAW'
df6.loc[df6['Model'].str.contains('DNB36FANMT FOR'), 'Model']='DNB36FANMT'
df6.loc[df6['Model'].str.contains('CWFVRL125430 3Y003 FVR L 125 430'), 'Model']='FVR-L-125-430'
df6.loc[df6['Model'].str.contains('COMPRESSOR KSN140D63UFZ3 F O C '), 'Model']='KSN140D63UFZ3'
df6.loc[df6['Model'].str.contains('ACCESSORIESMODEL ANB33FQFMT'), 'Model']='ANB33FQFMT'
df6.loc[df6['Model'].str.contains(' BG96YAAMTS '), 'Model']='BG96YAAMTS'
df6.loc[df6['Model'].str.contains(' BG82YAAMTS '), 'Model']='BG82YAAMTS'
df6.loc[df6['Model'].str.contains('FH COPELAND 111ZR380KCE TWD 523'), 'Model']='ZR380KCE-TWD-523'
df6.loc[df6['Model'].str.contains('FH COPELAND 111ZR310KCE TWD 523'), 'Model']='ZR310KCE-TWD-523'
df6.loc[df6['Model'].str.contains('LSGP 40 EWL X0000'), 'Model']='LSGP-40-EWL-X0000'
df6.loc[df6['Model'].str.contains('FR4AXL 3 0VR 400V 50HZ'), 'Model']='FR4AXL-3-0VR'
df6.loc[df6['Model'].str.contains('EMX S 55CLC SK '), 'Model']='EMX-S-55CLC'
df6.loc[df6['Model'].str.contains('CWFVRL050160 3Y003 FVR L 50 160'), 'Model']='FVR-L-50-160'
df6.loc[df6['Model'].str.contains('CTS45SBAD400 200'), 'Model']='CTS45SBAD'
df6.loc[df6['Model'].str.contains('BF1V1556323 013000 2V15 56 32Y'), 'Model']='2V15-56-32Y'
df6.loc[df6['Model'].str.contains('ANB66FGDMT  POA0124 '), 'Model']='ANB66FGDMT'
df6.loc[df6['Model'].str.contains(' ANB66FGDMT '), 'Model']='ANB66FGDMT'
df6.loc[df6['Model'].str.contains('FH COPELAND 111ZR250KCE TWD 523'), 'Model']='ZR250KCE-TWD-523'
df6.loc[df6['Model'].str.contains('CWFVRL100350 3Y003FVR L 100'), 'Model']='FVR-L-100'

df6.loc[df6['Model'].str.contains('GSX073CKTA6JT8A '), 'Model']='GSX073CKTA6JT8A'
df6.loc[df6['Model'].str.contains('KSK89D53UFZ '), 'Model']='KSK89D53UFZ'
df6.loc[df6['Model'].str.contains('NO14D7151ALTT9'), 'Model']='NO14D7151ALTT9'
df6.loc[df6['Model'].str.contains('KSF165S2VFPC3 '), 'Model']='KSF165S2VFPC3'
df6.loc[df6['Model'].str.contains('HSM215V03VDZ '), 'Model']='HSM215V03VDZ'
df6.loc[df6['Model'].str.contains('KSF170V01VFTB  AS'), 'Model']='KSF170V01VFTB'
df6.loc[df6['Model'].str.contains(' KSG220S1VMP  W O'), 'Model']=' KSG220S1VMP'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN133D42UFZ'), 'Model']='ACCESSORIESKSN133D42UFZ'
df6.loc[df6['Model'].str.contains('TAG2525Z  '), 'Model']='TAG2525Z'
df6.loc[df6['Model'].str.contains('GTH345UC3C9EUC '), 'Model']='GTH345UC3C9EUC'
df6.loc[df6['Model'].str.contains('KTQ440Y1UMTA COMPRESSOR'), 'Model']='KTQ440Y1UMTA'
df6.loc[df6['Model'].str.contains('COMPRESSOR MODEL MLZ076T4LC9A'), 'Model']='MLZ076T4LC9A'
df6.loc[df6['Model'].str.contains('DTN210D32UFZ '), 'Model']='DTN210D32UFZ'
df6.loc[df6['Model'].str.contains('DTN210D22UFZ '), 'Model']='DTN210D22UFZ'
df6.loc[df6['Model'].str.contains(' KSN66N11VBZB'), 'Model']=' KSN66N11VBZB'
df6.loc[df6['Model'].str.contains('4MH2 25 AWM D N X0000 COMPRESSOR'), 'Model']='4MH2-25-AWM-D-N-X0000'
df6.loc[df6['Model'].str.contains('SFZ501853  Z50185Y'), 'Model']='Z50-18-5Y'
df6.loc[df6['Model'].str.contains('SFZ501683  Z50168Y'), 'Model']='Z50-16-8Y'
df6.loc[df6['Model'].str.contains('CSZ40126Z 3Y '), 'Model']='CS-Z40-12-6Z-3Y'
df6.loc[df6['Model'].str.contains('E537 PA250G2CS 4KTM2PARTS'), 'Model']='PA250G2CS-4KTM2'
df6.loc[df6['Model'].str.contains('D504 PA291X3CS 4MTM1PARTS'), 'Model']='PA291X3CS-4MTM1'
df6.loc[df6['Model'].str.contains(' ATH550SDPC9EQ '), 'Model']='ATH550SDPC9EQ'

df6.loc[df6['Model'].str.contains(' KSG220S1VMP  W O'), 'Model']='KSG220S1VMP'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN133D42UFZ'), 'Model']='KSN133D42UFZ'
df6.loc[df6['Model'].str.contains(' KSN66N11VBZB'), 'Model']='KSN66N11VBZB'

df6.loc[df6['Model'].str.contains('LJ68WU1 '), 'Model']='LJ68WU1'
df6.loc[df6['Model'].str.contains('5CW073ZA02 '), 'Model']='5CW073ZA02'
df6.loc[df6['Model'].str.contains('NN54T9902AVTS3 '), 'Model']='NN54T9902AVTS3'
df6.loc[df6['Model'].str.contains('M L90FBAM 220V 240V '), 'Model']='ML90FBAM'
df6.loc[df6['Model'].str.contains('D 2 15 1Y'), 'Model']='D2151Y'
df6.loc[df6['Model'].str.contains('AV195ET 037 B4 '), 'Model']='AV195ET037B4'
df6.loc[df6['Model'].str.contains('HSM190S1VFT '), 'Model']='HSM190S1VFT'
df6.loc[df6['Model'].str.contains(' DST102MMA C11EIL'), 'Model']=' DST102MMA-C11EIL'
df6.loc[df6['Model'].str.contains('NO YM240E1G 105 '), 'Model']='YM240E1G105'
df6.loc[df6['Model'].str.contains('MODEL  ATE550UC3Q9RK '), 'Model']='ATE550UC3Q9RK'
df6.loc[df6['Model'].str.contains('M NO ZF28KQE TFD 551'), 'Model']='ZF28KQE-TFD-551'
df6.loc[df6['Model'].str.contains('COMPRESSORPZ99H1E 9'), 'Model']='PZ99H1E-9'

df6.loc[df6['Model'].str.contains('BS5 S4G 12 2Y 40P '), 'Model']='BS5-S4G-12-2Y-40P'
df6.loc[df6['Model'].str.contains('COMPPZ99H1E 9'), 'Model']='PZ99H1E-9'
df6.loc[df6['Model'].str.contains('AE4440YGH '), 'Model']='AE4440YGH'
df6.loc[df6['Model'].str.contains(' PTR H310PG4'), 'Model']='PTR-H310PG4'

df6.loc[df6['Model'].str.contains('NO GSD102SKQA6JV6BCOMPRESSOR'), 'Model']='GSD102SKQA6JV6B'
df6.loc[df6['Model'].str.contains('GSD102SKQA6JV6BCOMPRESSOR'), 'Model']='GSD102SKQA6JV6B'

df6.loc[df6['Model'].str.contains('MODEL GSD102RKQA6JT6BCOMPRESSOR'), 'Model']='GSD102RKQA6JT6B'
df6.loc[df6['Model'].str.contains(' MODEL  GSD108RKRA6JV8B'), 'Model']='GSD108RKRA6JV8B'
df6.loc[df6['Model'].str.contains(' KSG220S1VMP'), 'Model']='KSG220S1VMP'
df6.loc[df6['Model'].str.contains('KSF210S2VFPC3  AS'), 'Model']='KSF210S2VFPC3'
df6.loc[df6['Model'].str.contains('KSM130V02VDZ  AS'), 'Model']='KSM130V02VDZ'
df6.loc[df6['Model'].str.contains('EHA0   CV'), 'Model']='EHA0-CV'
df6.loc[df6['Model'].str.contains('SFZ501403  Z50140Y'), 'Model']='SFZ501403-Z50140Y'
df6.loc[df6['Model'].str.contains('ZP51KSE TFM  52E'), 'Model']='ZP51KSE-TFM-52E'
df6.loc[df6['Model'].str.contains('ZP50K3E TFD  522'), 'Model']='ZP50K3E-TFD-522'
df6.loc[df6['Model'].str.contains('CSZ50154Z 3Y '), 'Model']='CSZ50154Z-3Y'
df6.loc[df6['Model'].str.contains('MTZ22 4VI '), 'Model']='MTZ22-4VI'
df6.loc[df6['Model'].str.contains('ZP50K3E PFZ  522'), 'Model']='ZP50K3E-PFZ-522'
df6.loc[df6['Model'].str.contains('SFV401233  V40123Y'), 'Model']='SFV401233-V40123Y'
df6.loc[df6['Model'].str.contains('SFV301233  V30123Y'), 'Model']='SFV301233-V30123Y'
df6.loc[df6['Model'].str.contains('N ANB66FVFMT SERVICE '), 'Model']='ANB66FVFMT'
df6.loc[df6['Model'].str.contains('FR3BL2  0VR VFD'), 'Model']='FR3BL2-0VR-VFD'
df6.loc[df6['Model'].str.contains('MODEL GSD088RKQA6JK6B'), 'Model']='GSD088RKQA6JK6B'
df6.loc[df6['Model'].str.contains('COMPRESSORAXE433U FZ3C'), 'Model']='AXE433U-FZ3C'
df6.loc[df6['Model'].str.contains('ZB76KQE TFD 551460 3 60'), 'Model']='ZB76KQE-TFD-551'
df6.loc[df6['Model'].str.contains('GSD102RKQA6JT6BCOMPRESSOR'), 'Model']='GSD102RKQA6JT6B'
df6.loc[df6['Model'].str.contains('GSD108RKRA6JV8BACOMPRESSOR'), 'Model']='GSD108RKRA6JV8BA'
df6.loc[df6['Model'].str.contains('NO GSD102SKQA6JV6'), 'Model']='GSD102SKQA6JV6'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN108D43UFZA'), 'Model']='KSN108D43UFZA'
df6.loc[df6['Model'].str.contains('SERIESCSB035NKEG'), 'Model']='CSB035NKEG'
df6.loc[df6['Model'].str.contains('EM 55HHR220'), 'Model']='EM-55HHR'
df6.loc[df6['Model'].str.contains('ZR380KCE-TWD-52231117111'), 'Model']='ZR380KCE-TWD-522'
df6.loc[df6['Model'].str.contains('COM08291CSHC093K0A0'), 'Model']='CSHC093K0A0'
df6.loc[df6['Model'].str.contains('COM08289CSHC075K0A0'), 'Model']='CSHC075K0A0'
df6.loc[df6['Model'].str.contains('COM08246SSE057A4BPZ'), 'Model']='SSE057A4BPZ'
df6.loc[df6['Model'].str.contains('COM06434CSHA140K0E0'), 'Model']='CSHA140K0E0'
df6.loc[df6['Model'].str.contains('COM06426CSHA093K0E0'), 'Model']='CSHA093K0E0'
df6.loc[df6['Model'].str.contains('COM12474E405DHD38D2'), 'Model']='E405DHD38D2'
df6.loc[df6['Model'].str.contains('COM09843CSHN374K0BK'), 'Model']='CSHN374K0BK'
df6.loc[df6['Model'].str.contains('51390303498XYVEMT9C'), 'Model']='VEMT9C'
df6.loc[df6['Model'].str.contains('SCROLLYH355C7210'), 'Model']='YH355C7210'
df6.loc[df6['Model'].str.contains('MKV190CL2J   SM1'), 'Model']='MKV190CL2J-SM1'
df6.loc[df6['Model'].str.contains('NEK614 4GK B '), 'Model']='NEK6144GK'
df6.loc[df6['Model'].str.contains('H300CC '), 'Model']='H300CC'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSM113S2VERL'), 'Model']='KSM113S2VERL'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSF175S1VFPB'), 'Model']='KSF175S1VFPB'
df6.loc[df6['Model'].str.contains('GTD130UKSA7JV8BAIR'), 'Model']='GTD130UKSA7JV8B'
df6.loc[df6['Model'].str.contains('PCAJ4517ZF2'), 'Model']='CAJ4517ZF2'
df6.loc[df6['Model'].str.contains('ANB78FVAMTSCROLL'), 'Model']='ANB78FVAMT'
df6.loc[df6['Model'].str.contains('ANB52FKFMTSCROLL'), 'Model']='ANB52FKFMT'
df6.loc[df6['Model'].str.contains('COMPRESSORPW2.5VKMF'), 'Model']='PW2.5VKMF'
df6.loc[df6['Model'].str.contains('ARABCA105770'), 'Model']='VMH1113Y'
df6.loc[df6['Model'].str.contains('IFMXY6C'), 'Model']='FMXY6C'
df6.loc[df6['Model'].str.contains('GMCCSZ90E1J N'), 'Model']='SZ90E1J-N'
df6.loc[df6['Model'].str.contains('10NFIDZ90X1F'), 'Model']='DZ90X1F'
df6.loc[df6['Model'].str.contains('COMPVFL 11 OCY1'), 'Model']='VFL110CY1'
df6.loc[df6['Model'].str.contains('SC12CLX2220V'), 'Model']='SC12CLX2'
df6.loc[df6['Model'].str.contains('NE K2150GK'), 'Model']='NEK2150GK'
df6.loc[df6['Model'].str.contains('121L1396HLJ072T4LC6'), 'Model']='HLJ072T4LC6'
df6.loc[df6['Model'].str.contains('120G0306VZH088CGAMA'), 'Model']='VZH088CGAMA'
df6.loc[df6['Model'].str.contains('120H0254SH380A4AAE'), 'Model']='SH380A4AAE'
df6.loc[df6['Model'].str.contains('120G0357VZH044CGAMB'), 'Model']='VZH044CGAMB'
df6.loc[df6['Model'].str.contains('120G0307VZH088CJDM'), 'Model']='VZH088CJDM'
df6.loc[df6['Model'].str.contains('PZ65E 1B 9YEL'), 'Model']='PZ65E1B-9'
df6.loc[df6['Model'].str.contains('VES2456C5F00'), 'Model']='VES'
df6.loc[df6['Model'].str.contains('KSK103D33SERC3ROTARY'), 'Model']='KSK103D33SERC3'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSF170S1VKPA'), 'Model']='KSF170S1VKPA'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN140D21UFZ'), 'Model']='KSN140D21UFZ'
df6.loc[df6['Model'].str.contains('ACCESSORIESASG230V1SKP'), 'Model']='ASG230V1SKP'
df6.loc[df6['Model'].str.contains('SJ96FOR'), 'Model']='SJ96'
df6.loc[df6['Model'].str.contains('MM 1110Y'), 'Model']='MM1110Y'
df6.loc[df6['Model'].str.contains('T 1114Y'), 'Model']='T1114Y'
df6.loc[df6['Model'].str.contains('T 1112Y'), 'Model']='T1112Y'
df6.loc[df6['Model'].str.contains('BESF1'), 'Model']='VESF1'
df6.loc[df6['Model'].str.contains('REFRIGERATORCOMPRESSORATD66XL'), 'Model']='ATD66XL'
df6.loc[df6['Model'].str.contains('COMPRESSORATA80XL'), 'Model']='ATA80XL'
df6.loc[df6['Model'].str.contains('REFRIGERATORCOMPRESSORAKD46K'), 'Model']='AKD46K'
df6.loc[df6['Model'].str.contains('COMPRESSORAKD55X'), 'Model']='AKD55X'
df6.loc[df6['Model'].str.contains('213NI34N9802ADSA1'), 'Model']='NI34N9802ADSA1'
df6.loc[df6['Model'].str.contains('VFL 126CY'), 'Model']='VFL126CY'
df6.loc[df6['Model'].str.contains('EM25 75HLP'), 'Model']='EM2S-75HLP'
df6.loc[df6['Model'].str.contains('EM25 70CLP'), 'Model']='EM2S-70CLP'
df6.loc[df6['Model'].str.contains('UB AR 090FE6TS'), 'Model']='UB1AR1090FE6TS'
df6.loc[df6['Model'].str.contains('UB9TA9 80FEQTS'), 'Model']='UB9TA9180FEQTS'
df6.loc[df6['Model'].str.contains('MSV4A ALR TSJ'), 'Model']='MSV4A1AL1RTSJ'
df6.loc[df6['Model'].str.contains('MSE 66CL H ASH'), 'Model']='MSE166CL1H ASH'
df6.loc[df6['Model'].str.contains('NF54M7 5 ANASH'), 'Model']='NF54M7151ANASH'
df6.loc[df6['Model'].str.contains('NI34T9 02ADTS7'), 'Model']='NI34T9102ADTS7'
df6.loc[df6['Model'].str.contains('ATA72X  OC '), 'Model']='ATA72X'
df6.loc[df6['Model'].str.contains('VTX1113YA '), 'Model']='VTX1113YA'
df6.loc[df6['Model'].str.contains('9Z99H1E'), 'Model']='PZ99H1E'
df6.loc[df6['Model'].str.contains('EXA 40K'), 'Model']='EXA40K'
df6.loc[df6['Model'].str.contains('VNTZ165'), 'Model']='VNTZ165'
df6.loc[df6['Model'].str.contains(' KSG175S1VKP '), 'Model']='KSG175S1VKP'
df6.loc[df6['Model'].str.contains('PZ99HIE'), 'Model']='PZ99H1E'
df6.loc[df6['Model'].str.contains('PW2 OVVKMF'), 'Model']='PW2 0VKMF'
df6.loc[df6['Model'].str.contains('LQDZY75G'), 'Model']='L QDZY75G'
df6.loc[df6['Model'].str.contains('L72CZI'), 'Model']='L72CZ1'
df6.loc[df6['Model'].str.contains('VATZ72 L'), 'Model']='VATZ72L'
df6.loc[df6['Model'].str.contains('PW2 5VKMF'), 'Model']='PW2.5VKMF'
df6.loc[df6['Model'].str.contains('DZ90ZIC'), 'Model']='DZ90Z1C'
df6.loc[df6['Model'].str.contains('SZ85E1H 9'), 'Model']='SZ85E1H9'
df6.loc[df6['Model'].str.contains('PZ65E1B 9YEL'), 'Model']='PZ65E1B-9'
df6.loc[df6['Model'].str.contains('PZ90H1Y 9'), 'Model']='PZ90H1Y-9'
df6.loc[df6['Model'].str.contains('AU130WY11A'), 'Model']='AU130WY1A'
df6.loc[df6['Model'].str.contains('BSA90NJMV'), 'Model']='BSA90NJMV'
df6.loc[df6['Model'].str.contains('PE90H1F 9'), 'Model']='PE90H1F-9'
df6.loc[df6['Model'].str.contains('VTG11113Y'), 'Model']='VTG1113Y'
df6.loc[df6['Model'].str.contains('NL73MF'), 'Model']='NL7.3MF'
df6.loc[df6['Model'].str.contains('EZ90H1W U URBL'), 'Model']='EZ90H1W-URBL'
df6.loc[df6['Model'].str.contains('WR01F04537'), 'Model']='MKH113L2'
df6.loc[df6['Model'].str.contains('WR01F04600'), 'Model']='EZ75HIY-U'
df6.loc[df6['Model'].str.contains('W10666797'), 'Model']='EGYS 90CLP'
df6.loc[df6['Model'].str.contains('A21258104'), 'Model']='EL60H'
df6.loc[df6['Model'].str.contains('A21258102'), 'Model']='EL100H'
df6.loc[df6['Model'].str.contains('A20732401 '), 'Model']='ECLA002'
df6.loc[df6['Model'].str.contains('WR01F05170'), 'Model']='VEMY'
df6.loc[df6['Model'].str.contains('268GA51M1AJ'), 'Model']='NEU6212Z'
df6.loc[df6['Model'].str.contains('120G0358VZH044CGAMA'), 'Model']='VZH044CGAMA'
df6.loc[df6['Model'].str.contains('120G0146VZH065CGANB'), 'Model']='VZH065CGANB'
df6.loc[df6['Model'].str.contains('COMPVFL110CY1'), 'Model']='VFL110CY1'
df6.loc[df6['Model'].str.contains('GTD150RKRF6JV8BCOMPRESSOR'), 'Model']='GTD150RKRF6JV8B'
df6.loc[df6['Model'].str.contains('GTD186RKQA8JT8CCOMPRESSOR'), 'Model']='GTD186RKQA8JT8C'
df6.loc[df6['Model'].str.contains('GTD130UKSF6JV8BCOMPRESSOR'), 'Model']='GTD130UKSF6JV8B'
df6.loc[df6['Model'].str.contains('GSX088RKUA6JT8AIR'), 'Model']='GSX088RKUA6JT8'
df6.loc[df6['Model'].str.contains('EY40L58W'), 'Model']='EY40L'
df6.loc[df6['Model'].str.contains('EY40L50W'), 'Model']='EY40L'
df6.loc[df6['Model'].str.contains('ATA 72XL'), 'Model']='ATA72XL'
df6.loc[df6['Model'].str.contains('ATA 80XL'), 'Model']='ATA80XL'
df6.loc[df6['Model'].str.contains('ATD 66XL'), 'Model']='ATD66XL'
df6.loc[df6['Model'].str.contains('ATD 60XL'), 'Model']='ATD60XL'
df6.loc[df6['Model'].str.contains('VEKY 70L'), 'Model']='VEKY70L'
df6.loc[df6['Model'].str.contains('AN 150ML'), 'Model']='AN150ML'

df6.loc[df6['Model'].str.contains('NUK140NK160'), 'Model']='NUK140NK'
df6.loc[df6['Model'].str.contains('COMPRESSORSE45E1J 9'), 'Model']='SE45E1J-9'
df6.loc[df6['Model'].str.contains('COMPRESSORPE65H1H 9'), 'Model']='PE65H1H-9'
df6.loc[df6['Model'].str.contains('COMPRESSORPE90H1F 9'), 'Model']='PE90H1F-9'
df6.loc[df6['Model'].str.contains('COMPRESSORSE50H1F 9'), 'Model']='SE50H1F-9'
df6.loc[df6['Model'].str.contains('COMPRESSORSE40E1K 9'), 'Model']='SE40E1K-9'
df6.loc[df6['Model'].str.contains('COMPRESSORPZ90H1Y 3'), 'Model']='PZ90H1Y-3'
df6.loc[df6['Model'].str.contains('COMPRESSORBSA075NHMV'), 'Model']='BSA075NHMV'
df6.loc[df6['Model'].str.contains('COMPRESSORBMK110NHMV'), 'Model']='BMK110NHMV'
df6.loc[df6['Model'].str.contains('COMPRESSORBSA090NAMV'), 'Model']='BSA090NAMV'
df6.loc[df6['Model'].str.contains('39X1H3GJ&5JMA0010735394'), 'Model']='39X1H3GJ&5JMA'
df6.loc[df6['Model'].str.contains('NL 11 MF'), 'Model']='NL11MF'
df6.loc[df6['Model'].str.contains('COMPRESSORNUY90RA B'), 'Model']='NUY90RA.B'
df6.loc[df6['Model'].str.contains('COMPRESSORFTA55L'), 'Model']='FTA55L'
df6.loc[df6['Model'].str.contains('000000000A22691602'), 'Model']='T1110DY'
df6.loc[df6['Model'].str.contains('000000000A23010706'), 'Model']='TT1116DY'
df6.loc[df6['Model'].str.contains('KSK 103D33VE73'), 'Model']='KSK103D33VE73'

df6.loc[df6['Model'].str.contains('EMC 3134U'), 'Model']='EMC 3134U'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSG210S1VMP'), 'Model']='KSG210S1VMP'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSF210S2VFPC3'), 'Model']='KSF210S2VFPC3'
df6.loc[df6['Model'].str.contains('ACCESSORIESASG190V1SKP'), 'Model']='ASG190V1SKP'
df6.loc[df6['Model'].str.contains('GS21 MLX'), 'Model']='GS21MLX'
df6.loc[df6['Model'].str.contains('(MODEL:2YC71AXD-A2)'), 'Model']='2YC71AXD-A2'
df6.loc[df6['Model'].str.contains('(2YC71AXD-A2)'), 'Model']='2YC71AXD-A2'
df6.loc[df6['Model'].str.contains(' (MODEL:2YC71AXD#A6)'), 'Model']='2YC71AXD#A6'
df6.loc[df6['Model'].str.contains('(MODEL:2YC71AXD#A6) '), 'Model']='2YC71AXD#A6'
df6.loc[df6['Model'].str.contains('(MODEL: 2Y350APAX1N#A)'), 'Model']='2Y350APAX1N#A '
df6.loc[df6['Model'].str.contains('(2Y350APAX2N#A)'), 'Model']='2Y350APAX2N#A'
df6.loc[df6['Model'].str.contains('(MODEL:2Y350APAXIN#A)'), 'Model']='2Y350APAXIN#A'
df6.loc[df6['Model'].str.contains('ZRD125KCE-TFD-522'), 'Model']='ZR125KCE-TFD-522'
df6.loc[df6['Model'].str.contains('(EDTM280D85EMS) '), 'Model']='EDTM280D85EMS'
df6.loc[df6['Model'].str.contains('60032000ZR72KC-TFD-52E'), 'Model']='ZR72KC-TFD-52E'
df6.loc[df6['Model'].str.contains('60032000ZR72KC-TFD-52E'), 'Model']='ZR72KC-TFD-52E'
df6.loc[df6['Model'].str.contains('60032526ZR61KE-TFM-52E'), 'Model']='ZR61KE-TFM-52E'
df6.loc[df6['Model'].str.contains(' 60031678ZP61KUE-TFM-52ESCROLL'), 'Model']='ZP61KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('E655DHD-65D2YG'), 'Model']='E655DHD-65D2YG'
df6.loc[df6['Model'].str.contains('60032526ZP36KUE-TFM-52E'), 'Model']='ZP36KUE-TFM-52E'
df6.loc[df6['Model'].str.contains('60031678ZR48K3-TFD-522'), 'Model']='ZR48K3-TFD-522'
df6.loc[df6['Model'].str.contains('WR01F04736'), 'Model']='EZ90H1V-U'
df6.loc[df6['Model'].str.contains('WR01F04881'), 'Model']='EZ65HIY-U'
df6.loc[df6['Model'].str.contains(' DST102MMA-C11EIL'), 'Model']='DST102MMA-C11EIL'
df6.loc[df6['Model'].str.contains(' KSN92V11VDE3'), 'Model']='KSN92V11VDE3'
df6.loc[df6['Model'].str.contains(' KTM240D43UKP '), 'Model']='KTM240D43UKP'

df6.loc[df6['Model'].str.contains('VETX72LCOMPRESSOR'), 'Model']='VETX72L'
df6.loc[df6['Model'].str.contains('VETZ110LCOMPRESSOR'), 'Model']='VETZ110L'
df6.loc[df6['Model'].str.contains('KK480WUPART'), 'Model']='KK480WU'
df6.loc[df6['Model'].str.contains('VTX1111YCOMPRESSOR'), 'Model']='VTX1111Y'
df6.loc[df6['Model'].str.contains('KSN108D31UEZ31COMPRESSOR'), 'Model']='KSN108D31UEZ31'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN108D22UFZ'), 'Model']='KSN108D22UFZ'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN120D53UFZ3'), 'Model']='KSN120D53UFZ3'
df6.loc[df6['Model'].str.contains('ACCESSORIESKTN130D42UFZ'), 'Model']='KTN130D42UFZ'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSF160S2VFPC3'), 'Model']='KSF160S2VFPC3'
df6.loc[df6['Model'].str.contains('ACCESSORIESKTN150D42UFZ'), 'Model']='KTN150D42UFZ'
df6.loc[df6['Model'].str.contains('KSN108D66UFZ3COMPRESSOR'), 'Model']='KSN108D66UFZ3'
df6.loc[df6['Model'].str.contains('ECP0055D127V'), 'Model']='ECP0055D'
df6.loc[df6['Model'].str.contains('MODELO9CE051WA01'), 'Model']='9CE051WA01'
df6.loc[df6['Model'].str.contains('MODELO9CE051QA01'), 'Model']='9CE051QA01'
df6.loc[df6['Model'].str.contains('MODELO9CE051WA01'), 'Model']='9CE051WA01'
df6.loc[df6['Model'].str.contains('MODELOZR57KE TF7'), 'Model']='ZR57KE-TF7'
df6.loc[df6['Model'].str.contains('MODELOZB76KQE TF5'), 'Model']='ZB76KQE-TF5'
df6.loc[df6['Model'].str.contains('MODELOYS26KAE TF5'), 'Model']='YS26KAE-TF5'
df6.loc[df6['Model'].str.contains('MODELOZF41K5E TFD'), 'Model']='ZF41K5E-TFD'
df6.loc[df6['Model'].str.contains('ELOZS15KAE PFJ'), 'Model']='ZS15KAE-PFJ'
df6.loc[df6['Model'].str.contains('MODELOCS20K6ME TF5'), 'Model']='CS20K6ME-TF5'
df6.loc[df6['Model'].str.contains('MODELOLNB65FAGMC'), 'Model']='LNB65FAGMC'
df6.loc[df6['Model'].str.contains('MODELOGJT240MA'), 'Model']='GJT240MA'

df6.loc[df6['Model'].str.contains('AZ110WY1HERMETIC'), 'Model']='AZ110WY1A'
df6.loc[df6['Model'].str.contains('AZ110WY1AHERETIC'), 'Model']='AZ110WY1A'
df6.loc[df6['Model'].str.contains('COMPRESSORDZ90V1S 4QN'), 'Model']='DZ90V1S-4QN'
df6.loc[df6['Model'].str.contains('PW4.0VKMF220'), 'Model']='PW4.0VKMF'
df6.loc[df6['Model'].str.contains('PW4 0VKMF220'), 'Model']='PW4.0VKMF'
df6.loc[df6['Model'].str.contains('ACCESSORIESEYA60'), 'Model']='EYA60'
df6.loc[df6['Model'].str.contains('ETK103L'), 'Model']='ETK130L'

df6.loc[df6['Model'].str.contains('KSN98D21UEZ31COMPRESSOR'), 'Model']='KSN98D21UEZ31'
df6.loc[df6['Model'].str.contains('ACCESSORIESKSN108D66UFZ3'), 'Model']='KSN108D66UFZ3'
df6.loc[df6['Model'].str.contains('KSK89D35UEZ3ACA20CMP04701'), 'Model']='KSK89D35UEZ3'
df6.loc[df6['Model'].str.contains('KTN130D30UFZCOMPRESSOR '), 'Model']='KTN130D30UFZ'
df6.loc[df6['Model'].str.contains('SVB130FHDMCPARTS'), 'Model']='SVB130FHDMC'
df6.loc[df6['Model'].str.contains('ASM89D10UFZPARTS'), 'Model']='ASM89D10UFZ'
df6.loc[df6['Model'].str.contains('RSA201A036SPARE'), 'Model']='RSA201A036'
df6.loc[df6['Model'].str.contains('AGT201C824FFSPARE'), 'Model']='AGT201C824FF'
df6.loc[df6['Model'].str.contains('AHT201F264DMSPARE'), 'Model']='AHT201F264DM'
df6.loc[df6['Model'].str.contains('AGT201B827HDSPARE'), 'Model']='AGT201B827HD'
df6.loc[df6['Model'].str.contains('5VD420ZAA21SPARE'), 'Model']='5VD420ZAA21'
df6.loc[df6['Model'].str.contains('PUS18KEB145UGSD102SKQA6JV6B'), 'Model']='GSD102SKQA6JV6B'
df6.loc[df6['Model'].str.contains('PUS12KEB131UGSD120RKQA6JV6B'), 'Model']='GSD120RKQA6JV6B'
df6.loc[df6['Model'].str.contains('AK110WY1AHERMETIC'), 'Model']='AK110WY1A'
df6.loc[df6['Model'].str.contains('AZ110WY1AHERMETIC'), 'Model']='AZ110WY1A'
df6.loc[df6['Model'].str.contains('KSN108D34UEZ3ROTARY'), 'Model']='KSN108D34UEZ3'
df6.loc[df6['Model'].str.contains('SVB130FADMCPARTS'), 'Model']='SVB130FADMC'
df6.loc[df6['Model'].str.contains('5JD420PAA02PARTS'), 'Model']='5JD420PAA02'
df6.loc[df6['Model'].str.contains('ASG180V1SKTPARTS'), 'Model']='ASG180V1SKT'
df6.loc[df6['Model'].str.contains('GMCCASM135'), 'Model']='ASM135'
df6.loc[df6['Model'].str.contains('GMCCPA200M2CS'), 'Model']='PA200M2CS'
df6.loc[df6['Model'].str.contains('GMCCPA240M2CS'), 'Model']='PA240M2CS'
df6.loc[df6['Model'].str.contains('121L1401HLJ083T4LC6'), 'Model']='HLJ083T4LC6'
df6.loc[df6['Model'].str.contains('120H1512DSH090A4ALD'), 'Model']='DSH090A4ALD'
df6.loc[df6['Model'].str.contains('120G0156VLZ044TGNE9'), 'Model']='VLZ044TGNE9'
df6.loc[df6['Model'].str.contains('SANCSBP160H36A'), 'Model']='CSBP160H36A'
df6.loc[df6['Model'].str.contains('SANCSBP120H36A'), 'Model']='CSBP120H36A'
df6.loc[df6['Model'].str.contains('TOSPA210M2CS-3KTU2'), 'Model']='PA210M2CS-3KTU2'
df6.loc[df6['Model'].str.contains('45CS33KQMETFD'), 'Model']='CS33KQMETFD'
df6.loc[df6['Model'].str.contains('45CS14K6METFD'), 'Model']='CS14K6METFD'

df6.loc[df6['Model'].str.contains('KSN133D42UFZACA20CMP07801'), 'Model']='KSN133D42UFZ'
df6.loc[df6['Model'].str.contains('KSN98D66UFZ3ACA22CMP03301'), 'Model']='KSN98D66UFZ3'
df6.loc[df6['Model'].str.contains('GSX089JKRA7JT8COMPRESSOR'), 'Model']='GSX089JKRA7JT8'
df6.loc[df6['Model'].str.contains('GSX088RKUA6JT8COMPRESSOR'), 'Model']='GSX088RKUA6JT8'
df6.loc[df6['Model'].str.contains('GSD102TKTA6JV6BCOMPRESSOR'), 'Model']='GSD102TKTA6JV6B'
df6.loc[df6['Model'].str.contains('GSX089JKRA7JT8COMPRESSOR'), 'Model']='GSX089JKRA7JT8'
df6.loc[df6['Model'].str.contains('ATH650SKTC8FQAIR'), 'Model']='ATH650SKTC8FQ'
df6.loc[df6['Model'].str.contains('ATE848SKTQ9JKPAIR'), 'Model']='ATE848SKTQ9JK'
df6.loc[df6['Model'].str.contains('ATE752SKTQ9JKPAIR'), 'Model']='ATE752SKTQ9JK'
df6.loc[df6['Model'].str.contains('GSX088BKQA6JT8ACOMPRESSOR'), 'Model']='GSX088BKQA6JT8A'
df6.loc[df6['Model'].str.contains('GSX102CKTA6JT8COMPRESSOR'), 'Model']='GSX102CKTA6JT8'
df6.loc[df6['Model'].str.contains('VDU070CY12400100542'), 'Model']='VDU070CY1'
df6.loc[df6['Model'].str.contains('T1114Y2400100142'), 'Model']='T1114Y'
df6.loc[df6['Model'].str.contains('123B2714  MX21TGA'), 'Model']='MX21TGA'
df6.loc[df6['Model'].str.contains('123B2132  MX23FBA'), 'Model']='MX23FBA'
df6.loc[df6['Model'].str.contains('QR3 74P'), 'Model']='QR374P'
df6.loc[df6['Model'].str.contains('QR3 90P'), 'Model']='QR390P'
df6.loc[df6['Model'].str.contains('QR3 124P'), 'Model']='QR3124P'
df6.loc[df6['Model'].str.contains('QR3 112P R404A'), 'Model']='QR3112P'
df6.loc[df6['Model'].str.contains('QL3 134PR404A'), 'Model']='QL3134P'
df6.loc[df6['Model'].str.contains('AZ110WY1BHERMETIC'), 'Model']='AZ110WY1B'
df6.loc[df6['Model'].str.contains('VDZ060SYHERMETIC'), 'Model']='VDZ060SY'
df6.loc[df6['Model'].str.contains('45ZS26KAEPFV'), 'Model']='ZS26KAE PFV'
df6.loc[df6['Model'].str.contains('45ZS21KAETF5'), 'Model']='ZS21KAE TF5'
df6.loc[df6['Model'].str.contains('45ZS15KAEPFV'), 'Model']='ZS15KAE PFV'
df6.loc[df6['Model'].str.contains('45ZS26KAE TF5'), 'Model']='ZS26KAE TF5'
df6.loc[df6['Model'].str.contains('45ZS33KAETF5'), 'Model']='ZS33KAE TF5'
df6.loc[df6['Model'].str.contains('45ZS33KAEPFV'), 'Model']='ZS33KAE PFV'
df6.loc[df6['Model'].str.contains('TT1116YTOTAL'), 'Model']='TT1116Y'
df6.loc[df6['Model'].str.contains('TT1114YTOTAL'), 'Model']='TT1114Y'
df6.loc[df6['Model'].str.contains('NT1119YTOTAL'), 'Model']='NT1119Y'
df6.loc[df6['Model'].str.contains('tbz37416201'), 'Model']='JQC068MBA'
df6.loc[df6['Model'].str.contains('VNL1113Y0060707941J'), 'Model']='VNL1113Y'

df6.loc[df6['Model'].str.contains('ACCESSORIESEKZ95'), 'Model']='EKZ95'
df6.loc[df6['Model'].str.contains('ACCESSORIESEKZ75L'), 'Model']='EKZ75L'
df6.loc[df6['Model'].str.contains('VEKY90TOTAL'), 'Model']='VEKY90'
df6.loc[df6['Model'].str.contains('VEXH90TOTAL'), 'Model']='VEXH90'
df6.loc[df6['Model'].str.contains('VETZ9LTOTAL'), 'Model']='VETZ9L'
df6.loc[df6['Model'].str.contains('VEXH90TOTAL'), 'Model']='VEXH90'
df6.loc[df6['Model'].str.contains('EYD68KTOTAL'), 'Model']='EYD68K'
df6.loc[df6['Model'].str.contains('VEXH90TOTAL'), 'Model']='VEXH90'
df6.loc[df6['Model'].str.contains('FTK80MTOTAL'), 'Model']='FTK80M'
df6.loc[df6['Model'].str.contains('VMH1113YGTOTAL'), 'Model']='VMH1113YG'
df6.loc[df6['Model'].str.contains('VNC3124UTOTAL'), 'Model']='VNC3124U'
df6.loc[df6['Model'].str.contains('VMU1113YTOTAL'), 'Model']='VMU1113Y'
df6.loc[df6['Model'].str.contains('VNC3124UTOTAL'), 'Model']='VNC3124U'
df6.loc[df6['Model'].str.contains('GMCCPA80HMZR290'), 'Model']='PA80HMZ'
df6.loc[df6['Model'].str.contains('COMPRESSORNUG100NA'), 'Model']='NUG100NA'
df6.loc[df6['Model'].str.contains('VTG1116YTOTAL'), 'Model']='VTG1116Y'
df6.loc[df6['Model'].str.contains('GVE30AABERUPA'), 'Model']='GVE30AA'

#df6.loc[df6['Model'].str.contains(''), 'Model']=''
#df6.loc[df6['Model'].str.contains(''), 'Model']=''

In [107]:
# Aplica a função para remover modelos curtos
df6 = remover_modelos_curtos(df6)

In [108]:
# Substitui toda instancia de espaço por um traço.
df6.loc[:, 'Model'] = df6['Model'].str.replace(' ', '-')
# A IDEIA AQUI É QUE ANTES PARA DAR MERGE COM O CATALOGO FOI PRECISO RETIRAR TODOS OS CARACTERES ESPECIAIS,
# AGORA EU RECOLOCO (OS CARACTERES ESPECIAIS COSTUMAM SER '-')

## 7.5 COLUNA RANGE

In [109]:
df6.columns

Index(['Date', 'Importer Raw', 'Importer Tax ID', 'Package Type', 'Unit',
       'Carrier Flag', 'Unit Value CIF (USD)', 'Carrier ID', 'Carrier',
       'Payment Method', 'Incoterm', 'Transport Document',
       'Transport Document Date', 'Agreement', 'Customs Declaration ID',
       'Shipment Load Type', 'Bill of Lading', 'Operation Type', 'Regime',
       'Item', 'Agent', 'Agent ID', 'Observations/Code',
       'Observations/Description', 'HS Code', 'HS Description',
       'Merchandise Code', 'Merchandise', 'Full HS Description', 'Player Raw',
       'Description', 'Country of Purchase', 'Country Raw', 'Customs',
       'Transport Method', 'Shipping Port', 'Delivery Port', 'Value (USD)',
       'Calculated Freight (USD)', 'Calculated Insurance (USD)',
       'Gross Weight (Clearance Total)', 'Quantity of Packages', 'Volume',
       'Unit Value FOB (USD)', 'CIF Value (USD)', 'Declaration Quantity',
       'CIF/FOB', 'Year', 'Month', 'Quarter CY', 'Samples', 'Range', 'Model',
       '

In [110]:
# Converte as colunas para numerica
df6.loc[:, 'Value (USD)'] = pd.to_numeric(df6['Value (USD)'], errors='coerce')
df6.loc[:, 'Volume'] = pd.to_numeric(df6['Volume'], errors='coerce')

# Cria uma coluna UNIT PRICE que é o Value / Volume
df6.loc[:, 'UNIT PRICE'] = df6['Value (USD)'] / df6['Volume']


# Modifica a coluna do UNIT PRICE no caso da colombia
if country_name == 'Colombia':
 df6 = modificar_unit_price(df6)

# Aplica a função no dataset.
df6 = range_by_description(df6)

# Crio um filtro. Se o UNIT PRICE for maior que 300 ou menor que 8
filtro = (df6['UNIT PRICE'] > 300) | (df6['UNIT PRICE'] < 8)
# Crio um filtro. Se a tecnologia for SCROLL ou SCREWe
filtro2 = (df6['COMPRESSOR TYPE'] == 'SCROLL') | (df6['COMPRESSOR TYPE'] == 'SCREW') | (df6['COMPRESSOR TYPE'] == 'ROTARY')

# Passa os dois filtros definidos acima para a função 'preenche_range'.
# Então, tudo que cai em alguns dos dois filtros se torna 'Out of Range'
df6 = preenche_range(df6, filtro )
df6 = preenche_range(df6, filtro2 )

# Tudo que é EXTRA é Out of Range
df6.loc[df6['Segment'] == 'EXTRA', 'Range'] = 'Out of Range'
# Tudo que é INTRACOMPANY é Out of Range
df6.loc[df6['Segment'] == 'INTRACOMPANY', 'Range'] = 'Out of Range'

df6.loc[df6['Importer Final'] == 'NIDEC', 'Range'] = 'Range'
df6.loc[df6['Player Final'] == 'NIDEC', 'Range'] = 'Range'

df6.loc[df6['Model'] == 'BRO300', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'C210', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QP21XD-1873', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QP21XD-5791', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HVU-VZM16ADL', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H751CS-POE46', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H901CS-POE46', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H1001CS-POE46', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QR50', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QL362', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H1501CS-POE46', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H505CS-POE32', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'CDS181B-POEC85E', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H300CS-POE32', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H5000CC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H2700CS', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H1003CC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H2001CC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H3500CS', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H3000CC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H201CS', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'F-40H', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'BRO300', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'C210', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QP21XD-1873', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'QP21XD-5791', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'TABLI200281', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'TABLI204282', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'FE140Y-E P04554-01', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'RT-130E', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H2700CS', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H300CC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H201CS,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H201CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H300CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H755CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H755CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H505CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H201CC,POE32)', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'H3501CC,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'L-QDZY75G', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'LQDZY75G', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HSO4225 ', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HSO4222 ', 'Range'] = 'Out of Range'

df6.loc[df6['Model'] == 'HGX44E-475-4-S ', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '-SFQ421-TES-SH2523ZYZ-Q-4021-1Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '-SFD215-TES-SH2519ZYZ-D-2-15-1Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'SH2523ZTZ-Q-4-21-1Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '-SFV25713-TES-SH4627ZMZ-V25-71Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'Z-2Z30-102-51Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'Z-2V20-62-35Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HGX66E-1750-4,', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '-SFV251033-TES-SH2614ZMZ-V25103Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'BD35K', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '101Z0401', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HSO4227', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HS04225', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HSO5228', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HSL4224', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'N6WBHE', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'N62WA', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'N200-VLD-HX', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'N160-VLD-HX', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'N-250-VLD-HX', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'MK6D-WRVI255193', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK6D-WRVI-255193', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK6D-WRVI-255130', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK6D-WRV20419336', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK6C-WRVI-321193', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK5B-XRV-204165', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MK4A-XRV127-R4', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'HAK0750', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H6B4000', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H5E2500', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H4812034403A0', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H2N0361', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H2L0381', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'H1R0182', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DVXF13ME-AWD-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DUNF13ME-AWD-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DMXR37ME-TSN-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DJNR40ME-TSN-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DJNF11ME-TSK-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DHNR23ME-TSK-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DGXR37ME-TSN-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '6DGNR37ME-TSN-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '4DJNR28ME-TSK-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '4DHXR22ME-TSK-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3DSDR17ME-TFD-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3DB3R12ME_TFC_C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3DA3R10ME-TFD-C00', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3CMSD8252', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3CMSD5658', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3CMSD5354', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == '3CMDU5350', 'Range'] = 'Out of range'
df6.loc[df6['Model'] == 'LS4010', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'SHL615Z-2Z30-102.51Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'CTS70SBAF400', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'CXH93 160-810Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'ZHA7WSGLYE', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'EX-HGX34E/665-4', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'CIFVRH180540-3Y', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'N2016LLC', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'DSR15-8-5', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'CTS51SBAE400', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == '2AM-210-A0', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'SGC1918', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'SB446760', 'Range'] = 'Out of Range'
df6.loc[df6['Model'] == 'MSE166CL1H-ASH', 'Range'] = 'Range'
df6.loc[df6['Model'] == 'NI34T9102ADTS7', 'Range'] = 'Range'
df6.loc[df6['Model'] == 'NF54M7151ANASH', 'Range'] = 'Range'
df6.loc[df6['Model'] == 'MSV4A1AL1RTSJ', 'Range'] = 'Range'

# df6.loc[df6['Model'] == 'TAJ5519E', 'Range'] = 'Out of Range'
# df6.loc[df6['Model'] == '', 'Range'] = 'Out of Range'

# df6.loc[df6['Model'] == '', 'Range'] = 'Out of Range'
# df6.loc[df6['Model'] == '', 'Range'] = 'Out of Range'
# df6.loc[df6['Model'] == '', 'Range'] = 'Out of Range'

# Depois de definir tudo que é Out of Range, o que sobra vira Range
# Ou seja, transformo o que era vazio (não tinha sido definido) em Range
df6['Range'] = df6['Range'].replace('', 'Range')

#df6.loc[:, 'Model'] = df6['Model'].str.replace('-', '')

In [111]:
# Se o player for 'JABIL CIRCUIT (GUANGZHOU) LIMITED', o segmento é EXTRA
df6.loc[df6['Player Raw'] == 'JABIL CIRCUIT (GUANGZHOU) LIMITED', 'Segment'] = 'EXTRA'
df6.loc[df6['Importer Raw'] == 'TATA HITACHI CONSTRUCTION MACHINERY COMPANY PRIVAT', 'Segment'] = 'EXTRA'

# Define um filtro para ser alterado. Se o Importer for Midea e o Player for Toshiba.
filtro = (df6['Importer Final'] == 'MIDEA' ) & (df6['Player Final'] == 'TOSHIBA')
# Pega os valores do filtro acima e define segmento como HVAC. (Tudo que tem midea de importer e toshiba de player é HVAC)
df6.loc[filtro, 'Segment'] = 'HVAC'

# Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
filtro2 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'SCROLL')
# Pega os valores do filtro acima e define segmento como HVAC.
df6.loc[filtro2, 'Segment'] = 'HVAC'

# Define um filtro. Tudo que é HA e o modelo tem Technology Screw
filtro3 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'SCREW')
# Pega os valores do filtro acima e define segmento como EXTRA.
df6.loc[filtro3, 'Segment'] = 'EXTRA'

filtro4 = (df6['Importer Raw'] == 'SERVICIOS LOGISTICOS PALCO S DE RL DE CV' ) & (df6['Player Final'] == 'WHIRLPOOL')
df6.loc[filtro4, 'Segment'] = 'HA'

filtro4 = (df6['Importer Raw'] == 'VOLTAS LIMITED Total' ) & (df6['Player Raw'] == 'MIDEA ELECTRIC TRADING SINGAPORE  C')
df6.loc[filtro4, 'Segment'] = 'HVAC'

# Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
filtro5 = (df6['Segment'] == 'HA') & (df6['COMPRESSOR TYPE'] == 'ROTARY')
# Pega os valores do filtro acima e define segmento como HVAC.
df6.loc[filtro5, 'Segment'] = 'HVAC'

# Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
filtro6 = (df6['Segment'] == 'HA') & (df6['Player Final'] == 'TCL')
# Pega os valores do filtro acima e define segmento como HVAC.
df6.loc[filtro6, 'Segment'] = 'HVAC'


# filtro7 = (df6['ADUANA'] == 'Pakistan') & (df6['Player Raw'] == 'NINGBO AUX IMPORT AND EXPORT CO LTD')
# df6.loc[filtro7, 'Segment'] = 'HVAC'

# filtro8 = (df6['ADUANA'] == 'Pakistan') & (df6['Player Raw'] == 'NINGBO AUX ELECTRIC CO.LTD')
# df6.loc[filtro8, 'Segment'] = 'HVAC'


def set_nidec_player_final(row):
  if re.search(r"nidec", str(row['Player Raw']), re.IGNORECASE):
    return "NIDEC"
  else:
    return row['Player Final']

df6['Player Final'] = df6.apply(set_nidec_player_final, axis=1)

In [112]:
if country_name == 'Pakistan':
  df6.loc[df6['Player Raw'].str.contains('NINGBO AUX IMPORT AND EXPORT CO LTD|NINGBO AUX ELECTRIC CO.LTD', case=False, na=False), 'Segment'] = 'HVAC'
  df6.loc[df6['Player Raw'].str.contains('NOIDEC', case=False, na=False), 'Player Final'] = 'Nidec'

## 7.6. PREENCHER UNKNOWN



In [113]:
# Deixa tudo de Importer Final como maisculo
df6['Importer Final'] = df6['Importer Final'].str.upper()
# Deixa tudo de Player Final como maisculo
df6['Player Final'] = df6['Player Final'].str.upper()

In [114]:
# Troca os valores da lista para UNKNOWN
# A IDEIA É PADRONIZAR TODAS AS FORMAS DE SE ESCREVER UNKNOWN
df6 = df6.replace(['0', np.nan, '', 'nan', 'NOT DECLARED', 'NOT AVAILABLE', 'NO DISPONIBLE/NOT AVAILABLE',
                   'SIN RAZON SOCIAL', 'SIN MARCA', 'SIN MODELO', 'NAN', 'SIN-MODELO', 'SIN'], 'UNKNOWN')

# Tudo que está vazio vira UNKNOWN
df6 = df6.fillna('UNKNOWN')

In [115]:
df6['ADUANA'] = df6['ADUANA'].map(str)

country_region_mapping = {
'Vietnam':'APA',
'United Arab Emirates':'APA',
'Thailand':'APA',
'Indonesia':'APA',
'South Korea':'APA',
'Singapore':'APA',
'Philippines':'APA',
'Pakistan':'APA',
'New Zeland':'APA',
'Japan':'APA',
'India':'APA',
'Bangladesh':'APA',
'China':'CHINA',
'Turkey':'EMEA',
'Switzerland':'EMEA',
'Spain':'EMEA',
'Slovenia':'EMEA',
'Slovakia':'EMEA',
'Serbia':'EMEA',
'Saudi Arabia':'EMEA',
'Russia':'EMEA',
'Romania':'EMEA',
'Poland':'EMEA',
'Lithuania':'EMEA',
'Lebanon':'EMEA',
'Latvia':'EMEA',
'Italy':'EMEA',
'Hungary':'EMEA',
'Germany':'EMEA',
'Eswatini':'EMEA',
'Egypt':'EMEA',
'Bulgaria':'EMEA',
'Austria':'EMEA',
'Kenya':'EMEA',
'Kazakhstan':'EMEA',
'Ecuador':'LAR',
'Colombia':'LAR',
'Brazil':'LAR',
'Argentina':'LAR',
'United States':'NAR',
'Mexico':'NAR',
'Canada':'NAR'
}

df6['Region'] = df6['ADUANA'].map(country_region_mapping)

country_region_mapping_new_region = {
'Vietnam':'APAME',
'Indonesia':'APAME',
'United Arab Emirates':'APAME',
'Turkey':'APAME',
'Thailand':'APAME',
'South Korea':'APAME',
'Singapore':'APAME',
'Serbia':'APAME',
'Saudi Arabia':'APAME',
'Romania':'APAME',
'Philippines':'APAME',
'Pakistan':'APAME',
'New Zeland':'APAME',
'Lebanon':'APAME',
'Japan':'APAME',
'India':'APAME',
'Egypt':'APAME',
'Bangladesh':'APAME',
'Kenya':'APAME',


'China':'CHINA',

'Switzerland':'EUROPE',
'Spain':'EUROPE',
'Slovenia':'EUROPE',
'Slovakia':'EUROPE',
'Russia':'EUROPE',
'Poland':'EUROPE',
'Lithuania':'EUROPE',
'Latvia':'EUROPE',
'Italy':'EUROPE',
'Hungary':'EUROPE',
'Germany':'EUROPE',
'Eswatini':'EUROPE',
'Bulgaria':'EUROPE',
'Austria':'EUROPE',
'Kazakhstan':'EUROPE',


'Ecuador':'LAR',
'Colombia':'LAR',
'Brazil':'LAR',
'Argentina':'LAR',

'United States':'NAR',
'Mexico':'NAR',
'Canada':'NAR'
}

df6['New Region'] = df6['ADUANA'].map(country_region_mapping_new_region)

# 8.0. RETURN

In [116]:
# Define o diretório de OUTPUT
#df6['Date'] = df6['Date'].astype(str)
df6['Volume'] = pd.to_numeric(df6['Volume'], errors='coerce')

results = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/RESULTS/'
output_path  = os.path.join(results, f'{country_name}.xlsx')

# Salva o dataset resultado de tudo no OUTPUT escolhido acima
df6.to_excel(output_path, index=False)

In [117]:
# Seleciona somente as colunas relevantes para o Report
cols = ['Date', 'Year', 'Month', 'Quarter CY', 'Segment', 'HS Code',  'Importer Raw', 'Importer Final', 'Country Raw', 'Player Raw',
        'Player Final', 'Vertical', 'Range' ,'Description', 'Model', 'Technology', 'Volume', 'Value (USD)', 'UNIT PRICE',
        'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'COMPRESSOR TYPE', 'Region', 'New Region']


df7 = df6[cols]
# Define o diretório de OUTPUT do Report
results = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/REPORTS/'
output_path  = os.path.join(results, f'{country_name}.xlsx')

# Salva o dataset resultado de tudo no OUTPUT escolhido acima
df7.to_excel(output_path, index=False)

In [118]:
# #PARA GERAR NO ultimoRodado
# df6['Volume'] = pd.to_numeric(df6['Volume'], errors='coerce')
# check = '/content/drive/Shareddrives/Trade_Marketing_HA/CUSTOMS/Customs-HA/RESULTS/Check Customs/ultimoRodado'
# output_path_check  = os.path.join(check, f'{country_name}.xlsx')
# df6.to_excel(output_path_check, index=False)

# !!!!!!!!!!!! RODAR TUDO DE UMA VEZ !!!!!!!!!!!!!!!!!

Essa parte vai ficar comentada, quando quiser rodar todas as aduanas de uma única vez, basta descomentar e colocar pra rodar.


Para descomentar todas as linhas de uma unica vez, clique na célula e depois aperte Ctrl + A para selecionar todas as linhas e depois clique Ctrl + / para comentar tudo de uma vez ( comentar e descomentar)

In [119]:
# # Diretório BASE onde tiraremos as informações
# pasta_drive = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/'
# # Lista todos os diretórios dentro da pasta BASE
# diretorios = os.listdir(pasta_drive)
# # Lista de diretórios desejados
# diretorios_desejados = ["APA", "EMEA", "LAR", "NAR"]

# # Loop para passar por todos os diretórios
# for i, diretorio in enumerate(diretorios):
#   # Se for um diretório:
#   if os.path.isdir(os.path.join(pasta_drive, diretorio)) and diretorio in diretorios_desejados:
#     # Printa os diretórios
#     print(f"{i+1}. {diretorio}")

#     # Lista todos os arquivos que existem dentro da pasta escolhida
#     arquivos = os.listdir(os.path.join(pasta_drive, diretorio))
#     # Se for um arquivo:
#     for arquivo in arquivos:
#       # Caminho completo do arquivo
#       arquivo_escolhido = os.path.join(pasta_drive, diretorio, arquivo)
#       # Verifica se o arquivo é um arquivo Excel (pode ser necessário ajustar a extensão)
#       if arquivo.endswith(('.xlsx', '.xlsm', '.xls')):
#         print(f"Processando arquivo: {arquivo_escolhido}")

#         # ***Atualização do country_name***
#         file_name = os.path.basename(arquivo_escolhido)
#         country_name = country_mapping.get(file_name)
#         print(country_name)

#         # Carrega o arquivo usando pandas
#         df_raw = pd.read_excel(arquivo_escolhido)
#         # Faz uma cópia do arquivo original, para não sobreescrevê-lo
#         df = df_raw.copy()
#         # Aplica todas as funções de padronização de colunas.
#         df = padronizar_coluna_hs_code(df)
#         df = padronizar_coluna_importer(df)
#         df = padronizar_coluna_country(df)
#         df = padronizar_coluna_player_raw(df)
#         df = padronizar_coluna_description(df)
#         df = padronizar_coluna_volume(df)
#         df, valor = padronizar_coluna_value(df)

#         # Faz uma cópia do df, para não sobreescrevê-lo, assim se garante consistência.
#         df1 = df.copy()
#         # Lista todas as colunas
#         colunas_para_modificar = df1.columns.tolist()
#         # Remove a coluna 'Volume' pois precisamos dela e ela não pode ser excluída de jeito algum
#         colunas_para_modificar.remove('Volume')

#         # Percorre cada uma das colunas
#         for coluna in colunas_para_modificar:
#           # Se a coluna for inteiramente nula:
#           if df1[coluna].isnull().all():
#             # Exclui ela da nossa database
#             df1.drop(coluna, axis=1, inplace=True)

#         # A IDEIA AQUI É TIRAR COLUNAS QUE DEIXARÃO NOSSO DATASET MAIS PESADO MAS QUE NÃO CARREGAM NENHUMA INFORMAÇÃO.

#         # Faz uma cópia do df1, para não sobreescrevê-lo, assim se garante consistência.
#         df2 = df1.copy()

#         # Lista dos possíveis nomes das colunas em que o valor é CIF
#         cif_values = ["CIF Value [USD]", "Valor CIF (USD)", "U$S CIF"]
#         # Lista dos possíveis nomes das colunas em que o valor é FOB
#         fob_values = ["Valor Total FOB US$", "Valor FOB (USD)", "FOB Calculado  (USD)", "Subitem: Calculated FOB Value (US$)", "FOB Value (USD)"]

#         # Se nossa coluna 'Value' for uma entre as colunas CIF:
#         if valor in cif_values:
#           # Atribua na coluna "CIF/FOB" o valor 'CIF'
#           df2['CIF/FOB'] = 'CIF'
#         # Se nossa coluna 'Value' for uma entre as colunas CIF:
#         elif valor in fob_values:
#           # Atribua na coluna "CIF/FOB" o valor 'FOB'
#           df2['CIF/FOB'] = 'FOB'
#         else:
#           # Caso nossa coluna Value não tenha sido derivada de nenhuma daquelas nas listas presentes então mantenha como UNKNOWN
#           df2['CIF/FOB'] = 'UNKNOWN'

#         # A IDEIA AQUI É SABER SE OS VALORES QUE ESTAREMOS VENDO SERÃO FOB, CIF OU NÃO TEREMOS ESSA INFORMAÇÃO


#         # Aplica todas as funções para criar novas colunas
#         df2 = criar_coluna_year_month(df2)
#         df2 = criar_coluna_data(df2)
#         df2 = criar_coluna_quarter(df2)
#         df2 = criar_coluna_samples(df2)
#         df2 = criar_coluna_range(df2)
#         df2 = criar_coluna_model(df2)
#         df2 = criar_coluna_segmento(df2)
#         df2 = criar_coluna_unit_price(df2)
#         df2 = criar_coluna_aduana(df2)
#         df2 = criar_coluna_vertical(df2)
#         df2 = criar_coluna_manufacturer_category(df2)
#         df2 = criar_coluna_market_aviability(df2)
#         df2 = criar_coluna_cap_range(df2)
#         df2 = criar_coluna_cop_range(df2)

#         # Faz uma cópia do df2, para não sobreescrevê-lo, assim se garante consistência.
#         df3 = df2.copy()

#         # Importer Final se torna uma cópia de Importer Raw.
#         # A IDEIA É PRIMEIRO FAZER UMA CÓPIA PARA DEPOIS TRATARMOS ESSES DADOS E CHEGAR NO RESULTADO ESPERADO
#         df3['Importer Final'] = df3['Importer Raw']

#         # Player Final se torna uma cópia de Player Raw.
#         # A IDEIA É PRIMEIRO FAZER UMA CÓPIA PARA DEPOIS TRATARMOS ESSES DADOS E CHEGAR NO RESULTADO ESPERADO
#         df3['Player Final'] = df3['Player Raw']

#         # Aplica a função 'categorize_technology' em cada uma das linhas do meu dataset
#         df3['Technology'] = df3['Description'].apply(categorize_technology)

#         # Aplica a função abaixo no dataset
#         df3 = categorize_compressor_type(df3)



#         # Faz uma cópia do df3, para não sobreescrevê-lo, assim se garante consistência.
#         df4 = df3.copy()

#         # Converter a coluna para string (formato de texto)
#         df4['Importer Final'] = df4['Importer Final'].astype(str)

#         # Aplica a função 'renomear_importers' no dataset
#         df4 = renomear_importers(df4)

#         # Deixa tudo da coluna 'Importer Final' em maisculo
#         df4['Importer Final'] = df4['Importer Final'].str.upper()

#         # Aplica a função 'unknown_importer' em cada linha
#         df4['Importer Final'] = df4.apply(unknown_importer, axis=1)

#         # Converter a coluna para string (formato de texto)
#         df4['Player Final'] = df4['Player Final'].astype(str)

#         # Aplica a função 'renomear_players' no dataset
#         df4 = renomear_players(df4)

#         # Deixa tudo da coluna 'Player Final' em maisculo
#         df4['Player Final'] = df4['Player Final'].str.upper()

#         # Aplica a função 'player_in_description' em cada linha
#         df4['Player Final'] = df4.apply(player_in_description, axis=1)

#         # Aplica a função 'extrair_player_description' no dataset
#         df4 = extrair_player_description(df4)

#         # Se o Player Final for 'EMBRACO', troca para 'NIDEC'
#         df4.loc[df4['Player Final'] == 'EMBRACO', 'Player Final'] = 'NIDEC'

#         # Aplica a função renomear_models no dataset
#         df4 = renomear_models(df4)

#         if country_name == 'Colombia':
#             # Aplica a função 'extrair_quantidades' à coluna 'Description'
#             # e armazena os resultados na nova coluna 'Quantidades'.
#             df4['Quantidades'] = df4['Description'].apply(extrair_quantidades)

#             # Aplica a função 'extrair_modelos' à coluna 'Description'
#             # e armazena os resultados na nova coluna 'Modelos'.
#             df4['Modelos'] = df4['Description'].apply(extrair_modelos)

#             # Aplica a função processar_dataframe
#             df4 = processar_dataframe(df4)


#         # Faz uma cópia do df4, para não sobreescrevê-lo, assim se garante consistência.
#         df5 = df4.copy()
#         # Lê o catalogo que está no drive
#         catalogo  = pd.read_excel('/content/drive//Shared drives/Trade_Marketing_HA/CUSTOMS/Catalogos/Catalogo 2024.xlsx')

#         # Se a aduana for da Argentina:
#         if country_name == 'Argentina':
#           # Exclui a coluna Model
#           df5.drop(columns = 'Model', axis = 1, inplace = True)
#           # Troca o nome de 'Subitem: Model' para 'Model'
#           df5.rename(columns = {'Subitem: Model': 'Model'}, inplace = True)

#           # Exclui linhas do catálogo cujos modelos são repetidos
#           catalogo.drop_duplicates(subset = 'Model', inplace = True)

#           # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
#           df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player']], on='Model', how='left')

#         # Se a aduana for do Ecuador:
#         elif country_name == 'Ecuador':
#           # Exclui linhas do catálogo cujos modelos são repetidos
#           catalogo.drop_duplicates(subset = 'Model', inplace = True)

#           # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
#           df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player']], on='Model', how='left')

#         # Se a aduana for qualquer outra:
#         else:
#           # Tira todos os simbolos que não sejam letras ou números dos modelos do catálogo
#           catalogo['Model'] = catalogo['Model'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)
#           # Tira todos os simbolos que não sejam letras ou números da Description e atribua isso a uma nova coluna chamada 'Description_Aux'
#           df5['Description_Aux'] = df5['Description'].str.lower().str.replace('[^a-zA-Z0-9\s]', ' ', regex=True)

#           # Aplica a função 'find_models_on_description' na coluna 'Description_Aux'
#           df5['Model'] = df5['Description_Aux'].apply(find_models_on_description)

#           # Exclui linhas do catálogo cujos modelos são repetidos
#           catalogo.drop_duplicates(subset = 'Model', inplace = True)

#           # Dá um merge entre o Model e o Catalogo, buscando todas as informações do Model.
#           df5 = df5.merge(catalogo[['Model', 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)', 'Technology', 'Player']], on='Model', how='left')

#           # Exclui a coluna auxiliar de description
#           df5.drop(columns = ['Description_Aux'], inplace = True)

#           # Preenche as linhas cujo Model esteja vazio com 'Unknown'
#         df5['Model'] = df5['Model'].fillna('Unknown')

#         # Deixa tudo da coluna Model em maisculo
#         df5['Model'] = df5['Model'].str.upper()

#         # Aplica a funcao 'arrumar_coluna_tech' no dataset
#         df5 = arrumar_coluna_tech(df5)


#         # Faz uma cópia do df5, para não sobreescrevê-lo, assim se garante consistência.
#         df6 = df5.copy()

#         # Converte a coluna COP para numerico
#         df6['COP (W/W)'] = pd.to_numeric(df6['COP (W/W)'], errors='coerce')

#         # Aplica a função 'preencher_market_aviability' no dataset
#         df6['Market Aviability'] = df6.apply(preencher_market_aviability, axis=1)

#         # Aplica a função 'encontrar_vertical' no dataset
#         df6['Vertical'] = df6.apply(encontrar_vertical, axis=1)

#         # Aplica a função 'preencher_competitor_category' no dataset
#         df6['Compressor Manufacturer Category'] = df6.apply(preencher_competitor_category, axis=1)

#         # Converte a coluna CAP para numerico
#         df6['CAP (W)'] = pd.to_numeric(df6['CAP (W)'], errors='coerce')
#         # Converte a coluna COP para numerico
#         df6['COP (W/W)'] = pd.to_numeric(df6['COP (W/W)'], errors='coerce')

#         # Preenche a coluna CAP Range aplicando a função 'preencher_cap_range'
#         df6['CAP Range'] = df6.apply(preencher_cap_range, axis=1)
#         # Preenche a coluna COP Range aplicando a função 'preencher_cop_range'
#         df6['COP Range'] = df6.apply(preencher_cop_range, axis=1)

#                 # Cria um dicionario de segmentos
#         dic = cria_dicionario_segmento()

#         # Faz um dictionary em que cada Importer está associado a um Segment
#         seg_dic = dict(zip(dic['Importer'], dic['Segment']))
#         # Usa o dicionario para preencher a coluna Segment de acordo com o Importer Final
#         df6['Segment'] = df6['Importer Final'].map(seg_dic).fillna(df6['Segment'])

#         # Aplica a função 'segment_by_hscode' no dataset
#         df6 = segment_by_hscode(df6)

#         # Aplica todas as funções para definir segmento a partir dos importers
#         df6 = importers_extra(df6)
#         df6 = importers_ac(df6)
#         df6 = importers_ca(df6)

#         # Aplica todas as funções para definir segmento a partir dos players
#         df6 = players_extra(df6)
#         df6 = players_ac(df6)
#         df6 = players_ca(df6)

#         # Aplica todas as funções para definir segmento a partir do que está escrito na descrição
#         df6 = ac_by_description(df6)
#         df6 = extra_by_description(df6)

#         # Aplica a função 'intracompany_segment' no dataset
#         df6['Segment'] = df6.apply(intracompany_segment, axis=1)

#         # Reaplicar as funções para definir EXTRA como segmento para garantir que não erros
#         df6 = importers_extra(df6)
#         df6 = players_extra(df6)
#         df6 = extra_by_description(df6)
#         df6 = wet_by_description(df6)
#         df6 = segment_by_hscode(df6)

#         # Se o player for 'JABIL CIRCUIT (GUANGZHOU) LIMITED', o segmento é EXTRA
#         df6.loc[df6['Player Raw'] == 'JABIL CIRCUIT (GUANGZHOU) LIMITED', 'Segment'] = 'EXTRA'

#         # Define um filtro para ser alterado. Se o Importer for Midea e o Player for Toshiba.
#         filtro = (df6['Importer Final'] == 'MIDEA' ) & (df6['Player Final'] == 'TOSHIBA')
#         # Pega os valores do filtro acima e define segmento como HVAC. (Tudo que tem midea de importer e toshiba de player é HVAC)
#         df6.loc[filtro, 'Segment'] = 'HVAC'

#         # Define um filtro. Tudo que é HA e o modelo tem Technology Scroll
#         filtro2 = (df6['Segment'] == 'HA') & (df6['Technology'] == 'SCROLL')
#         # Pega os valores do filtro acima e define segmento como HVAC.
#         df6.loc[filtro2, 'Segment'] = 'HVAC'

#         # Define um filtro. Tudo que é HA e o modelo tem Technology Screw
#         filtro3 = (df6['Segment'] == 'HA') & (df6['Technology'] == 'SCREW')
#         # Pega os valores do filtro acima e define segmento como EXTRA.
#         df6.loc[filtro3, 'Segment'] = 'EXTRA'

#         # Aplica as funções para corrigir os modelos
#         df6 = corrigir_models(df6)
#         df6 = corrigir_models_com_players(df6)

#         # Aqui, cada uma dessas linhas faz o seguinte:
#         # Se o modelo for o especificado, altera o segmento para o definido ao fim.
#         # Exemplo: Se o modelo for 'L126WY1', então segmento vai ser CA.
#         df6.loc[df6['Model'] == 'L126WY1',  'Segment'] = 'CA'
#         df6.loc[df6['Model'] == 'A120WY1A',  'Segment'] = 'CA'
#         df6.loc[df6['Model'] == 'ANA90XL',  'Segment'] = 'CA'
#         df6.loc[df6['Model'] == 'ANA120XL ',  'Segment'] = 'CA'
#         df6.loc[df6['Model'] == 'DST156MAA',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'SNB140FCAMC',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'MNB36FABMC',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'C 6RZ132H3AAF',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'KSN108D34UEZ3',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'KSK89D29UEZD',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'PH290M2C 4FT6',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'KSF180S1VFPA',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'ZB45KQE TFD 523',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'ZB29KQE TFD 524',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == '5RD132XBB21',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == '5RD132XHA21',  'Segment'] = 'HVAC'
#         df6.loc[df6['Model'] == 'PBL-CB417BLDU',  'Segment'] = 'EXTRA'
#         df6.loc[df6['Model'] == 'C SWS225H01C',  'Segment'] = 'EXTRA'
#         df6.loc[df6['Model'] == 'ANB52FKFMT',  'Segment'] = 'EXTRA'
#         df6.loc[df6['Model'] == 'PZ65E1',  'Segment'] = 'HA'

#         # Aqui, cada uma dessas linhas faz o seguinte:
#         # Se o modelo for o especificado, altera o Player para o definido ao fim.
#         # Exemplo: Se o modelo for 'EY40L58W', então Player vai ser HUAGUANG.
#         df6.loc[df6['Model'] == 'EY40L58W', 'Player Final'] = 'HUAGUANG'
#         df6.loc[df6['Model'] == 'PZ65E1C 9', 'Player Final'] = 'GMCC'
#         df6.loc[df6['Model'] == 'MSV172AL2J SM3', 'Player Final'] = 'SAMSUNG'
#         df6.loc[df6['Model'] == 'MKV190CL2B   ASH', 'Player Final'] = 'SAMSUNG'

#         # Aplica a função para remover modelos curtos
#         df6 = remover_modelos_curtos(df6)

#         # Substitui toda instancia de espaço por um traço.
#         df6['Model'] = df6['Model'].str.replace(' ', '-')
#         # A IDEIA AQUI É QUE ANTES PARA DAR MERGE COM O CATALOGO FOI PRECISO RETIRAR TODOS OS CARACTERES ESPECIAIS,
#         # AGORA EU RECOLOCO (OS CARACTERES ESPECIAIS COSTUMAM SER '-')

#         # Converte as colunas para numerica
#         df6['Value (USD)'] = pd.to_numeric(df6['Value (USD)'], errors='coerce')
#         df6['Volume'] = pd.to_numeric(df6['Volume'], errors='coerce')

#         # Cria uma coluna UNIT PRICE que é o Value / Volume
#         df6['UNIT PRICE'] = df6['Value (USD)'] / df6['Volume']

#         # Modifica a coluna do UNIT PRICE no caso da colombia
#         if country_name == 'Colombia':
#           df6 = modificar_unit_price(df6)

#         # Aplica a função no dataset.
#         df6 = range_by_description(df6)

#         # Crio um filtro. Se o UNIT PRICE for maior que 300 ou menor que 8
#         filtro = (df6['UNIT PRICE'] > 300) | (df6['UNIT PRICE'] < 8)
#         # Crio um filtro. Se a tecnologia for SCROLL ou SCREWe
#         filtro2 = (df6['Technology'] == 'SCROLL') | (df6['Technology'] == 'SCREW')

#         # Passa os dois filtros definidos acima para a função 'preenche_range'.
#         # Então, tudo que cai em alguns dos dois filtros se torna 'Out of Range'
#         df6 = preenche_range(df6, filtro )
#         df6 = preenche_range(df6, filtro2 )

#         # Tudo que é EXTRA é Out of Range
#         df6.loc[df6['Segment'] == 'EXTRA', 'Range'] = 'Out of Range'
#         # Tudo que é INTRACOMPANY é Out of Range
#         df6.loc[df6['Segment'] == 'INTRACOMPANY', 'Range'] = 'Out of Range'

#         # Depois de definir tudo que é Out of Range, o que sobra vira Range
#         # Ou seja, transformo o que era vazio (não tinha sido definido) em Range
#         df6['Range'] = df6['Range'].replace('', 'Range')

#         # Deixa tudo de Importer Final como maisculo
#         df6['Importer Final'] = df6['Importer Final'].str.upper()
#         # Deixa tudo de Player Final como maisculo
#         df6['Player Final'] = df6['Player Final'].str.upper()

#         # Troca os valores da lista para UNKNOWN
#         # A IDEIA É PADRONIZAR TODAS AS FORMAS DE SE ESCREVER UNKNOWN
#         df6 = df6.replace(['0', np.nan, '', 'nan', 'NOT DECLARED', 'NOT AVAILABLE', 'NO DISPONIBLE/NOT AVAILABLE',
#                           'SIN RAZON SOCIAL', 'SIN MARCA', 'SIN MODELO', 'NAN', 'SIN-MODELO', 'SIN'], 'UNKNOWN')

#         # Tudo que está vazio vira UNKNOWN
#         df6 = df6.fillna('UNKNOWN')

#         # Define o diretório de OUTPUT
#         results = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/RESULTS/'
#         output_path  = os.path.join(results, f'{country_name}.xlsx')

#         # Salva o dataset resultado de tudo no OUTPUT escolhido acima
#         df6.to_excel(output_path, index=False)

#         # Seleciona somente as colunas relevantes para o Report
#         cols = ['Date', 'Year', 'Month', 'Quarter CY', 'Segment', 'HS Code',  'Importer Raw', 'Importer Final', 'Country Raw', 'Player Raw',
#                 'Player Final', 'Vertical', 'Range' ,'Description', 'Model', 'Technology', 'Volume', 'Value (USD)', 'UNIT PRICE',
#                 'Displacement (cm³)', 'Voltage', 'Frequency/RPM', 'CAP (W)', 'COP (W/W)']


#         df7 = df6[cols]
#         # Define o diretório de OUTPUT do Report
#         results = '/content/drive/Shared drives/Trade_Marketing_HA/CUSTOMS/Customs-HA/REPORTS/'
#         output_path  = os.path.join(results, f'{country_name}.xlsx')

#         # Salva o dataset resultado de tudo no OUTPUT escolhido acima
#         df7.to_excel(output_path, index=False)